# **Contextual Multi-Label Identification of Plant Species in Multi-Species Plots**

# **CELL 1— Create virtual environment**

In [ ]:
import subprocess
from pathlib import Path

VENV_DIR = Path("/workspace/data/andrea/venvs/plantclef_gpu_cu121")
VENV_DIR.parent.mkdir(parents=True, exist_ok=True)

if not VENV_DIR.exists():
    print("Creando entorno:", VENV_DIR)
    subprocess.run(["python3.11", "-m", "venv", str(VENV_DIR)], check=True)
else:
    print("El entorno ya existe:", VENV_DIR)

pip = str(VENV_DIR / "bin" / "pip")
python = str(VENV_DIR / "bin" / "python")

print("\nActualizando pip...")
subprocess.run([pip, "install", "-q", "--upgrade", "pip"], check=True)

print("\nInstalando PyTorch compatible con CUDA 12.1...")
subprocess.run([
    pip, "install", "-q",
    "torch==2.5.1",
    "torchvision==0.20.1",
    "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

print("\nInstalando librerías del proyecto...")
subprocess.run([
    pip, "install", "-q",
    "timm",
    "pandas",
    "numpy",
    "pillow",
    "opencv-python-headless",
    "pyarrow",
    "tqdm",
    "scikit-learn",
    "pyyaml",
    "ipykernel"
], check=True)

print("\nRegistrando kernel en Jupyter...")
subprocess.run([
    python, "-m", "ipykernel", "install",
    "--user",
    "--name", "plantclef_gpu_cu121",
    "--display-name", "Python (plantclef_gpu_cu121)"
], check=True)

print("\nProbando CUDA dentro del entorno...")
subprocess.run([
    python, "-c",
    """
import torch
print('torch:', torch.__version__)
print('torch.version.cuda:', torch.version.cuda)
print('CUDA disponible:', torch.cuda.is_available())
print('Número de GPUs:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
"""
], text=True)

print("\nListo. Si CUDA disponible aparece True, cambia el kernel a:")
print("Python (plantclef_gpu_cu121)")

# **CELL 2 — Final verification after changing kerne**

In [ ]:
import torch
import subprocess

print("torch:", torch.__version__)
print("torch.version.cuda:", torch.version.cuda)
print("CUDA disponible:", torch.cuda.is_available())
print("Número de GPUs:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU no disponible para PyTorch")

print("\nResultado de nvidia-smi:")
r = subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
print(r.stdout)
print(r.stderr)

# **CELL 3 — Main Configuration**

In [ ]:
from pathlib import Path
import json


NOTEBOOK_ROOT = Path("/workspace/notebooks/andrea/plantclef-2026")
STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")

INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
RAW_ROOT = STORAGE_ROOT / "data" / "raw"
MODELS_DIR = STORAGE_ROOT / "models"
RUNS_DIR = STORAGE_ROOT / "runs"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

for p in [INDEX_DIR, RAW_ROOT, MODELS_DIR, RUNS_DIR, SUBMISSIONS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CFG = {
    "run_name": "dinov2_tiling_adaptive_k_v1",

    "model": {
        "backbone": "vit_base_patch14_reg4_dinov2.lvd142m",
        "num_classes": 7806,
        "img_size": 518,
    },

    "tiling": {
        "tile_size": 518,
        "stride": 259,
        "include_full_image": True,
        "include_center_crop": True,
        "max_tiles_per_image": None,
    },

    "inference": {
        "batch_size": 16,          
        "num_workers": 4,
        "topk_per_view": 40,
        "amp": True,
    },

    "aggregation": {
        "agg_topk": 5,
        "w_mean_top": 0.55,
        "w_max": 0.25,
        "w_freq": 0.10,
        "w_veg": 0.10,
    },

    "selection": {
        "min_k": 2,
        "max_k": 12,

        "low_green_k": 2,
        "medium_green_k": 6,
        "high_green_k": 12,

        "low_green_thr": 0.05,
        "medium_green_thr": 0.15,

        "score_drop_ratio": 0.42,
    },

    "submission": {
        "filename": "submission_dinov2_tiling_adaptive_k_v1.csv",
    }
}

RUN_DIR = RUNS_DIR / CFG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = RUN_DIR / "config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(CFG, f, indent=2)

print("=" * 80)
print("CONFIGURACIÓN DEL EXPERIMENTO")
print("=" * 80)

print("NOTEBOOK_ROOT:", NOTEBOOK_ROOT)
print("STORAGE_ROOT:", STORAGE_ROOT)
print("INDEX_DIR:", INDEX_DIR)
print("RAW_ROOT:", RAW_ROOT)
print("MODELS_DIR:", MODELS_DIR)
print("RUN_DIR:", RUN_DIR)
print("SUBMISSIONS_DIR:", SUBMISSIONS_DIR)

print("\nConfig guardada en:")
print(CONFIG_PATH)

print("\nCFG:")
print(json.dumps(CFG, indent=2))

# **CELL 4 — Load clean test and species_table**

In [ ]:
from pathlib import Path
import pandas as pd


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"

test_manifest_path = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
species_table_path = INDEX_DIR / "species_table.pkl.gz"


print("=" * 80)
print("VERIFICANDO ARCHIVOS NECESARIOS")
print("=" * 80)

required_files = {
    "test_2025_manifest_clean": test_manifest_path,
    "species_table": species_table_path,
}

for name, path in required_files.items():
    print(f"{name:30s} -> {path} | existe: {path.exists()}")

missing = [str(p) for p in required_files.values() if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Faltan archivos necesarios:\n" + "\n".join(missing)
    )


test_df = pd.read_pickle(test_manifest_path, compression="gzip")
species_table = pd.read_pickle(species_table_path, compression="gzip")


test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)
test_df["file_path"] = test_df["file_path"].astype(str)

species_table["species_id"] = species_table["species_id"].astype(int)

VALID_QUADRATS = set(test_df["quadrat_id"].tolist())
VALID_SPECIES = set(species_table["species_id"].tolist())


print("\n" + "=" * 80)
print("RESUMEN TEST")
print("=" * 80)

print("Test shape:", test_df.shape)
print("Quadrat_id únicos:", test_df["quadrat_id"].nunique())
print("Duplicados quadrat_id:", test_df["quadrat_id"].duplicated().sum())
print("Rutas de imagen nulas:", test_df["file_path"].isna().sum())

# Verificar si las imágenes existen
test_df["file_exists"] = test_df["file_path"].apply(lambda p: Path(p).exists())
print("Imágenes existentes:", test_df["file_exists"].sum())
print("Imágenes faltantes:", (~test_df["file_exists"]).sum())

if (~test_df["file_exists"]).sum() > 0:
    print("\nEjemplos de imágenes faltantes:")
    display(test_df.loc[~test_df["file_exists"], ["quadrat_id", "file_path"]].head())

print("\n" + "=" * 80)
print("RESUMEN SPECIES TABLE")
print("=" * 80)

print("Species table shape:", species_table.shape)
print("Species_id únicos:", species_table["species_id"].nunique())
print("Species_id mínimo:", species_table["species_id"].min())
print("Species_id máximo:", species_table["species_id"].max())

print("\nPrimeras filas de test_df:")
display(test_df.head())

print("\nPrimeras filas de species_table:")
display(species_table.head())

print("\nÚltimas filas de species_table:")
display(species_table.tail())

# **CELL 5 — Search for checkpoint broadly**

In [ ]:
from pathlib import Path
import pandas as pd
import os
import time

NOTEBOOK_ROOT = Path("/workspace/notebooks/andrea/plantclef-2026")
STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")

SEARCH_CKPT_ROOTS = [
    Path("/workspace/notebooks/andrea/PlantNet_PlantCLEF2024_pretrained_models_on_the_flora_of_south-western_europe"),
    Path("/workspace/notebooks/andrea"),
    NOTEBOOK_ROOT,
    STORAGE_ROOT / "models",
    STORAGE_ROOT,
]


valid_suffixes = {
    ".pth", ".pt", ".bin", ".ckpt", ".tar", ".safetensors", ".onnx"
}

name_keywords = [
    "checkpoint",
    "ckpt",
    "model",
    "weights",
    "state",
    "dinov2",
    "vit",
    "plantclef",
    "plantnet",
    "onlyclassifier",
    "then_all",
    "pth",
]

rows = []

print("=" * 80)
print("BUSCANDO CHECKPOINTS DE FORMA AMPLIA")
print("=" * 80)

for root in SEARCH_CKPT_ROOTS:
    print("\nRevisando:", root)
    print("Existe:", root.exists())
    
    if not root.exists():
        continue
    
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        
        name_lower = p.name.lower()
        path_lower = str(p).lower()
        suffix_lower = p.suffix.lower()
        
        looks_like_ckpt = (
            suffix_lower in valid_suffixes
            or any(k in name_lower for k in name_keywords)
            or any(k in path_lower for k in ["onlyclassifier", "dinov2", "plantnet", "plantclef"])
        )
        
        if not looks_like_ckpt:
            continue
        
        try:
            size_mb = p.stat().st_size / 1024**2
            
            
            if size_mb < 1:
                continue
            
            priority = 0
            
            if "onlyclassifier_then_all" in path_lower:
                priority += 200
            if "then_all" in path_lower:
                priority += 120
            if "onlyclassifier" in path_lower:
                priority += 100
            if "dinov2" in path_lower:
                priority += 80
            if "vit_base_patch14" in path_lower:
                priority += 60
            if "plantclef" in path_lower:
                priority += 40
            if "plantnet" in path_lower:
                priority += 30
            if suffix_lower in [".pth", ".pt", ".ckpt", ".safetensors"]:
                priority += 20
            if ".pth.tar" in name_lower:
                priority += 25
            
            rows.append({
                "file": p.name,
                "path": str(p),
                "suffix": suffix_lower,
                "size_mb": round(size_mb, 2),
                "priority": priority,
                "modified_time": time.ctime(p.stat().st_mtime),
            })
            
        except Exception as e:
            print("Error leyendo:", p, e)

if len(rows) == 0:
    print("\n No encontré archivos candidatos.")
    print("\nVamos a listar el contenido de la carpeta PlantNet para revisar manualmente.")
    
    root = Path("/workspace/notebooks/andrea/PlantNet_PlantCLEF2024_pretrained_models_on_the_flora_of_south-western_europe")
    if root.exists():
        print("\nContenido inmediato de:")
        print(root)
        for p in sorted(root.iterdir()):
            try:
                size_mb = p.stat().st_size / 1024**2 if p.is_file() else None
                print(
                    "[DIR] " + p.name if p.is_dir()
                    else f"[FILE] {p.name} | {size_mb:.2f} MB"
                )
            except Exception as e:
                print(p, e)
    
    raise FileNotFoundError(
        "No encontré checkpoint. Necesitamos inspeccionar la carpeta o descargar/copiar los pesos."
    )

ckpts = pd.DataFrame(rows)

ckpts = ckpts.sort_values(
    ["priority", "size_mb"],
    ascending=[False, False]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("CHECKPOINTS / ARCHIVOS CANDIDATOS ENCONTRADOS")
print("=" * 80)

display(ckpts.head(100))

CHECKPOINT_PATH = Path(ckpts.iloc[0]["path"])

print("\n" + "=" * 80)
print("CHECKPOINT SELECCIONADO AUTOMÁTICAMENTE")
print("=" * 80)
print(CHECKPOINT_PATH)
print("Tamaño MB:", ckpts.iloc[0]["size_mb"])
print("Prioridad:", ckpts.iloc[0]["priority"])

print("\nIMPORTANTE:")
print("Antes de seguir, revisa la tabla.")
print("El mejor candidato debería parecerse a:")
print("- onlyclassifier_then_all")
print("- dinov2")
print("- vit_base_patch14")
print("- .pth.tar, .pth, .pt o .safetensors")

# **CELL 6 — Create DINOv2 model and load PlantCLEF checkpoint**

In [ ]:
import torch
import timm
from pathlib import Path
import gc

CHECKPOINT_PATH = Path(
    "/workspace/notebooks/andrea/PlantNet_PlantCLEF2024_pretrained_models_on_the_flora_of_south-western_europe/"
    "pretrained_models/vit_base_patch14_reg4_dinov2_lvd142m_pc24_onlyclassifier_then_all/model_best.pth.tar"
)

print("=" * 80)
print("CHECKPOINT")
print("=" * 80)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("Existe:", CHECKPOINT_PATH.exists())

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)

print("Tamaño GB:", round(CHECKPOINT_PATH.stat().st_size / 1024**3, 3))


def extract_and_clean_state_dict(ckpt):
    """
    Extrae el state_dict de diferentes formatos posibles de checkpoint
    y limpia prefijos como module. o model.
    """
    print("\n" + "=" * 80)
    print("INSPECCIÓN DEL CHECKPOINT")
    print("=" * 80)

    if isinstance(ckpt, dict):
        print("Checkpoint es dict.")
        print("Keys principales:")
        print(list(ckpt.keys())[:30])

       
        candidate_keys = [
            "state_dict",
            "model",
            "model_state_dict",
            "net",
            "teacher",
            "student",
        ]

        state = None
        used_key = None

        for k in candidate_keys:
            if k in ckpt and isinstance(ckpt[k], dict):
                state = ckpt[k]
                used_key = k
                break

        if state is None:
            
            tensor_like = [k for k, v in ckpt.items() if torch.is_tensor(v)]
            if len(tensor_like) > 0:
                state = ckpt
                used_key = "direct_state_dict"
            else:
                raise RuntimeError(
                    "No pude encontrar un state_dict dentro del checkpoint. "
                    "Revisa las keys impresas arriba."
                )

        print("State dict usado desde:", used_key)

    else:
        raise RuntimeError("El checkpoint no es un diccionario compatible.")

    print("Número de parámetros en state_dict original:", len(state))

    
    clean_state = {}

    for k, v in state.items():
        nk = k

        prefixes = [
            "module.",
            "model.",
            "backbone.",
        ]

        for prefix in prefixes:
            if nk.startswith(prefix):
                nk = nk[len(prefix):]

        clean_state[nk] = v

    print("Número de parámetros en state_dict limpio:", len(clean_state))

    print("\nPrimeras keys limpias:")
    for k in list(clean_state.keys())[:20]:
        shape = tuple(clean_state[k].shape) if torch.is_tensor(clean_state[k]) else type(clean_state[k])
        print(f"{k:60s} {shape}")

    return clean_state

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n" + "=" * 80)
print("CREANDO MODELO")
print("=" * 80)

print("Backbone:", CFG["model"]["backbone"])
print("Num classes:", CFG["model"]["num_classes"])
print("Img size:", CFG["model"]["img_size"])

model = timm.create_model(
    CFG["model"]["backbone"],
    pretrained=False,
    num_classes=CFG["model"]["num_classes"],
    img_size=CFG["model"]["img_size"],
)

print("Modelo creado correctamente.")


print("\n" + "=" * 80)
print("CARGANDO CHECKPOINT EN CPU")
print("=" * 80)

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu")
state = extract_and_clean_state_dict(ckpt)

print("\n" + "=" * 80)
print("CARGANDO PESOS EN EL MODELO")
print("=" * 80)

load_result = model.load_state_dict(state, strict=False)

missing = load_result.missing_keys
unexpected = load_result.unexpected_keys

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if len(missing) > 0:
    print("\nPrimeros missing keys:")
    for k in missing[:30]:
        print("-", k)

if len(unexpected) > 0:
    print("\nPrimeros unexpected keys:")
    for k in unexpected[:30]:
        print("-", k)


print("\n" + "=" * 80)
print("VERIFICACIONES")
print("=" * 80)


head_weight_keys = [k for k in state.keys() if "head.weight" in k or "classifier.weight" in k or "fc.weight" in k]

print("Keys candidatas de cabeza clasificadora:")
for k in head_weight_keys:
    try:
        print(k, tuple(state[k].shape))
    except Exception:
        print(k)


print("\n" + "=" * 80)
print("MOVIENDO MODELO A GPU")
print("=" * 80)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)
model.eval()

print("Modelo en:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria reservada GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
    print("Memoria asignada GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))

print("\n modelo cargado y listo.")

# **CELL 7 — Transformations, tiling and quick test in 1 image**

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import gc


IMG_SIZE = CFG["model"]["img_size"]
TILE_SIZE = CFG["tiling"]["tile_size"]
STRIDE = CFG["tiling"]["stride"]
TOPK_TEST = 20

print("=" * 80)
print("PARÁMETROS")
print("=" * 80)
print("IMG_SIZE:", IMG_SIZE)
print("TILE_SIZE:", TILE_SIZE)
print("STRIDE:", STRIDE)
print("TOPK_TEST:", TOPK_TEST)


MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def pil_to_tensor(img: Image.Image):
    """
    Convierte PIL RGB a tensor normalizado [3,H,W].
    """
    img = img.convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1)
    x = (x - MEAN) / STD
    return x


def resize_square_pil(img: Image.Image, size: int):
    """
    Redimensiona una imagen PIL a size x size.
    """
    return img.convert("RGB").resize((size, size), Image.BICUBIC)


def center_crop_square(img: Image.Image):
    """
    Extrae crop cuadrado central.
    """
    img = img.convert("RGB")
    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    return img.crop((left, top, left + side, top + side))


def compute_green_ratio_pil(img: Image.Image):
    """
    Heurística simple de vegetación.
    No es una segmentación perfecta, pero ayuda a detectar imágenes con baja vegetación.
    """
    arr = np.array(img.convert("RGB")).astype(np.float32)

    r = arr[..., 0]
    g = arr[..., 1]
    b = arr[..., 2]

    
    mask = (g > r * 1.05) & (g > b * 1.05) & (g > 35)

    return float(mask.mean())


def generate_tiles(img: Image.Image, tile_size=518, stride=259):
    """
    Genera tiles de tamaño tile_size con stride.
    Si la imagen es menor al tile, la redimensiona.
    """
    img = img.convert("RGB")
    w, h = img.size

    tiles = []

    if w < tile_size or h < tile_size:
        resized = resize_square_pil(img, tile_size)
        tiles.append({
            "view_type": "small_resized",
            "tile": resized,
            "x": 0,
            "y": 0,
            "green_ratio": compute_green_ratio_pil(resized),
        })
        return tiles

    xs = list(range(0, max(w - tile_size + 1, 1), stride))
    ys = list(range(0, max(h - tile_size + 1, 1), stride))

    if xs[-1] != w - tile_size:
        xs.append(w - tile_size)

    if ys[-1] != h - tile_size:
        ys.append(h - tile_size)

    for y in ys:
        for x in xs:
            crop = img.crop((x, y, x + tile_size, y + tile_size))
            tiles.append({
                "view_type": "tile",
                "tile": crop,
                "x": x,
                "y": y,
                "green_ratio": compute_green_ratio_pil(crop),
            })

    return tiles


def build_views_for_image(path):
    """
    Construye vistas para una imagen:
    - full image resized
    - center crop
    - dense tiles
    """
    img = Image.open(path).convert("RGB")

    views = []

    # Full image resized
    if CFG["tiling"]["include_full_image"]:
        full = resize_square_pil(img, IMG_SIZE)
        views.append({
            "view_type": "full",
            "tile": full,
            "x": -1,
            "y": -1,
            "green_ratio": compute_green_ratio_pil(full),
        })

    # Center crop
    if CFG["tiling"]["include_center_crop"]:
        cc = resize_square_pil(center_crop_square(img), IMG_SIZE)
        views.append({
            "view_type": "center",
            "tile": cc,
            "x": -1,
            "y": -1,
            "green_ratio": compute_green_ratio_pil(cc),
        })

    # Dense tiles
    tiles = generate_tiles(img, TILE_SIZE, STRIDE)
    views.extend(tiles)

   
    max_tiles = CFG["tiling"]["max_tiles_per_image"]

    if max_tiles is not None and len(views) > max_tiles:
        fixed = [v for v in views if v["view_type"] in ["full", "center"]]
        rest = [v for v in views if v["view_type"] not in ["full", "center"]]
        rest = sorted(rest, key=lambda d: d["green_ratio"], reverse=True)
        views = fixed + rest[:max_tiles - len(fixed)]

    return views


species_table_sorted = species_table.sort_values("species_id").reset_index(drop=True)
idx_to_species = dict(enumerate(species_table_sorted["species_id"].astype(int).tolist()))

print("\n" + "=" * 80)
print("MAPEO DE CLASES")
print("=" * 80)
print("Número de clases en mapeo:", len(idx_to_species))
print("Primeros 5 mapeos:")
for i in range(5):
    print(i, "->", idx_to_species[i])

print("Últimos 5 mapeos:")
for i in range(len(idx_to_species) - 5, len(idx_to_species)):
    print(i, "->", idx_to_species[i])


sample_row = test_df.iloc[0]
sample_qid = sample_row["quadrat_id"]
sample_path = sample_row["file_path"]

print("\n" + "=" * 80)
print("IMAGEN DE PRUEBA")
print("=" * 80)
print("quadrat_id:", sample_qid)
print("file_path:", sample_path)
print("Existe:", Path(sample_path).exists())

img_sample = Image.open(sample_path).convert("RGB")
print("Tamaño original:", img_sample.size)
print("Green ratio global aproximado:", round(compute_green_ratio_pil(img_sample), 4))


views = build_views_for_image(sample_path)

print("\n" + "=" * 80)
print("VISTAS GENERADAS")
print("=" * 80)
print("Número de vistas:", len(views))

views_summary = pd.DataFrame([
    {
        "view_id": i,
        "view_type": v["view_type"],
        "x": v["x"],
        "y": v["y"],
        "green_ratio": v["green_ratio"],
    }
    for i, v in enumerate(views)
])

display(views_summary.head(20))
display(views_summary["view_type"].value_counts().reset_index())


print("\n" + "=" * 80)
print("INFERENCIA DE PRUEBA")
print("=" * 80)

model.eval()

batch_size_test = min(8, len(views))
pred_rows = []

if torch.cuda.is_available():
    torch.cuda.empty_cache()

with torch.no_grad():
    for start in range(0, len(views), batch_size_test):
        batch_views = views[start:start + batch_size_test]

        x = torch.stack([
            pil_to_tensor(v["tile"]) for v in batch_views
        ], dim=0).to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", enabled=CFG["inference"]["amp"]):
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            vals, inds = torch.topk(probs, k=TOPK_TEST, dim=1)

        vals = vals.detach().cpu().numpy()
        inds = inds.detach().cpu().numpy()

        for i, v in enumerate(batch_views):
            view_global_id = start + i

            for rank in range(TOPK_TEST):
                class_index = int(inds[i, rank])
                species_id = int(idx_to_species[class_index])

                pred_rows.append({
                    "quadrat_id": sample_qid,
                    "view_id": view_global_id,
                    "view_type": v["view_type"],
                    "green_ratio": v["green_ratio"],
                    "rank": rank + 1,
                    "class_index": class_index,
                    "species_id": species_id,
                    "score": float(vals[i, rank]),
                })

        del x, logits, probs, vals, inds

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pred_test = pd.DataFrame(pred_rows)

print("Predicciones generadas:", pred_test.shape)

print("\nMemoria GPU:")
print("Asignada GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reservada GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

display(pred_test.head(20))


agg_test = (
    pred_test
    .groupby("species_id")
    .agg(
        mean_score=("score", "mean"),
        max_score=("score", "max"),
        n_views=("view_id", "nunique"),
        mean_green=("green_ratio", "mean"),
    )
    .reset_index()
)

agg_test["final_score_simple"] = (
    0.6 * agg_test["max_score"] +
    0.3 * agg_test["mean_score"] +
    0.1 * (agg_test["n_views"] / len(views))
)

agg_test = agg_test.sort_values("final_score_simple", ascending=False).reset_index(drop=True)

# Añadir nombres científicos
agg_test = agg_test.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left"
)

print("\n" + "=" * 80)
print("TOP ESPECIES AGREGADAS PARA LA IMAGEN DE PRUEBA")
print("=" * 80)

display(agg_test.head(20))

print("\n Prueba rápida de inferencia correcta.")

# **CELL 8 — Benchmark without multiprocessing**

In [ ]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


required_objects = [
    "test_df",
    "species_table",
    "model",
    "DEVICE",
    "CFG",
    "RUN_DIR",
    "pil_to_tensor",
    "resize_square_pil",
    "center_crop_square",
    "IMG_SIZE",
    "TILE_SIZE",
    "idx_to_species",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise RuntimeError(
        "Faltan objetos en memoria. Asegúrate de haber ejecutado las celdas 0 a 5. "
        f"Faltan: {missing}"
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Revisa el kernel GPU.")

model.eval()


N_BENCHMARK_IMAGES = 30
TOPK = CFG["inference"]["topk_per_view"]


SAFE_BATCH_SIZE = 8
SAFE_NUM_WORKERS = 0
SAFE_PIN_MEMORY = False

benchmark_dir = RUN_DIR / "benchmark_30_images"
benchmark_dir.mkdir(parents=True, exist_ok=True)

views_bench_path = benchmark_dir / "views_benchmark.csv"

print("=" * 80)
print("CONFIGURACIÓN BENCHMARK SEGURO")
print("=" * 80)
print("N_BENCHMARK_IMAGES:", N_BENCHMARK_IMAGES)
print("TOPK:", TOPK)
print("SAFE_BATCH_SIZE:", SAFE_BATCH_SIZE)
print("SAFE_NUM_WORKERS:", SAFE_NUM_WORKERS)
print("SAFE_PIN_MEMORY:", SAFE_PIN_MEMORY)
print("benchmark_dir:", benchmark_dir)


if "views_bench" in globals() and isinstance(views_bench, pd.DataFrame) and len(views_bench) > 0:
    print("\nUsando views_bench que ya está en memoria.")
    views_bench = views_bench.copy()
elif views_bench_path.exists():
    print("\nCargando vistas desde archivo:")
    print(views_bench_path)
    views_bench = pd.read_csv(views_bench_path)
else:
    print("\nNo encontré views_bench en memoria ni archivo. Construyendo vistas...")
    
    idxs = np.linspace(0, len(test_df) - 1, N_BENCHMARK_IMAGES).astype(int)
    test_bench = test_df.iloc[idxs].reset_index(drop=True).copy()
    
    views_rows = []
    
    for row in tqdm(test_bench.itertuples(index=False), total=len(test_bench), desc="build_views_benchmark"):
        qid = row.quadrat_id
        path = row.file_path
        
        try:
            views = build_views_for_image(path)
            
            for j, v in enumerate(views):
                views_rows.append({
                    "quadrat_id": qid,
                    "file_path": path,
                    "view_id": j,
                    "view_type": v["view_type"],
                    "x": v["x"],
                    "y": v["y"],
                    "green_ratio": float(v["green_ratio"]),
                })
        except Exception as e:
            print("Error construyendo vistas:", qid, path, e)
    
    views_bench = pd.DataFrame(views_rows)
    views_bench.to_csv(views_bench_path, index=False)

print("\n" + "=" * 80)
print("VISTAS BENCHMARK")
print("=" * 80)
print("Vistas totales:", len(views_bench))

views_per_image = views_bench.groupby("quadrat_id").size().reset_index(name="n_views")
display(views_per_image.describe())

display(views_bench["view_type"].value_counts().reset_index(name="count"))


# Dataset

class PlantCLEFViewsDatasetSafe(Dataset):
    def __init__(self, views_df):
        self.df = views_df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        
        img = Image.open(r["file_path"]).convert("RGB")
        view_type = r["view_type"]
        
        if view_type == "full":
            view = resize_square_pil(img, IMG_SIZE)
        
        elif view_type == "center":
            view = resize_square_pil(center_crop_square(img), IMG_SIZE)
        
        elif view_type == "small_resized":
            view = resize_square_pil(img, IMG_SIZE)
        
        elif view_type == "tile":
            x0 = int(r["x"])
            y0 = int(r["y"])
            view = img.crop((x0, y0, x0 + TILE_SIZE, y0 + TILE_SIZE))
            view = resize_square_pil(view, IMG_SIZE)
        
        else:
            view = resize_square_pil(img, IMG_SIZE)
        
        x_tensor = pil_to_tensor(view)
        
        return {
            "image": x_tensor,
            "quadrat_id": r["quadrat_id"],
            "view_id": int(r["view_id"]),
            "view_type": view_type,
            "green_ratio": float(r["green_ratio"]),
        }


def collate_fn_safe(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)
    
    return {
        "image": images,
        "quadrat_id": [b["quadrat_id"] for b in batch],
        "view_id": [b["view_id"] for b in batch],
        "view_type": [b["view_type"] for b in batch],
        "green_ratio": [b["green_ratio"] for b in batch],
    }


bench_dataset = PlantCLEFViewsDatasetSafe(views_bench)

bench_loader = DataLoader(
    bench_dataset,
    batch_size=SAFE_BATCH_SIZE,
    shuffle=False,
    num_workers=SAFE_NUM_WORKERS,
    pin_memory=SAFE_PIN_MEMORY,
    collate_fn=collate_fn_safe,
)

print("\n" + "=" * 80)
print("DATALOADER SEGURO")
print("=" * 80)
print("Dataset views:", len(bench_dataset))
print("Batches:", len(bench_loader))


# Inferencia benchmark

pred_rows = []

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start_infer = time.time()

print("\n" + "=" * 80)
print("INFERENCIA BENCHMARK SEGURO")
print("=" * 80)

with torch.no_grad():
    for batch in tqdm(bench_loader, desc="infer_benchmark_safe"):
        images = batch["image"].to(DEVICE, non_blocking=False)
        
        with torch.autocast(device_type="cuda", enabled=CFG["inference"]["amp"]):
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            vals, inds = torch.topk(probs, k=TOPK, dim=1)
        
        vals_np = vals.detach().cpu().numpy()
        inds_np = inds.detach().cpu().numpy()
        
        for i in range(len(batch["quadrat_id"])):
            qid = batch["quadrat_id"][i]
            view_id = batch["view_id"][i]
            view_type = batch["view_type"][i]
            green_ratio = batch["green_ratio"][i]
            
            for rank in range(TOPK):
                class_index = int(inds_np[i, rank])
                species_id = int(idx_to_species[class_index])
                
                pred_rows.append({
                    "quadrat_id": qid,
                    "view_id": view_id,
                    "view_type": view_type,
                    "green_ratio": green_ratio,
                    "rank": rank + 1,
                    "class_index": class_index,
                    "species_id": species_id,
                    "score": float(vals_np[i, rank]),
                })
        
        del images, logits, probs, vals, inds, vals_np, inds_np

infer_time = time.time() - start_infer

pred_bench = pd.DataFrame(pred_rows)

pred_bench_path = benchmark_dir / "view_topk_predictions_benchmark_safe.csv"
pred_bench.to_csv(pred_bench_path, index=False)

print("\n" + "=" * 80)
print("RESULTADOS BENCHMARK SEGURO")
print("=" * 80)

print("Predicciones shape:", pred_bench.shape)
print("Guardado:", pred_bench_path)
print("Tiempo inferencia:", round(infer_time / 60, 2), "min")

print("\nMemoria GPU:")
print("Asignada actual GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
print("Reservada actual GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
print("Pico asignado GB:", round(torch.cuda.max_memory_allocated() / 1024**3, 3))
print("Pico reservado GB:", round(torch.cuda.max_memory_reserved() / 1024**3, 3))

print("\nPrimeras predicciones:")
display(pred_bench.head(20))

print("\nResumen score:")
display(pred_bench["score"].describe())

print("\nPredicciones por quadrat:")
display(
    pred_bench.groupby("quadrat_id")
    .agg(
        n_rows=("score", "size"),
        n_views=("view_id", "nunique"),
        mean_green=("green_ratio", "mean"),
        max_score=("score", "max"),
        mean_score=("score", "mean"),
    )
    .reset_index()
    .head()
)

gc.collect()
torch.cuda.empty_cache()

print("\n benchmark seguro listo.")

# **CELL 9 — Speed ​​test to choose optimal batch_size**

In [ ]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


required_objects = [
    "views_bench",
    "model",
    "DEVICE",
    "CFG",
    "pil_to_tensor",
    "resize_square_pil",
    "center_crop_square",
    "IMG_SIZE",
    "TILE_SIZE",
    "idx_to_species",
    "test_df",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise RuntimeError(
        "Faltan objetos en memoria. Debes haber ejecutado hasta la CELDA 6B. "
        f"Faltan: {missing}"
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible.")

model.eval()


TOPK = CFG["inference"]["topk_per_view"]


N_SPEED_VIEWS = min(512, len(views_bench))


BATCH_SIZES_TO_TEST = [8, 16, 24, 32, 48, 64]

views_speed = views_bench.iloc[:N_SPEED_VIEWS].reset_index(drop=True).copy()

print("=" * 80)
print("SPEED TEST")
print("=" * 80)
print("N_SPEED_VIEWS:", len(views_speed))
print("TOPK:", TOPK)
print("Batches a probar:", BATCH_SIZES_TO_TEST)
print("num_workers: 0")
print("pin_memory: False")


class PlantCLEFViewsDatasetSpeed(Dataset):
    def __init__(self, views_df):
        self.df = views_df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        
        img = Image.open(r["file_path"]).convert("RGB")
        view_type = r["view_type"]
        
        if view_type == "full":
            view = resize_square_pil(img, IMG_SIZE)
        
        elif view_type == "center":
            view = resize_square_pil(center_crop_square(img), IMG_SIZE)
        
        elif view_type == "small_resized":
            view = resize_square_pil(img, IMG_SIZE)
        
        elif view_type == "tile":
            x0 = int(r["x"])
            y0 = int(r["y"])
            view = img.crop((x0, y0, x0 + TILE_SIZE, y0 + TILE_SIZE))
            view = resize_square_pil(view, IMG_SIZE)
        
        else:
            view = resize_square_pil(img, IMG_SIZE)
        
        x_tensor = pil_to_tensor(view)
        
        return {
            "image": x_tensor,
            "quadrat_id": r["quadrat_id"],
            "view_id": int(r["view_id"]),
            "view_type": view_type,
            "green_ratio": float(r["green_ratio"]),
        }


def collate_fn_speed(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)
    
    return {
        "image": images,
        "quadrat_id": [b["quadrat_id"] for b in batch],
        "view_id": [b["view_id"] for b in batch],
        "view_type": [b["view_type"] for b in batch],
        "green_ratio": [b["green_ratio"] for b in batch],
    }


speed_dataset = PlantCLEFViewsDatasetSpeed(views_speed)


results = []

for bs in BATCH_SIZES_TO_TEST:
    print("\n" + "=" * 80)
    print(f"Probando batch_size={bs}")
    print("=" * 80)
    
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    loader = DataLoader(
        speed_dataset,
        batch_size=bs,
        shuffle=False,
        num_workers=0,
        pin_memory=False,
        collate_fn=collate_fn_speed,
    )
    
    start = time.time()
    ok = True
    error_msg = ""
    n_views_done = 0
    
    try:
        with torch.no_grad():
            for batch in tqdm(loader, desc=f"speed_bs_{bs}"):
                images = batch["image"].to(DEVICE, non_blocking=False)
                
                with torch.autocast(device_type="cuda", enabled=CFG["inference"]["amp"]):
                    logits = model(images)
                    probs = F.softmax(logits, dim=1)
                    vals, inds = torch.topk(probs, k=TOPK, dim=1)
                
                # Simulamos transferencia a CPU como en inferencia real
                vals_np = vals.detach().cpu().numpy()
                inds_np = inds.detach().cpu().numpy()
                
                n_views_done += images.shape[0]
                
                del images, logits, probs, vals, inds, vals_np, inds_np
    
    except RuntimeError as e:
        ok = False
        error_msg = str(e)
        
        if "out of memory" in error_msg.lower():
            print(" OOM con batch_size:", bs)
        else:
            print(" Error con batch_size:", bs)
            print(error_msg[:500])
    
    elapsed = time.time() - start
    
    peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
    
    sec_per_view = elapsed / max(n_views_done, 1)
    views_per_sec = n_views_done / max(elapsed, 1e-9)
    
    results.append({
        "batch_size": bs,
        "ok": ok,
        "n_views_done": n_views_done,
        "elapsed_min": elapsed / 60,
        "sec_per_view": sec_per_view,
        "views_per_sec": views_per_sec,
        "peak_alloc_gb": peak_alloc,
        "peak_reserved_gb": peak_reserved,
        "error": error_msg[:200],
    })
    
    print(f"ok: {ok}")
    print(f"n_views_done: {n_views_done}")
    print(f"elapsed_min: {elapsed/60:.3f}")
    print(f"views_per_sec: {views_per_sec:.3f}")
    print(f"peak_alloc_gb: {peak_alloc:.3f}")
    print(f"peak_reserved_gb: {peak_reserved:.3f}")
    
    gc.collect()
    torch.cuda.empty_cache()

speed_results = pd.DataFrame(results)

print("\n" + "=" * 80)
print("RESULTADOS SPEED TEST")
print("=" * 80)
display(speed_results)


mean_views_per_image = views_bench.groupby("quadrat_id").size().mean()
estimated_total_views = int(round(len(test_df) * mean_views_per_image))

print("\n" + "=" * 80)
print("ESTIMACIÓN TEST COMPLETO")
print("=" * 80)
print("Imágenes test:", len(test_df))
print("Vistas promedio por imagen benchmark:", round(mean_views_per_image, 2))
print("Vistas estimadas test completo:", estimated_total_views)

ok_results = speed_results[speed_results["ok"]].copy()

if len(ok_results) > 0:
    ok_results["estimated_full_hours"] = (
        ok_results["sec_per_view"] * estimated_total_views / 3600
    )
    
    print("\nEstimación por batch_size:")
    display(
        ok_results[
            [
                "batch_size",
                "views_per_sec",
                "peak_alloc_gb",
                "peak_reserved_gb",
                "estimated_full_hours",
            ]
        ].sort_values("estimated_full_hours")
    )
    
    best = ok_results.sort_values("estimated_full_hours").iloc[0]
    
    print("\n" + "=" * 80)
    print("RECOMENDACIÓN")
    print("=" * 80)
    print("Batch recomendado:", int(best["batch_size"]))
    print("Tiempo estimado horas:", round(best["estimated_full_hours"], 2))
    print("Pico reservado GB:", round(best["peak_reserved_gb"], 3))
else:
    print("No hubo ningún batch exitoso.")

print("\n Finalizado.")

# **CELL 10 — Complete inference test_2025 by chunks**

In [ ]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

required_objects = [
    "test_df",
    "species_table",
    "model",
    "DEVICE",
    "CFG",
    "RUN_DIR",
    "pil_to_tensor",
    "resize_square_pil",
    "center_crop_square",
    "build_views_for_image",
    "IMG_SIZE",
    "TILE_SIZE",
    "idx_to_species",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise RuntimeError(
        "Faltan objetos en memoria. Debes haber ejecutado las celdas 0 a 6C. "
        f"Faltan: {missing}"
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Revisa el kernel GPU.")

model.eval()


TOPK = CFG["inference"]["topk_per_view"]


FULL_BATCH_SIZE = 32
FULL_NUM_WORKERS = 0
FULL_PIN_MEMORY = False


CHUNK_SIZE_IMAGES = 50

FULL_INFER_DIR = RUN_DIR / "full_test_inference"
CHUNKS_DIR = FULL_INFER_DIR / "chunks"
VIEWS_DIR = FULL_INFER_DIR / "views"

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
VIEWS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CONFIGURACIÓN INFERENCIA COMPLETA")
print("=" * 80)
print("Test imágenes:", len(test_df))
print("TOPK:", TOPK)
print("FULL_BATCH_SIZE:", FULL_BATCH_SIZE)
print("FULL_NUM_WORKERS:", FULL_NUM_WORKERS)
print("FULL_PIN_MEMORY:", FULL_PIN_MEMORY)
print("CHUNK_SIZE_IMAGES:", CHUNK_SIZE_IMAGES)
print("FULL_INFER_DIR:", FULL_INFER_DIR)
print("CHUNKS_DIR:", CHUNKS_DIR)
print("VIEWS_DIR:", VIEWS_DIR)


class PlantCLEFViewsDatasetFull(Dataset):
    def __init__(self, views_df):
        self.df = views_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        img = Image.open(r["file_path"]).convert("RGB")
        view_type = r["view_type"]

        if view_type == "full":
            view = resize_square_pil(img, IMG_SIZE)

        elif view_type == "center":
            view = resize_square_pil(center_crop_square(img), IMG_SIZE)

        elif view_type == "small_resized":
            view = resize_square_pil(img, IMG_SIZE)

        elif view_type == "tile":
            x0 = int(r["x"])
            y0 = int(r["y"])
            view = img.crop((x0, y0, x0 + TILE_SIZE, y0 + TILE_SIZE))
            view = resize_square_pil(view, IMG_SIZE)

        else:
            view = resize_square_pil(img, IMG_SIZE)

        x_tensor = pil_to_tensor(view)

        return {
            "image": x_tensor,
            "quadrat_id": r["quadrat_id"],
            "view_id": int(r["view_id"]),
            "view_type": view_type,
            "green_ratio": float(r["green_ratio"]),
        }


def collate_fn_full(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)

    return {
        "image": images,
        "quadrat_id": [b["quadrat_id"] for b in batch],
        "view_id": [b["view_id"] for b in batch],
        "view_type": [b["view_type"] for b in batch],
        "green_ratio": [b["green_ratio"] for b in batch],
    }


def build_views_for_chunk(test_chunk: pd.DataFrame, chunk_id: int):
    rows = []

    for row in tqdm(
        test_chunk.itertuples(index=False),
        total=len(test_chunk),
        desc=f"build_views_chunk_{chunk_id:04d}"
    ):
        qid = row.quadrat_id
        path = row.file_path

        try:
            views = build_views_for_image(path)

            for j, v in enumerate(views):
                rows.append({
                    "chunk_id": chunk_id,
                    "quadrat_id": qid,
                    "file_path": path,
                    "view_id": j,
                    "view_type": v["view_type"],
                    "x": v["x"],
                    "y": v["y"],
                    "green_ratio": float(v["green_ratio"]),
                })

        except Exception as e:
            print(f"❌ Error construyendo vistas | chunk={chunk_id} | qid={qid} | {e}")

    return pd.DataFrame(rows)



def infer_views_chunk(views_chunk: pd.DataFrame, chunk_id: int):
    dataset = PlantCLEFViewsDatasetFull(views_chunk)

    loader = DataLoader(
        dataset,
        batch_size=FULL_BATCH_SIZE,
        shuffle=False,
        num_workers=FULL_NUM_WORKERS,
        pin_memory=FULL_PIN_MEMORY,
        collate_fn=collate_fn_full,
    )

    pred_rows = []

    model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"infer_chunk_{chunk_id:04d}"):
            images = batch["image"].to(DEVICE, non_blocking=False)

            with torch.autocast(device_type="cuda", enabled=CFG["inference"]["amp"]):
                logits = model(images)
                probs = F.softmax(logits, dim=1)
                vals, inds = torch.topk(probs, k=TOPK, dim=1)

            vals_np = vals.detach().cpu().numpy()
            inds_np = inds.detach().cpu().numpy()

            for i in range(len(batch["quadrat_id"])):
                qid = batch["quadrat_id"][i]
                view_id = batch["view_id"][i]
                view_type = batch["view_type"][i]
                green_ratio = batch["green_ratio"][i]

                for rank in range(TOPK):
                    class_index = int(inds_np[i, rank])
                    species_id = int(idx_to_species[class_index])

                    pred_rows.append({
                        "chunk_id": chunk_id,
                        "quadrat_id": qid,
                        "view_id": view_id,
                        "view_type": view_type,
                        "green_ratio": green_ratio,
                        "rank": rank + 1,
                        "class_index": class_index,
                        "species_id": species_id,
                        "score": float(vals_np[i, rank]),
                    })

            del images, logits, probs, vals, inds, vals_np, inds_np

    return pd.DataFrame(pred_rows)



test_df_full = test_df.copy().reset_index(drop=True)

n_images = len(test_df_full)
n_chunks = int(np.ceil(n_images / CHUNK_SIZE_IMAGES))

print("\n" + "=" * 80)
print("PLAN DE CHUNKS")
print("=" * 80)
print("n_images:", n_images)
print("n_chunks:", n_chunks)

chunk_plan = []

for chunk_id in range(n_chunks):
    start_idx = chunk_id * CHUNK_SIZE_IMAGES
    end_idx = min((chunk_id + 1) * CHUNK_SIZE_IMAGES, n_images)

    pred_path = CHUNKS_DIR / f"predictions_chunk_{chunk_id:04d}.csv"
    views_path = VIEWS_DIR / f"views_chunk_{chunk_id:04d}.csv"

    chunk_plan.append({
        "chunk_id": chunk_id,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "n_images": end_idx - start_idx,
        "views_path": views_path,
        "pred_path": pred_path,
        "done": pred_path.exists(),
    })

chunk_plan_df = pd.DataFrame(chunk_plan)

display(chunk_plan_df.head())
print("Chunks ya completados:", chunk_plan_df["done"].sum(), "/", len(chunk_plan_df))

# Guardar plan
chunk_plan_path = FULL_INFER_DIR / "chunk_plan.csv"
chunk_plan_df.to_csv(chunk_plan_path, index=False)
print("Plan guardado:", chunk_plan_path)


start_total = time.time()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

completed_now = 0
skipped = 0

for item in chunk_plan:
    chunk_id = item["chunk_id"]
    start_idx = item["start_idx"]
    end_idx = item["end_idx"]
    views_path = item["views_path"]
    pred_path = item["pred_path"]

    print("\n" + "#" * 100)
    print(f"CHUNK {chunk_id:04d}/{n_chunks-1:04d} | imágenes {start_idx}:{end_idx}")
    print("#" * 100)

    if pred_path.exists():
        print(" Chunk ya existe. Se omite:")
        print(pred_path)
        skipped += 1
        continue

    test_chunk = test_df_full.iloc[start_idx:end_idx].reset_index(drop=True).copy()

    # Construir o cargar vistas del chunk
    if views_path.exists():
        print("Cargando vistas existentes:")
        print(views_path)
        views_chunk = pd.read_csv(views_path)
    else:
        views_chunk = build_views_for_chunk(test_chunk, chunk_id)
        views_chunk.to_csv(views_path, index=False)
        print("Vistas guardadas:", views_path)

    print("Imágenes en chunk:", len(test_chunk))
    print("Vistas en chunk:", len(views_chunk))

    if len(views_chunk) == 0:
        print(" Chunk sin vistas. Se continúa.")
        continue

  
    chunk_start_time = time.time()

    pred_chunk = infer_views_chunk(views_chunk, chunk_id)

    chunk_time = time.time() - chunk_start_time

   
    pred_chunk.to_csv(pred_path, index=False)

    print("\n Chunk finalizado:", chunk_id)
    print("Predicciones:", pred_chunk.shape)
    print("Guardado:", pred_path)
    print("Tiempo chunk inferencia:", round(chunk_time / 60, 2), "min")

  
    print("GPU asignada GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("GPU reservada GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
    print("GPU pico asignado GB:", round(torch.cuda.max_memory_allocated() / 1024**3, 3))
    print("GPU pico reservado GB:", round(torch.cuda.max_memory_reserved() / 1024**3, 3))

    completed_now += 1

   
    del views_chunk, pred_chunk, test_chunk
    gc.collect()
    torch.cuda.empty_cache()

    
    chunk_plan_df.loc[chunk_plan_df["chunk_id"] == chunk_id, "done"] = True
    chunk_plan_df.to_csv(chunk_plan_path, index=False)

elapsed_total = time.time() - start_total

print("\n" + "=" * 80)
print("INFERENCIA COMPLETA / ESTADO FINAL")
print("=" * 80)


existing_pred_chunks = sorted(CHUNKS_DIR.glob("predictions_chunk_*.csv"))

print("Chunks existentes:", len(existing_pred_chunks), "/", n_chunks)
print("Chunks completados en esta corrida:", completed_now)
print("Chunks omitidos porque ya existían:", skipped)
print("Tiempo de esta corrida:", round(elapsed_total / 3600, 2), "horas")

print("\nArchivos generados en:")
print(CHUNKS_DIR)

if len(existing_pred_chunks) == n_chunks:
    print("\n Todos los chunks están completos.")
else:
    print("\n Aún faltan chunks. Puedes volver a ejecutar esta misma celda y continuará donde quedó.")

print("\n Finalizado.")

# **CELL 11 — Aggregation by quadrat/species**

In [ ]:
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


FULL_INFER_DIR = RUN_DIR / "full_test_inference"
CHUNKS_DIR = FULL_INFER_DIR / "chunks"
AGG_DIR = RUN_DIR / "aggregation"
AGG_CHUNKS_DIR = AGG_DIR / "agg_chunks"

AGG_DIR.mkdir(parents=True, exist_ok=True)
AGG_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

pred_chunk_files = sorted(CHUNKS_DIR.glob("predictions_chunk_*.csv"))

print("=" * 80)
print("AGREGACIÓN INTELIGENTE")
print("=" * 80)

print("CHUNKS_DIR:", CHUNKS_DIR)
print("Chunks encontrados:", len(pred_chunk_files))

if len(pred_chunk_files) == 0:
    raise FileNotFoundError(f"No encontré chunks en {CHUNKS_DIR}")


AGG_TOPK = CFG["aggregation"]["agg_topk"]


W_MEAN_TOP = 0.50
W_MAX = 0.25
W_VEG = 0.15
W_FREQ_BONUS = 0.10

print("\nParámetros:")
print("AGG_TOPK:", AGG_TOPK)
print("W_MEAN_TOP:", W_MEAN_TOP)
print("W_MAX:", W_MAX)
print("W_VEG:", W_VEG)
print("W_FREQ_BONUS:", W_FREQ_BONUS)


def aggregate_prediction_chunk(df_pred: pd.DataFrame, chunk_id: int):
    """
    Agrega predicciones top-k por vista/tile a nivel quadrat-species.
    """
    df = df_pred.copy()

    
    df["quadrat_id"] = df["quadrat_id"].astype(str)
    df["species_id"] = df["species_id"].astype(int)
    df["view_id"] = df["view_id"].astype(int)
    df["score"] = df["score"].astype(float)
    df["green_ratio"] = df["green_ratio"].astype(float)


    unique_views = (
        df[["quadrat_id", "view_id", "view_type", "green_ratio"]]
        .drop_duplicates()
        .copy()
    )

    q_stats = (
        unique_views
        .groupby("quadrat_id")
        .agg(
            n_views_total=("view_id", "nunique"),
            global_green=("green_ratio", "mean"),
            max_green=("green_ratio", "max"),
            min_green=("green_ratio", "min"),
            std_green=("green_ratio", "std"),
            pct_low_green_views=("green_ratio", lambda x: float((x < 0.05).mean())),
            pct_high_green_views=("green_ratio", lambda x: float((x >= 0.15).mean())),
        )
        .reset_index()
    )

    q_stats["std_green"] = q_stats["std_green"].fillna(0.0)


    base = (
        df
        .groupby(["quadrat_id", "species_id"])
        .agg(
            n_hits=("score", "size"),
            n_views_species=("view_id", "nunique"),
            mean_score_all=("score", "mean"),
            max_score=("score", "max"),
            min_rank=("rank", "min"),
            mean_rank=("rank", "mean"),
            mean_green_species=("green_ratio", "mean"),
        )
        .reset_index()
    )


    df_sorted = df.sort_values(
        ["quadrat_id", "species_id", "score"],
        ascending=[True, True, False]
    )

    top_scores = (
        df_sorted
        .groupby(["quadrat_id", "species_id"])
        .head(AGG_TOPK)
        .groupby(["quadrat_id", "species_id"])
        .agg(
            mean_top_score=("score", "mean"),
            sum_top_score=("score", "sum"),
        )
        .reset_index()
    )


    df["veg_weight"] = 0.5 + df["green_ratio"].clip(0, 1)
    df["score_x_veg_weight"] = df["score"] * df["veg_weight"]

    veg = (
        df
        .groupby(["quadrat_id", "species_id"])
        .agg(
            weighted_score_sum=("score_x_veg_weight", "sum"),
            veg_weight_sum=("veg_weight", "sum"),
        )
        .reset_index()
    )

    veg["veg_weighted_score"] = (
        veg["weighted_score_sum"] / veg["veg_weight_sum"].replace(0, np.nan)
    ).fillna(0.0)

    veg = veg[["quadrat_id", "species_id", "veg_weighted_score"]]


    view_support = (
        df
        .groupby(["quadrat_id", "species_id", "view_type"])
        .agg(
            n_views_type=("view_id", "nunique"),
            max_score_type=("score", "max"),
        )
        .reset_index()
    )

    
    view_pivot = view_support.pivot_table(
        index=["quadrat_id", "species_id"],
        columns="view_type",
        values="n_views_type",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    
    for c in ["full", "center", "tile", "small_resized"]:
        if c not in view_pivot.columns:
            view_pivot[c] = 0

    view_pivot = view_pivot.rename(columns={
        "full": "n_full_hits",
        "center": "n_center_hits",
        "tile": "n_tile_hits",
        "small_resized": "n_small_resized_hits",
    })


    agg = base.merge(
        top_scores,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        veg,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        view_pivot,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        q_stats,
        on="quadrat_id",
        how="left"
    )

    
    fill_cols = [
        "mean_top_score",
        "sum_top_score",
        "veg_weighted_score",
        "n_full_hits",
        "n_center_hits",
        "n_tile_hits",
        "n_small_resized_hits",
    ]

    for c in fill_cols:
        agg[c] = agg[c].fillna(0)


    agg["freq_score"] = (
        agg["n_views_species"] / agg["n_views_total"].replace(0, np.nan)
    ).fillna(0.0)

    agg["freq_bonus"] = agg["freq_score"] * agg["max_score"]


    agg["context_bonus"] = (
        (agg["n_full_hits"] > 0).astype(float) * 0.015
        + (agg["n_center_hits"] > 0).astype(float) * 0.010
    )


    agg["single_view_penalty"] = np.where(
        (agg["n_views_species"] <= 1) & (agg["max_score"] < 0.30),
        0.85,
        1.00
    )


    agg["final_score_raw"] = (
        W_MEAN_TOP * agg["mean_top_score"]
        + W_MAX * agg["max_score"]
        + W_VEG * agg["veg_weighted_score"]
        + W_FREQ_BONUS * agg["freq_bonus"]
        + agg["context_bonus"]
    )

    agg["final_score"] = agg["final_score_raw"] * agg["single_view_penalty"]

    agg["chunk_id"] = chunk_id


    agg = agg.sort_values(
        ["quadrat_id", "final_score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    return agg, q_stats


start_total = time.time()

agg_chunk_paths = []
q_stats_all = []

for pred_path in tqdm(pred_chunk_files, desc="aggregate_chunks"):
    chunk_id = int(pred_path.stem.split("_")[-1])

    out_path = AGG_CHUNKS_DIR / f"agg_chunk_{chunk_id:04d}.pkl.gz"

    if out_path.exists():
        print(f" Agg chunk ya existe, se omite: {out_path.name}")
        agg_chunk_paths.append(out_path)
        continue

    print("\n" + "=" * 80)
    print(f"Procesando chunk {chunk_id:04d}")
    print("=" * 80)
    print("Leyendo:", pred_path)

    df_pred = pd.read_csv(pred_path)

    print("Pred shape:", df_pred.shape)
    print("Quadrats:", df_pred["quadrat_id"].nunique())

    agg_chunk, q_stats = aggregate_prediction_chunk(df_pred, chunk_id)

    agg_chunk.to_pickle(out_path, compression="gzip")

    agg_chunk_paths.append(out_path)
    q_stats_all.append(q_stats)

    print("Agg shape:", agg_chunk.shape)
    print("Guardado:", out_path)

    del df_pred, agg_chunk, q_stats
    gc.collect()

elapsed = time.time() - start_total

print("\n" + "=" * 80)
print("AGREGACIÓN POR CHUNKS FINALIZADA")
print("=" * 80)
print("Agg chunks:", len(agg_chunk_paths))
print("Tiempo:", round(elapsed / 60, 2), "min")


print("\n" + "=" * 80)
print("COMBINANDO AGG CHUNKS")
print("=" * 80)

agg_parts = []

for p in tqdm(sorted(AGG_CHUNKS_DIR.glob("agg_chunk_*.pkl.gz")), desc="load_agg_chunks"):
    agg_parts.append(pd.read_pickle(p, compression="gzip"))

agg_df = pd.concat(agg_parts, ignore_index=True)


agg_df = agg_df.sort_values(
    ["quadrat_id", "final_score"],
    ascending=[True, False]
).reset_index(drop=True)

agg_out_path = AGG_DIR / "quadrat_species_scores.pkl.gz"
agg_csv_sample_path = AGG_DIR / "quadrat_species_scores_top100_per_quadrat.csv"

agg_df.to_pickle(agg_out_path, compression="gzip")


agg_top100 = (
    agg_df
    .groupby("quadrat_id", group_keys=False)
    .head(100)
    .copy()
)

agg_top100.to_csv(agg_csv_sample_path, index=False)

print("Agg total shape:", agg_df.shape)
print("Quadrats:", agg_df["quadrat_id"].nunique())
print("Guardado pkl:", agg_out_path)
print("Guardado CSV top100:", agg_csv_sample_path)


q_summary = (
    agg_df
    .groupby("quadrat_id")
    .agg(
        n_candidate_species=("species_id", "nunique"),
        global_green=("global_green", "first"),
        max_green=("max_green", "first"),
        pct_low_green_views=("pct_low_green_views", "first"),
        pct_high_green_views=("pct_high_green_views", "first"),
        n_views_total=("n_views_total", "first"),
        top1_score=("final_score", "max"),
    )
    .reset_index()
)


top2_scores = (
    agg_df
    .groupby("quadrat_id")
    .head(2)
    .groupby("quadrat_id")["final_score"]
    .apply(list)
    .reset_index(name="top_scores")
)

def get_top2_margin(scores):
    if len(scores) < 2:
        return np.nan
    return scores[0] - scores[1]

def get_top2_ratio(scores):
    if len(scores) < 2 or scores[0] == 0:
        return np.nan
    return scores[1] / scores[0]

top2_scores["top2_margin"] = top2_scores["top_scores"].apply(get_top2_margin)
top2_scores["top2_ratio"] = top2_scores["top_scores"].apply(get_top2_ratio)

q_summary = q_summary.merge(
    top2_scores[["quadrat_id", "top2_margin", "top2_ratio"]],
    on="quadrat_id",
    how="left"
)

q_summary_path = AGG_DIR / "quadrat_quality_summary.csv"
q_summary.to_csv(q_summary_path, index=False)

print("\nQuality summary guardado:")
print(q_summary_path)

print("\nResumen quality:")
display(q_summary.describe())

print("\nTop de agg_df:")
display(agg_df.head(20))

print("\nDistribución de candidatos por quadrat:")
display(q_summary["n_candidate_species"].describe())

print("\n Scores agregados listos para adaptive K.")

# **CELL 12 — Generate candidate submissions with adaptive K**

In [ ]:
import ast
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"
AGG_DIR = RUN_DIR / "aggregation"
CAND_DIR = RUN_DIR / "submission_candidates"

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
CAND_DIR.mkdir(parents=True, exist_ok=True)

agg_path = AGG_DIR / "quadrat_species_scores.pkl.gz"
q_summary_path = AGG_DIR / "quadrat_quality_summary.csv"

test_path = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
species_table_path = INDEX_DIR / "species_table.pkl.gz"

print("=" * 80)
print("CARGANDO ARCHIVOS")
print("=" * 80)

for p in [agg_path, q_summary_path, test_path, species_table_path]:
    print(p, "| existe:", p.exists())

if not agg_path.exists():
    raise FileNotFoundError(agg_path)

if not q_summary_path.exists():
    raise FileNotFoundError(q_summary_path)

agg_df = pd.read_pickle(agg_path, compression="gzip")
q_summary = pd.read_csv(q_summary_path)

test_df = pd.read_pickle(test_path, compression="gzip")
species_table = pd.read_pickle(species_table_path, compression="gzip")

test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)
q_summary["quadrat_id"] = q_summary["quadrat_id"].astype(str)
agg_df["quadrat_id"] = agg_df["quadrat_id"].astype(str)
agg_df["species_id"] = agg_df["species_id"].astype(int)

species_table["species_id"] = species_table["species_id"].astype(int)

VALID_QUADRATS = set(test_df["quadrat_id"])
VALID_SPECIES = set(species_table["species_id"])

print("\nAgg shape:", agg_df.shape)
print("Quadrats en agg:", agg_df["quadrat_id"].nunique())
print("Test shape:", test_df.shape)
print("Species válidas:", len(VALID_SPECIES))


agg_df = agg_df.sort_values(
    ["quadrat_id", "final_score"],
    ascending=[True, False]
).reset_index(drop=True)


candidate_configs = {

    "A_balanced_v1": {
        "min_k": 2,
        "max_k": 12,
        "low_green_thr": 0.05,
        "medium_green_thr": 0.20,
        "low_cap": 2,
        "medium_cap": 6,
        "high_cap": 12,
        "score_ratio": 0.18,
        "min_final_score": 0.010,
        "min_views_species": 2,
        "min_freq_score": 0.010,
        "bypass_max_score": 0.45,
        "confident_top2_ratio": 0.55,
        "confident_cap": 4,
        "use_elbow": False,
    },


    "B_precision_v1": {
        "min_k": 2,
        "max_k": 10,
        "low_green_thr": 0.08,
        "medium_green_thr": 0.25,
        "low_cap": 2,
        "medium_cap": 4,
        "high_cap": 8,
        "score_ratio": 0.26,
        "min_final_score": 0.018,
        "min_views_species": 3,
        "min_freq_score": 0.020,
        "bypass_max_score": 0.55,
        "confident_top2_ratio": 0.60,
        "confident_cap": 3,
        "use_elbow": False,
    },

 
    "C_recall_v1": {
        "min_k": 2,
        "max_k": 12,
        "low_green_thr": 0.04,
        "medium_green_thr": 0.16,
        "low_cap": 3,
        "medium_cap": 8,
        "high_cap": 12,
        "score_ratio": 0.12,
        "min_final_score": 0.006,
        "min_views_species": 1,
        "min_freq_score": 0.000,
        "bypass_max_score": 0.35,
        "confident_top2_ratio": 0.45,
        "confident_cap": 6,
        "use_elbow": False,
    },


    "D_vegetation_strict_v1": {
        "min_k": 2,
        "max_k": 12,
        "low_green_thr": 0.12,
        "medium_green_thr": 0.35,
        "low_cap": 2,
        "medium_cap": 5,
        "high_cap": 12,
        "score_ratio": 0.20,
        "min_final_score": 0.012,
        "min_views_species": 2,
        "min_freq_score": 0.012,
        "bypass_max_score": 0.50,
        "confident_top2_ratio": 0.55,
        "confident_cap": 4,
        "use_elbow": False,
    },


    "E_elbow_v1": {
        "min_k": 2,
        "max_k": 12,
        "low_green_thr": 0.05,
        "medium_green_thr": 0.20,
        "low_cap": 2,
        "medium_cap": 7,
        "high_cap": 12,
        "score_ratio": 0.10,
        "min_final_score": 0.006,
        "min_views_species": 1,
        "min_freq_score": 0.000,
        "bypass_max_score": 0.40,
        "confident_top2_ratio": 0.50,
        "confident_cap": 5,
        "use_elbow": True,
        "elbow_drop": 0.10,
        "elbow_min_ratio": 0.22,
    },
}


with open(CAND_DIR / "candidate_configs.json", "w") as f:
    json.dump(candidate_configs, f, indent=2)

print("\nConfiguraciones candidatas:")
for k, v in candidate_configs.items():
    print("-", k, v)


def parse_species_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def determine_cap(qrow, params):
    """
    Determina K máximo según vegetación y confianza.
    """
    green = float(qrow.get("global_green", 0.0))
    pct_high = float(qrow.get("pct_high_green_views", 0.0))
    pct_low = float(qrow.get("pct_low_green_views", 0.0))
    top2_ratio = float(qrow.get("top2_ratio", 1.0))
    top1_score = float(qrow.get("top1_score", 0.0))


    if green < params["low_green_thr"] or pct_high < 0.15 or pct_low > 0.75:
        cap = params["low_cap"]
    elif green < params["medium_green_thr"] or pct_high < 0.50:
        cap = params["medium_cap"]
    else:
        cap = params["high_cap"]


    if top2_ratio < params["confident_top2_ratio"] and top1_score >= 0.35:
        cap = min(cap, params["confident_cap"])

    cap = int(max(params["min_k"], min(params["max_k"], cap)))

    return cap


def apply_elbow_cut(df_sel, params, min_k, cap):
    """
    Corta la lista si aparece una caída fuerte en la curva de scores.
    """
    if not params.get("use_elbow", False):
        return df_sel.head(cap)

    if len(df_sel) <= min_k:
        return df_sel.head(cap)

    scores = df_sel["final_score"].values.astype(float)
    top1 = max(scores[0], 1e-12)

    cut = min(len(df_sel), cap)

    for i in range(min_k, min(len(scores), cap)):
        prev_score = scores[i - 1]
        curr_score = scores[i]

        relative_drop = (prev_score - curr_score) / top1
        curr_ratio = curr_score / top1

        if relative_drop >= params["elbow_drop"] and curr_ratio <= params["elbow_min_ratio"]:
            cut = i
            break

    cut = max(min_k, min(cut, cap))

    return df_sel.head(cut)


def select_species_for_quadrat(df_q, qrow, params):
    """
    Selección adaptive K para un quadrat.
    """
    df_q = df_q.sort_values("final_score", ascending=False).reset_index(drop=True)

    min_k = params["min_k"]
    cap = determine_cap(qrow, params)

    top1_score = float(df_q["final_score"].iloc[0])
    score_threshold = top1_score * params["score_ratio"]


    cond_score = df_q["final_score"] >= score_threshold
    cond_abs = df_q["final_score"] >= params["min_final_score"]


    cond_support = (
        (df_q["n_views_species"] >= params["min_views_species"])
        | (df_q["max_score"] >= params["bypass_max_score"])
    )

    cond_freq = (
        (df_q["freq_score"] >= params["min_freq_score"])
        | (df_q["max_score"] >= params["bypass_max_score"])
    )

    df_sel = df_q[cond_score & cond_abs & cond_support & cond_freq].copy()

 
    if len(df_sel) < min_k:
        df_sel = df_q.head(min_k).copy()

  
    df_sel = apply_elbow_cut(df_sel, params, min_k=min_k, cap=cap)

  
    df_sel = df_sel.head(cap).copy()

    selected = df_sel["species_id"].astype(int).tolist()

  
    selected = list(dict.fromkeys(selected))


    if len(selected) < min_k:
        fallback = df_q["species_id"].astype(int).tolist()
        for sp in fallback:
            if sp not in selected:
                selected.append(sp)
            if len(selected) >= min_k:
                break


    selected = selected[:params["max_k"]]

    return selected, cap, score_threshold


def validate_submission_df(df_sub, name="submission", max_k=12, min_k=1):
    """
    Valida estructura y especies.
    """
    errors = []

    if list(df_sub.columns) != ["quadrat_id", "species_ids"]:
        errors.append(f"Columnas incorrectas: {df_sub.columns.tolist()}")

    if len(df_sub) != len(test_df):
        errors.append(f"Filas incorrectas: {len(df_sub)} vs esperado {len(test_df)}")

    qids = set(df_sub["quadrat_id"].astype(str))
    missing_q = VALID_QUADRATS - qids
    extra_q = qids - VALID_QUADRATS

    if missing_q:
        errors.append(f"Faltan quadrats: {len(missing_q)}")

    if extra_q:
        errors.append(f"Sobran quadrats: {len(extra_q)}")

    if df_sub["quadrat_id"].duplicated().sum() > 0:
        errors.append("Hay quadrat_id duplicados")

    lists = df_sub["species_ids"].apply(parse_species_ids)
    k = lists.apply(len)

    if (k < min_k).any():
        errors.append(f"Filas con K < {min_k}: {(k < min_k).sum()}")

    if (k > max_k).any():
        errors.append(f"Filas con K > {max_k}: {(k > max_k).sum()}")

    repeated = lists.apply(lambda x: len(x) != len(set(x))).sum()
    if repeated > 0:
        errors.append(f"Filas con especies repetidas: {repeated}")

    invalid_count = 0
    for sp_list in lists:
        invalid_count += sum(1 for sp in sp_list if int(sp) not in VALID_SPECIES)

    if invalid_count > 0:
        errors.append(f"Species inválidas: {invalid_count}")

    return {
        "name": name,
        "valid": len(errors) == 0,
        "errors": errors,
        "mean_K": float(k.mean()),
        "min_K": int(k.min()),
        "max_K": int(k.max()),
        "pct_K2": float((k == 2).mean() * 100),
        "pct_K12": float((k == 12).mean() * 100),
        "pct_K_le_2": float((k <= 2).mean() * 100),
        "pct_K_ge_8": float((k >= 8).mean() * 100),
        "n_unique_species": int(len(set([sp for sp_list in lists for sp in sp_list]))),
        "k_distribution": k.value_counts().sort_index().to_dict(),
    }


q_summary_lookup = q_summary.set_index("quadrat_id").to_dict(orient="index")


summary_rows = []
debug_paths = []

for cand_name, params in candidate_configs.items():
    print("\n" + "=" * 100)
    print("GENERANDO:", cand_name)
    print("=" * 100)

    sub_rows = []
    debug_rows = []

    for qid in test_df["quadrat_id"].astype(str).tolist():
        df_q = agg_df[agg_df["quadrat_id"] == qid].copy()

        if len(df_q) == 0:
            raise RuntimeError(f"No hay scores agregados para quadrat_id={qid}")

        qrow = q_summary_lookup.get(qid, {})

        selected, cap, score_threshold = select_species_for_quadrat(df_q, qrow, params)

        species_str = "[" + ", ".join(map(str, selected)) + "]"

        sub_rows.append({
            "quadrat_id": qid,
            "species_ids": species_str,
        })

        debug_rows.append({
            "quadrat_id": qid,
            "K": len(selected),
            "cap": cap,
            "score_threshold": score_threshold,
            "top1_score": float(df_q["final_score"].iloc[0]),
            "top2_ratio": qrow.get("top2_ratio", np.nan),
            "global_green": qrow.get("global_green", np.nan),
            "pct_high_green_views": qrow.get("pct_high_green_views", np.nan),
            "pct_low_green_views": qrow.get("pct_low_green_views", np.nan),
            "selected_species": species_str,
        })

    df_sub = pd.DataFrame(sub_rows)
    df_debug = pd.DataFrame(debug_rows)

   
    val = validate_submission_df(
        df_sub,
        name=cand_name,
        max_k=params["max_k"],
        min_k=params["min_k"]
    )

 
    sub_path_run = CAND_DIR / f"submission_{cand_name}.csv"
    debug_path = CAND_DIR / f"debug_{cand_name}.csv"

    sub_path_public = SUBMISSIONS_DIR / f"submission_{cand_name}.csv"

    df_sub.to_csv(sub_path_run, index=False)
    df_debug.to_csv(debug_path, index=False)

   
    df_sub.to_csv(sub_path_public, index=False)

    summary_row = {
        "candidate": cand_name,
        "valid": val["valid"],
        "errors": "; ".join(val["errors"]),
        "mean_K": val["mean_K"],
        "min_K": val["min_K"],
        "max_K": val["max_K"],
        "pct_K2": val["pct_K2"],
        "pct_K12": val["pct_K12"],
        "pct_K_le_2": val["pct_K_le_2"],
        "pct_K_ge_8": val["pct_K_ge_8"],
        "n_unique_species": val["n_unique_species"],
        "run_submission_path": str(sub_path_run),
        "public_submission_path": str(sub_path_public),
        "debug_path": str(debug_path),
        "k_distribution": val["k_distribution"],
    }

    summary_rows.append(summary_row)
    debug_paths.append(debug_path)

    print("Válida:", val["valid"])
    if val["errors"]:
        print("Errores:", val["errors"])

    print("mean_K:", round(val["mean_K"], 4))
    print("pct_K2:", round(val["pct_K2"], 2))
    print("pct_K12:", round(val["pct_K12"], 2))
    print("pct_K_ge_8:", round(val["pct_K_ge_8"], 2))
    print("n_unique_species:", val["n_unique_species"])

    print("\nDistribución K:")
    display(
        df_debug["K"]
        .value_counts()
        .sort_index()
        .reset_index()
        .rename(columns={"index": "K", "K": "n_images"})
    )


summary_df = pd.DataFrame(summary_rows)

summary_path = CAND_DIR / "candidate_submissions_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 100)
print("RESUMEN DE SUBMISSIONS CANDIDATAS")
print("=" * 100)

display(
    summary_df[
        [
            "candidate",
            "valid",
            "mean_K",
            "min_K",
            "max_K",
            "pct_K2",
            "pct_K12",
            "pct_K_le_2",
            "pct_K_ge_8",
            "n_unique_species",
            "public_submission_path",
        ]
    ]
)

print("\nResumen guardado en:")
print(summary_path)

print("\nArchivos listos en:")
print(SUBMISSIONS_DIR)

print("\n Submissions .")

# **CELL 13 — Inference on validation pseudo-quadrats**

In [ ]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


required_objects = [
    "model",
    "DEVICE",
    "CFG",
    "RUN_DIR",
    "build_views_for_image",
    "pil_to_tensor",
    "resize_square_pil",
    "center_crop_square",
    "IMG_SIZE",
    "TILE_SIZE",
    "idx_to_species",
]

missing = [obj for obj in required_objects if obj not in globals()]

if missing:
    raise RuntimeError(
        "Faltan objetos en memoria. Debes haber ejecutado las celdas previas "
        "donde se cargó el modelo y se definieron las funciones de tiling. "
        f"Faltan: {missing}"
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible. Revisa el kernel GPU.")

model.eval()


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"

PSEUDO_DIR = RUN_DIR / "pseudo_quadrats"
PSEUDO_MANIFEST_PATH = PSEUDO_DIR / "pseudo_val_manifest.pkl.gz"

PSEUDO_INFER_DIR = PSEUDO_DIR / "inference"
PSEUDO_INFER_DIR.mkdir(parents=True, exist_ok=True)

VIEWS_PATH = PSEUDO_INFER_DIR / "pseudo_views_index.csv"
PREDS_PATH = PSEUDO_INFER_DIR / "pseudo_view_topk_predictions.csv"

print("=" * 80)
print("CELDA 13 — INFERENCIA SOBRE PSEUDO-QUADRATS")
print("=" * 80)
print("PSEUDO_MANIFEST_PATH:", PSEUDO_MANIFEST_PATH)
print("PSEUDO_INFER_DIR:", PSEUDO_INFER_DIR)

if not PSEUDO_MANIFEST_PATH.exists():
    raise FileNotFoundError(PSEUDO_MANIFEST_PATH)


TOPK = CFG["inference"]["topk_per_view"]


PSEUDO_BATCH_SIZE = 32
PSEUDO_NUM_WORKERS = 0
PSEUDO_PIN_MEMORY = False

print("\nConfiguración inferencia:")
print("TOPK:", TOPK)
print("PSEUDO_BATCH_SIZE:", PSEUDO_BATCH_SIZE)
print("PSEUDO_NUM_WORKERS:", PSEUDO_NUM_WORKERS)
print("PSEUDO_PIN_MEMORY:", PSEUDO_PIN_MEMORY)


pseudo_manifest = pd.read_pickle(PSEUDO_MANIFEST_PATH, compression="gzip")

pseudo_manifest["quadrat_id"] = pseudo_manifest["quadrat_id"].astype(str)
pseudo_manifest["file_path"] = pseudo_manifest["file_path"].astype(str)
pseudo_manifest["true_K"] = pseudo_manifest["true_K"].astype(int)

pseudo_manifest["file_exists"] = pseudo_manifest["file_path"].apply(lambda p: Path(p).exists())

print("\n" + "=" * 80)
print("PSEUDO MANIFEST")
print("=" * 80)
print("Shape:", pseudo_manifest.shape)
print("Archivos existentes:", pseudo_manifest["file_exists"].sum(), "/", len(pseudo_manifest))
print("Quadrats únicos:", pseudo_manifest["quadrat_id"].nunique())

if (~pseudo_manifest["file_exists"]).sum() > 0:
    display(pseudo_manifest.loc[~pseudo_manifest["file_exists"], ["quadrat_id", "file_path"]].head())
    raise RuntimeError("Hay pseudo-imágenes faltantes.")

display(
    pseudo_manifest["true_K"]
    .value_counts()
    .sort_index()
    .reset_index()
    .rename(columns={"index": "true_K", "true_K": "n_images"})
)


if VIEWS_PATH.exists():
    print("\nCargando vistas existentes:")
    print(VIEWS_PATH)
    pseudo_views = pd.read_csv(VIEWS_PATH)
else:
    print("\n" + "=" * 80)
    print("CONSTRUYENDO VISTAS PSEUDO")
    print("=" * 80)

    view_rows = []

    start_views = time.time()

    for row in tqdm(
        pseudo_manifest.itertuples(index=False),
        total=len(pseudo_manifest),
        desc="build_pseudo_views"
    ):
        qid = row.quadrat_id
        path = row.file_path

        try:
            views = build_views_for_image(path)

            for j, v in enumerate(views):
                view_rows.append({
                    "quadrat_id": qid,
                    "file_path": path,
                    "view_id": j,
                    "view_type": v["view_type"],
                    "x": v["x"],
                    "y": v["y"],
                    "green_ratio": float(v["green_ratio"]),
                    "true_K": int(row.true_K),
                    "true_species_ids": row.true_species_ids,
                    "construction_mode": row.construction_mode,
                })

        except Exception as e:
            print(f" Error construyendo vistas | qid={qid} | {e}")

    pseudo_views = pd.DataFrame(view_rows)
    pseudo_views.to_csv(VIEWS_PATH, index=False)

    elapsed_views = time.time() - start_views

    print("Vistas guardadas:", VIEWS_PATH)
    print("Tiempo construcción vistas:", round(elapsed_views / 60, 2), "min")

print("\n" + "=" * 80)
print("RESUMEN VISTAS PSEUDO")
print("=" * 80)
print("Pseudo views shape:", pseudo_views.shape)
print("Quadrats en vistas:", pseudo_views["quadrat_id"].nunique())

display(
    pseudo_views.groupby("quadrat_id")
    .size()
    .reset_index(name="n_views")
    .describe()
)

display(
    pseudo_views["view_type"]
    .value_counts()
    .reset_index(name="count")
)


class PseudoQuadratViewsDataset(Dataset):
    def __init__(self, views_df):
        self.df = views_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        img = Image.open(r["file_path"]).convert("RGB")
        view_type = r["view_type"]

        if view_type == "full":
            view = resize_square_pil(img, IMG_SIZE)

        elif view_type == "center":
            view = resize_square_pil(center_crop_square(img), IMG_SIZE)

        elif view_type == "small_resized":
            view = resize_square_pil(img, IMG_SIZE)

        elif view_type == "tile":
            x0 = int(r["x"])
            y0 = int(r["y"])
            view = img.crop((x0, y0, x0 + TILE_SIZE, y0 + TILE_SIZE))
            view = resize_square_pil(view, IMG_SIZE)

        else:
            view = resize_square_pil(img, IMG_SIZE)

        x_tensor = pil_to_tensor(view)

        return {
            "image": x_tensor,
            "quadrat_id": r["quadrat_id"],
            "view_id": int(r["view_id"]),
            "view_type": view_type,
            "green_ratio": float(r["green_ratio"]),
            "true_K": int(r["true_K"]),
            "true_species_ids": r["true_species_ids"],
            "construction_mode": r["construction_mode"],
        }


def collate_fn_pseudo(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)

    return {
        "image": images,
        "quadrat_id": [b["quadrat_id"] for b in batch],
        "view_id": [b["view_id"] for b in batch],
        "view_type": [b["view_type"] for b in batch],
        "green_ratio": [b["green_ratio"] for b in batch],
        "true_K": [b["true_K"] for b in batch],
        "true_species_ids": [b["true_species_ids"] for b in batch],
        "construction_mode": [b["construction_mode"] for b in batch],
    }


pseudo_dataset = PseudoQuadratViewsDataset(pseudo_views)

pseudo_loader = DataLoader(
    pseudo_dataset,
    batch_size=PSEUDO_BATCH_SIZE,
    shuffle=False,
    num_workers=PSEUDO_NUM_WORKERS,
    pin_memory=PSEUDO_PIN_MEMORY,
    collate_fn=collate_fn_pseudo,
)

print("\n" + "=" * 80)
print("DATALOADER PSEUDO")
print("=" * 80)
print("Dataset views:", len(pseudo_dataset))
print("Batches:", len(pseudo_loader))


if PREDS_PATH.exists():
    print("\nYa existe archivo de predicciones pseudo:")
    print(PREDS_PATH)
    print("Si quieres regenerar, borra ese archivo y vuelve a correr la celda.")

    pseudo_preds = pd.read_csv(PREDS_PATH)

else:
    print("\n" + "=" * 80)
    print("INFERENCIA PSEUDO")
    print("=" * 80)

    pred_rows = []

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    start_infer = time.time()

    model.eval()

    with torch.no_grad():
        for batch in tqdm(pseudo_loader, desc="infer_pseudo_quadrats"):
            images = batch["image"].to(DEVICE, non_blocking=False)

            with torch.autocast(device_type="cuda", enabled=CFG["inference"]["amp"]):
                logits = model(images)
                probs = F.softmax(logits, dim=1)
                vals, inds = torch.topk(probs, k=TOPK, dim=1)

            vals_np = vals.detach().cpu().numpy()
            inds_np = inds.detach().cpu().numpy()

            for i in range(len(batch["quadrat_id"])):
                qid = batch["quadrat_id"][i]
                view_id = batch["view_id"][i]
                view_type = batch["view_type"][i]
                green_ratio = batch["green_ratio"][i]
                true_K = batch["true_K"][i]
                true_species_ids = batch["true_species_ids"][i]
                construction_mode = batch["construction_mode"][i]

                for rank in range(TOPK):
                    class_index = int(inds_np[i, rank])
                    species_id = int(idx_to_species[class_index])

                    pred_rows.append({
                        "quadrat_id": qid,
                        "view_id": view_id,
                        "view_type": view_type,
                        "green_ratio": green_ratio,
                        "rank": rank + 1,
                        "class_index": class_index,
                        "species_id": species_id,
                        "score": float(vals_np[i, rank]),
                        "true_K": int(true_K),
                        "true_species_ids": true_species_ids,
                        "construction_mode": construction_mode,
                    })

            del images, logits, probs, vals, inds, vals_np, inds_np

    infer_time = time.time() - start_infer

    pseudo_preds = pd.DataFrame(pred_rows)
    pseudo_preds.to_csv(PREDS_PATH, index=False)

    print("\nInferencia guardada:", PREDS_PATH)
    print("Tiempo inferencia:", round(infer_time / 60, 2), "min")

    print("\nMemoria GPU:")
    print("Asignada actual GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("Reservada actual GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
    print("Pico asignado GB:", round(torch.cuda.max_memory_allocated() / 1024**3, 3))
    print("Pico reservado GB:", round(torch.cuda.max_memory_reserved() / 1024**3, 3))


print("\n" + "=" * 80)
print("RESUMEN PREDICCIONES PSEUDO")
print("=" * 80)
print("pseudo_preds shape:", pseudo_preds.shape)
print("Quadrats:", pseudo_preds["quadrat_id"].nunique())
print("Species candidatas únicas:", pseudo_preds["species_id"].nunique())

print("\nScore summary:")
display(pseudo_preds["score"].describe())

print("\nPrimeras predicciones:")
display(pseudo_preds.head(20))

print("\nPredicciones por true_K:")
display(
    pseudo_preds.groupby("true_K")
    .agg(
        n_rows=("score", "size"),
        n_quadrats=("quadrat_id", "nunique"),
        n_species_pred=("species_id", "nunique"),
        max_score=("score", "max"),
        mean_score=("score", "mean"),
    )
    .reset_index()
)

gc.collect()
torch.cuda.empty_cache()

print("\n inferencia pseudo-quadrats lista.")

# **CELL 14 — Pseudo-quadrat aggregation and internal diagnostics**

In [ ]:
import ast
import re
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"

PSEUDO_DIR = RUN_DIR / "pseudo_quadrats"
PSEUDO_INFER_DIR = PSEUDO_DIR / "inference"
PSEUDO_AGG_DIR = PSEUDO_DIR / "aggregation"
PSEUDO_AGG_DIR.mkdir(parents=True, exist_ok=True)

PSEUDO_PREDS_PATH = PSEUDO_INFER_DIR / "pseudo_view_topk_predictions.csv"
PSEUDO_MANIFEST_PATH = PSEUDO_DIR / "pseudo_val_manifest.pkl.gz"

PSEUDO_AGG_PATH = PSEUDO_AGG_DIR / "pseudo_quadrat_species_scores.pkl.gz"
PSEUDO_AGG_TOP100_PATH = PSEUDO_AGG_DIR / "pseudo_quadrat_species_scores_top100.csv"
PSEUDO_QSUMMARY_PATH = PSEUDO_AGG_DIR / "pseudo_quadrat_quality_summary.csv"
PSEUDO_RANK_DIAG_PATH = PSEUDO_AGG_DIR / "pseudo_true_species_rank_diagnostics.csv"

print("=" * 80)
print("CELDA 14 — AGREGACIÓN PSEUDO-QUADRATS")
print("=" * 80)

for p in [PSEUDO_PREDS_PATH, PSEUDO_MANIFEST_PATH]:
    print(p, "| existe:", p.exists())

if not PSEUDO_PREDS_PATH.exists():
    raise FileNotFoundError(PSEUDO_PREDS_PATH)

if not PSEUDO_MANIFEST_PATH.exists():
    raise FileNotFoundError(PSEUDO_MANIFEST_PATH)


AGG_TOPK = 5

W_MEAN_TOP = 0.50
W_MAX = 0.25
W_VEG = 0.15
W_FREQ_BONUS = 0.10

print("\nParámetros de agregación:")
print("AGG_TOPK:", AGG_TOPK)
print("W_MEAN_TOP:", W_MEAN_TOP)
print("W_MAX:", W_MAX)
print("W_VEG:", W_VEG)
print("W_FREQ_BONUS:", W_FREQ_BONUS)


def parse_species_list(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def aggregate_pseudo_predictions(df_pred: pd.DataFrame):
    """
    Agrega predicciones top-k por vista/tile a nivel pseudo_quadrat-species.
    """
    df = df_pred.copy()

    df["quadrat_id"] = df["quadrat_id"].astype(str)
    df["species_id"] = df["species_id"].astype(int)
    df["view_id"] = df["view_id"].astype(int)
    df["score"] = df["score"].astype(float)
    df["green_ratio"] = df["green_ratio"].astype(float)
    df["rank"] = df["rank"].astype(int)
    df["true_K"] = df["true_K"].astype(int)


    unique_views = (
        df[["quadrat_id", "view_id", "view_type", "green_ratio", "true_K", "true_species_ids", "construction_mode"]]
        .drop_duplicates()
        .copy()
    )

    q_stats = (
        unique_views
        .groupby("quadrat_id")
        .agg(
            true_K=("true_K", "first"),
            true_species_ids=("true_species_ids", "first"),
            construction_mode=("construction_mode", "first"),
            n_views_total=("view_id", "nunique"),
            global_green=("green_ratio", "mean"),
            max_green=("green_ratio", "max"),
            min_green=("green_ratio", "min"),
            std_green=("green_ratio", "std"),
            pct_low_green_views=("green_ratio", lambda x: float((x < 0.05).mean())),
            pct_high_green_views=("green_ratio", lambda x: float((x >= 0.15).mean())),
        )
        .reset_index()
    )

    q_stats["std_green"] = q_stats["std_green"].fillna(0.0)


    base = (
        df
        .groupby(["quadrat_id", "species_id"])
        .agg(
            n_hits=("score", "size"),
            n_views_species=("view_id", "nunique"),
            mean_score_all=("score", "mean"),
            max_score=("score", "max"),
            min_rank=("rank", "min"),
            mean_rank=("rank", "mean"),
            mean_green_species=("green_ratio", "mean"),
        )
        .reset_index()
    )


    df_sorted = df.sort_values(
        ["quadrat_id", "species_id", "score"],
        ascending=[True, True, False]
    )

    top_scores = (
        df_sorted
        .groupby(["quadrat_id", "species_id"])
        .head(AGG_TOPK)
        .groupby(["quadrat_id", "species_id"])
        .agg(
            mean_top_score=("score", "mean"),
            sum_top_score=("score", "sum"),
        )
        .reset_index()
    )


    df["veg_weight"] = 0.5 + df["green_ratio"].clip(0, 1)
    df["score_x_veg_weight"] = df["score"] * df["veg_weight"]

    veg = (
        df
        .groupby(["quadrat_id", "species_id"])
        .agg(
            weighted_score_sum=("score_x_veg_weight", "sum"),
            veg_weight_sum=("veg_weight", "sum"),
        )
        .reset_index()
    )

    veg["veg_weighted_score"] = (
        veg["weighted_score_sum"] / veg["veg_weight_sum"].replace(0, np.nan)
    ).fillna(0.0)

    veg = veg[["quadrat_id", "species_id", "veg_weighted_score"]]


    view_support = (
        df
        .groupby(["quadrat_id", "species_id", "view_type"])
        .agg(
            n_views_type=("view_id", "nunique"),
            max_score_type=("score", "max"),
        )
        .reset_index()
    )

    view_pivot = view_support.pivot_table(
        index=["quadrat_id", "species_id"],
        columns="view_type",
        values="n_views_type",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    for c in ["full", "center", "tile", "small_resized"]:
        if c not in view_pivot.columns:
            view_pivot[c] = 0

    view_pivot = view_pivot.rename(columns={
        "full": "n_full_hits",
        "center": "n_center_hits",
        "tile": "n_tile_hits",
        "small_resized": "n_small_resized_hits",
    })


    agg = base.merge(
        top_scores,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        veg,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        view_pivot,
        on=["quadrat_id", "species_id"],
        how="left"
    )

    agg = agg.merge(
        q_stats,
        on="quadrat_id",
        how="left"
    )

    fill_cols = [
        "mean_top_score",
        "sum_top_score",
        "veg_weighted_score",
        "n_full_hits",
        "n_center_hits",
        "n_tile_hits",
        "n_small_resized_hits",
    ]

    for c in fill_cols:
        agg[c] = agg[c].fillna(0)


    agg["freq_score"] = (
        agg["n_views_species"] / agg["n_views_total"].replace(0, np.nan)
    ).fillna(0.0)

    agg["freq_bonus"] = agg["freq_score"] * agg["max_score"]

    agg["context_bonus"] = (
        (agg["n_full_hits"] > 0).astype(float) * 0.015
        + (agg["n_center_hits"] > 0).astype(float) * 0.010
    )

    agg["single_view_penalty"] = np.where(
        (agg["n_views_species"] <= 1) & (agg["max_score"] < 0.30),
        0.85,
        1.00
    )

    agg["final_score_raw"] = (
        W_MEAN_TOP * agg["mean_top_score"]
        + W_MAX * agg["max_score"]
        + W_VEG * agg["veg_weighted_score"]
        + W_FREQ_BONUS * agg["freq_bonus"]
        + agg["context_bonus"]
    )

    agg["final_score"] = agg["final_score_raw"] * agg["single_view_penalty"]


    agg = agg.sort_values(
        ["quadrat_id", "final_score"],
        ascending=[True, False]
    ).reset_index(drop=True)

    agg["pred_rank"] = (
        agg.groupby("quadrat_id")
        .cumcount()
        + 1
    )

    return agg, q_stats


print("\n" + "=" * 80)
print("CARGANDO PREDICCIONES PSEUDO")
print("=" * 80)

start = time.time()

pseudo_preds = pd.read_csv(PSEUDO_PREDS_PATH)

print("pseudo_preds shape:", pseudo_preds.shape)
print("Quadrats:", pseudo_preds["quadrat_id"].nunique())


print("\n" + "=" * 80)
print("AGREGANDO PREDICCIONES")
print("=" * 80)

pseudo_agg, pseudo_qstats = aggregate_pseudo_predictions(pseudo_preds)

elapsed = time.time() - start

print("pseudo_agg shape:", pseudo_agg.shape)
print("pseudo_qstats shape:", pseudo_qstats.shape)
print("Tiempo agregación:", round(elapsed / 60, 2), "min")


pseudo_qsummary = (
    pseudo_agg
    .groupby("quadrat_id")
    .agg(
        true_K=("true_K", "first"),
        true_species_ids=("true_species_ids", "first"),
        construction_mode=("construction_mode", "first"),
        n_candidate_species=("species_id", "nunique"),
        global_green=("global_green", "first"),
        max_green=("max_green", "first"),
        pct_low_green_views=("pct_low_green_views", "first"),
        pct_high_green_views=("pct_high_green_views", "first"),
        n_views_total=("n_views_total", "first"),
        top1_score=("final_score", "max"),
    )
    .reset_index()
)

top2_scores = (
    pseudo_agg
    .groupby("quadrat_id")
    .head(2)
    .groupby("quadrat_id")["final_score"]
    .apply(list)
    .reset_index(name="top_scores")
)

def get_top2_margin(scores):
    if len(scores) < 2:
        return np.nan
    return scores[0] - scores[1]

def get_top2_ratio(scores):
    if len(scores) < 2 or scores[0] == 0:
        return np.nan
    return scores[1] / scores[0]

top2_scores["top2_margin"] = top2_scores["top_scores"].apply(get_top2_margin)
top2_scores["top2_ratio"] = top2_scores["top_scores"].apply(get_top2_ratio)

pseudo_qsummary = pseudo_qsummary.merge(
    top2_scores[["quadrat_id", "top2_margin", "top2_ratio"]],
    on="quadrat_id",
    how="left"
)


print("\n" + "=" * 80)
print("DIAGNÓSTICO DE RANKS DE ESPECIES VERDADERAS")
print("=" * 80)

diag_rows = []

for row in tqdm(pseudo_qsummary.itertuples(index=False), total=len(pseudo_qsummary), desc="true_rank_diag"):
    qid = row.quadrat_id
    true_species = parse_species_list(row.true_species_ids)
    true_set = set(true_species)

    df_q = pseudo_agg[pseudo_agg["quadrat_id"] == qid].copy()
    rank_lookup = dict(zip(df_q["species_id"].astype(int), df_q["pred_rank"].astype(int)))
    score_lookup = dict(zip(df_q["species_id"].astype(int), df_q["final_score"].astype(float)))

    for sp in true_species:
        rnk = rank_lookup.get(int(sp), np.nan)
        scr = score_lookup.get(int(sp), np.nan)

        diag_rows.append({
            "quadrat_id": qid,
            "true_K": int(row.true_K),
            "construction_mode": row.construction_mode,
            "true_species_id": int(sp),
            "found_in_candidates": not pd.isna(rnk),
            "pred_rank": rnk,
            "final_score": scr,
            "global_green": float(row.global_green),
            "top1_score": float(row.top1_score),
            "top2_ratio": float(row.top2_ratio),
        })

rank_diag = pd.DataFrame(diag_rows)


K_EVALS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 20, 50, 100]

for k in K_EVALS:
    rank_diag[f"hit_at_{k}"] = (
        rank_diag["found_in_candidates"]
        & (rank_diag["pred_rank"] <= k)
    ).astype(int)


q_eval_rows = []

for qid, g in rank_diag.groupby("quadrat_id"):
    true_K = int(g["true_K"].iloc[0])
    mode = g["construction_mode"].iloc[0]

    row = {
        "quadrat_id": qid,
        "true_K": true_K,
        "construction_mode": mode,
        "n_true": len(g),
        "n_found_candidates": int(g["found_in_candidates"].sum()),
        "candidate_recall": float(g["found_in_candidates"].mean()),
        "best_true_rank": float(g["pred_rank"].min()) if g["found_in_candidates"].any() else np.nan,
        "median_true_rank": float(g["pred_rank"].median()) if g["found_in_candidates"].any() else np.nan,
    }

    for k in K_EVALS:
        row[f"recall_at_{k}"] = float(g[f"hit_at_{k}"].mean())

    q_eval_rows.append(row)

q_eval = pd.DataFrame(q_eval_rows)


pseudo_agg.to_pickle(PSEUDO_AGG_PATH, compression="gzip")

pseudo_top100 = (
    pseudo_agg
    .groupby("quadrat_id", group_keys=False)
    .head(100)
    .copy()
)
pseudo_top100.to_csv(PSEUDO_AGG_TOP100_PATH, index=False)

pseudo_qsummary.to_csv(PSEUDO_QSUMMARY_PATH, index=False)
rank_diag.to_csv(PSEUDO_RANK_DIAG_PATH, index=False)

q_eval_path = PSEUDO_AGG_DIR / "pseudo_quadrat_recall_at_k.csv"
q_eval.to_csv(q_eval_path, index=False)

print("\n" + "=" * 80)
print("ARCHIVOS GUARDADOS")
print("=" * 80)
print("Pseudo agg:", PSEUDO_AGG_PATH)
print("Pseudo top100:", PSEUDO_AGG_TOP100_PATH)
print("Pseudo qsummary:", PSEUDO_QSUMMARY_PATH)
print("Rank diagnostics:", PSEUDO_RANK_DIAG_PATH)
print("Recall@K:", q_eval_path)


print("\n" + "=" * 80)
print("RESUMEN AGREGACIÓN")
print("=" * 80)

print("pseudo_agg shape:", pseudo_agg.shape)
print("Quadrats:", pseudo_agg["quadrat_id"].nunique())
print("Candidatos promedio por quadrat:", round(pseudo_qsummary["n_candidate_species"].mean(), 2))

print("\nTop pseudo_agg:")
display(pseudo_agg.head(20))

print("\nQuality summary:")
display(pseudo_qsummary.describe())

print("\n" + "=" * 80)
print("RECALL INTERNO @ K FIJO")
print("=" * 80)

overall_recall = {}
for k in K_EVALS:
    overall_recall[f"recall_at_{k}"] = q_eval[f"recall_at_{k}"].mean()

overall_recall_df = pd.DataFrame([overall_recall]).T.reset_index()
overall_recall_df.columns = ["metric", "mean_recall"]
display(overall_recall_df)

print("\nRecall@K por true_K:")
recall_cols = [f"recall_at_{k}" for k in K_EVALS]
display(
    q_eval
    .groupby("true_K")[recall_cols]
    .mean()
    .reset_index()
)

print("\nCandidate recall por true_K:")
display(
    q_eval
    .groupby("true_K")
    .agg(
        n_quadrats=("quadrat_id", "nunique"),
        candidate_recall=("candidate_recall", "mean"),
        best_true_rank_median=("best_true_rank", "median"),
        median_true_rank_median=("median_true_rank", "median"),
    )
    .reset_index()
)

print("\nRecall@K por modo de construcción:")
display(
    q_eval
    .groupby("construction_mode")[recall_cols]
    .mean()
    .reset_index()
)

# Limpieza
del pseudo_preds
gc.collect()

print("\n Agregación y diagnóstico interno listos.")

# **CELL 15 — Build visual prototypes by species and organ**

In [ ]:
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
PROTOTYPE_DIR = STORAGE_ROOT / "runs" / "prototype_rerank_B"

REF_EMB_DIR = PROTOTYPE_DIR / "reference_embeddings"
REF_EMB_PATH = REF_EMB_DIR / "reference_embeddings_all.npy"
REF_META_PATH = REF_EMB_DIR / "reference_embeddings_metadata.csv"

REFERENCE_BANK_PATH = PROTOTYPE_DIR / "reference_bank_B_candidates.pkl.gz"
SPECIES_CANDIDATES_PATH = PROTOTYPE_DIR / "champion_B_species_candidates.csv"

PROTO_DIR = PROTOTYPE_DIR / "prototypes"
PROTO_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CELDA 26 — CONSTRUIR PROTOTIPOS VISUALES")
print("=" * 80)

paths_to_check = [
    REF_EMB_PATH,
    REF_META_PATH,
    REFERENCE_BANK_PATH,
    SPECIES_CANDIDATES_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


print("\n" + "=" * 80)
print("CARGANDO EMBEDDINGS DE REFERENCIA")
print("=" * 80)

ref_emb = np.load(REF_EMB_PATH).astype(np.float32)
ref_meta = pd.read_csv(REF_META_PATH)
reference_bank = pd.read_pickle(REFERENCE_BANK_PATH, compression="gzip")
species_candidates = pd.read_csv(SPECIES_CANDIDATES_PATH)

print("ref_emb shape:", ref_emb.shape)
print("ref_meta shape:", ref_meta.shape)
print("reference_bank shape:", reference_bank.shape)
print("species_candidates shape:", species_candidates.shape)

if ref_emb.shape[0] != len(ref_meta):
    raise RuntimeError(
        f"Desalineación: embeddings {ref_emb.shape[0]} vs metadata {len(ref_meta)}"
    )


ref_meta["ref_id"] = ref_meta["ref_id"].astype(str)
ref_meta["species_id"] = ref_meta["species_id"].astype(int)
ref_meta["organ"] = ref_meta["organ"].fillna("unknown").astype(str).str.lower()
ref_meta["split_source"] = ref_meta["split_source"].astype(str)

reference_bank["ref_id"] = reference_bank["ref_id"].astype(str)
reference_bank["species_id"] = reference_bank["species_id"].astype(int)

species_candidates["species_id"] = species_candidates["species_id"].astype(int)


norms = np.linalg.norm(ref_emb, axis=1, keepdims=True)
ref_emb = ref_emb / np.maximum(norms, 1e-12)

print("\nNormas ref_emb después de normalizar:")
print("min:", float(np.linalg.norm(ref_emb, axis=1).min()))
print("mean:", float(np.linalg.norm(ref_emb, axis=1).mean()))
print("max:", float(np.linalg.norm(ref_emb, axis=1).max()))


bank_cols = [
    "ref_id",
    "species",
    "genus",
    "family",
    "n_quadrats",
    "n_occurrences",
    "mean_rank_in_B",
    "median_rank_in_B",
    "mean_K_context",
]

bank_cols = [c for c in bank_cols if c in reference_bank.columns]

ref_meta_full = ref_meta.merge(
    reference_bank[bank_cols].drop_duplicates("ref_id"),
    on="ref_id",
    how="left"
)

print("\nref_meta_full shape:", ref_meta_full.shape)
print("Species en metadata:", ref_meta_full["species_id"].nunique())
print("Órganos:")
display(ref_meta_full["organ"].value_counts().reset_index(name="n_refs"))


def normalized_mean(vectors: np.ndarray):
    """
    Calcula promedio y normaliza L2.
    """
    proto = vectors.mean(axis=0).astype(np.float32)
    norm = np.linalg.norm(proto)
    if norm <= 1e-12:
        return proto
    return proto / norm


def cosine_stats_to_proto(vectors: np.ndarray, proto: np.ndarray):
    """
    Estadísticas de similitud coseno de referencias contra su prototipo.
    Como embeddings y prototipo están normalizados, dot = coseno.
    """
    sims = vectors @ proto.astype(np.float32)

    return {
        "sim_mean": float(np.mean(sims)),
        "sim_std": float(np.std(sims)),
        "sim_min": float(np.min(sims)),
        "sim_p10": float(np.percentile(sims, 10)),
        "sim_p25": float(np.percentile(sims, 25)),
        "sim_median": float(np.percentile(sims, 50)),
        "sim_p75": float(np.percentile(sims, 75)),
        "sim_p90": float(np.percentile(sims, 90)),
        "sim_max": float(np.max(sims)),
    }


print("\n" + "=" * 80)
print("PARTE A — PROTOTIPOS POR ESPECIE")
print("=" * 80)

species_proto_rows = []
species_proto_vectors = []
species_quality_rows = []

species_ids = sorted(ref_meta_full["species_id"].unique().tolist())

for sp in tqdm(species_ids, desc="build_species_prototypes"):
    idx = ref_meta_full.index[ref_meta_full["species_id"] == sp].to_numpy()
    vectors = ref_emb[idx]

    proto = normalized_mean(vectors)
    stats = cosine_stats_to_proto(vectors, proto)

    g = ref_meta_full.loc[idx].copy()

    n_refs = len(g)
    n_train = int((g["split_source"] == "train").sum())
    n_val = int((g["split_source"] == "val").sum())

    organ_counts = g["organ"].value_counts().to_dict()
    dominant_organ = g["organ"].value_counts().index[0] if n_refs > 0 else "unknown"
    n_organs = int(g["organ"].nunique())

 
    species_name = g["species"].dropna().iloc[0] if "species" in g.columns and g["species"].notna().any() else None
    genus = g["genus"].dropna().iloc[0] if "genus" in g.columns and g["genus"].notna().any() else None
    family = g["family"].dropna().iloc[0] if "family" in g.columns and g["family"].notna().any() else None

    n_quadrats = float(g["n_quadrats"].dropna().iloc[0]) if "n_quadrats" in g.columns and g["n_quadrats"].notna().any() else np.nan
    mean_rank_in_B = float(g["mean_rank_in_B"].dropna().iloc[0]) if "mean_rank_in_B" in g.columns and g["mean_rank_in_B"].notna().any() else np.nan
    median_rank_in_B = float(g["median_rank_in_B"].dropna().iloc[0]) if "median_rank_in_B" in g.columns and g["median_rank_in_B"].notna().any() else np.nan

    proto_index = len(species_proto_vectors)
    species_proto_vectors.append(proto)

    species_proto_rows.append({
        "prototype_index": proto_index,
        "species_id": int(sp),
        "species": species_name,
        "genus": genus,
        "family": family,
        "n_refs": int(n_refs),
        "n_train_refs": n_train,
        "n_val_refs": n_val,
        "n_organs": n_organs,
        "dominant_organ": dominant_organ,
        "organ_counts_json": json.dumps(organ_counts),
        "n_quadrats_in_B": n_quadrats,
        "mean_rank_in_B": mean_rank_in_B,
        "median_rank_in_B": median_rank_in_B,
        **stats,
    })

    species_quality_rows.append({
        "species_id": int(sp),
        "n_refs": int(n_refs),
        "n_organs": n_organs,
        "dominant_organ": dominant_organ,
        **stats,
    })

species_prototypes = np.stack(species_proto_vectors, axis=0).astype(np.float32)
species_proto_meta = pd.DataFrame(species_proto_rows)
species_quality = pd.DataFrame(species_quality_rows)


species_prototypes = species_prototypes / np.maximum(
    np.linalg.norm(species_prototypes, axis=1, keepdims=True),
    1e-12
)

print("species_prototypes shape:", species_prototypes.shape)
print("species_proto_meta shape:", species_proto_meta.shape)

print("\nResumen calidad especie:")
display(
    species_proto_meta[
        [
            "n_refs",
            "n_organs",
            "sim_mean",
            "sim_std",
            "sim_p10",
            "sim_median",
            "sim_p90",
        ]
    ].describe()
)

print("\nTop especies con prototipos más compactos:")
display(
    species_proto_meta
    .sort_values("sim_mean", ascending=False)
    [
        [
            "species_id",
            "species",
            "n_refs",
            "n_organs",
            "dominant_organ",
            "sim_mean",
            "sim_p10",
            "sim_median",
        ]
    ]
    .head(20)
)

print("\nEspecies con prototipos menos compactos:")
display(
    species_proto_meta
    .sort_values("sim_mean", ascending=True)
    [
        [
            "species_id",
            "species",
            "n_refs",
            "n_organs",
            "dominant_organ",
            "sim_mean",
            "sim_p10",
            "sim_median",
        ]
    ]
    .head(20)
)


print("\n" + "=" * 80)
print("PARTE B — PROTOTIPOS POR ESPECIE + ÓRGANO")
print("=" * 80)

species_organ_proto_rows = []
species_organ_proto_vectors = []

group_cols = ["species_id", "organ"]

for (sp, organ), g in tqdm(
    ref_meta_full.groupby(group_cols),
    desc="build_species_organ_prototypes"
):
    idx = g.index.to_numpy()
    vectors = ref_emb[idx]

    proto = normalized_mean(vectors)
    stats = cosine_stats_to_proto(vectors, proto)

    n_refs = len(g)
    n_train = int((g["split_source"] == "train").sum())
    n_val = int((g["split_source"] == "val").sum())

    species_name = g["species"].dropna().iloc[0] if "species" in g.columns and g["species"].notna().any() else None
    genus = g["genus"].dropna().iloc[0] if "genus" in g.columns and g["genus"].notna().any() else None
    family = g["family"].dropna().iloc[0] if "family" in g.columns and g["family"].notna().any() else None

    proto_index = len(species_organ_proto_vectors)
    species_organ_proto_vectors.append(proto)

    species_organ_proto_rows.append({
        "prototype_index": proto_index,
        "species_id": int(sp),
        "organ": str(organ),
        "species": species_name,
        "genus": genus,
        "family": family,
        "n_refs": int(n_refs),
        "n_train_refs": n_train,
        "n_val_refs": n_val,
        **stats,
    })

species_organ_prototypes = np.stack(species_organ_proto_vectors, axis=0).astype(np.float32)
species_organ_proto_meta = pd.DataFrame(species_organ_proto_rows)

species_organ_prototypes = species_organ_prototypes / np.maximum(
    np.linalg.norm(species_organ_prototypes, axis=1, keepdims=True),
    1e-12
)

print("species_organ_prototypes shape:", species_organ_prototypes.shape)
print("species_organ_proto_meta shape:", species_organ_proto_meta.shape)
print("Species cubiertas por organ-prototypes:", species_organ_proto_meta["species_id"].nunique())
print("Órganos en organ-prototypes:")
display(species_organ_proto_meta["organ"].value_counts().reset_index(name="n_prototypes"))

print("\nResumen calidad especie+órgano:")
display(
    species_organ_proto_meta[
        [
            "n_refs",
            "sim_mean",
            "sim_std",
            "sim_p10",
            "sim_median",
            "sim_p90",
        ]
    ].describe()
)


print("\n" + "=" * 80)
print("GUARDANDO PROTOTIPOS")
print("=" * 80)

species_proto_path = PROTO_DIR / "species_prototypes.npy"
species_proto_meta_path = PROTO_DIR / "species_prototypes_metadata.csv"

species_organ_proto_path = PROTO_DIR / "species_organ_prototypes.npy"
species_organ_proto_meta_path = PROTO_DIR / "species_organ_prototypes_metadata.csv"

species_quality_path = PROTO_DIR / "species_prototype_quality.csv"
config_path = PROTO_DIR / "prototype_config.json"

np.save(species_proto_path, species_prototypes)
species_proto_meta.to_csv(species_proto_meta_path, index=False)

np.save(species_organ_proto_path, species_organ_prototypes)
species_organ_proto_meta.to_csv(species_organ_proto_meta_path, index=False)

species_quality.to_csv(species_quality_path, index=False)

config = {
    "source_reference_embeddings": str(REF_EMB_PATH),
    "source_reference_metadata": str(REF_META_PATH),
    "reference_bank_path": str(REFERENCE_BANK_PATH),
    "n_reference_embeddings": int(ref_emb.shape[0]),
    "embedding_dim": int(ref_emb.shape[1]),
    "n_species_prototypes": int(species_prototypes.shape[0]),
    "n_species_organ_prototypes": int(species_organ_prototypes.shape[0]),
    "n_species": int(species_proto_meta["species_id"].nunique()),
    "species_prototypes_path": str(species_proto_path),
    "species_prototypes_metadata_path": str(species_proto_meta_path),
    "species_organ_prototypes_path": str(species_organ_proto_path),
    "species_organ_prototypes_metadata_path": str(species_organ_proto_meta_path),
    "species_quality_path": str(species_quality_path),
    "notes": (
        "All prototypes are L2-normalized means of L2-normalized DINOv2 embeddings. "
        "species_prototypes are global per species; species_organ_prototypes are per species-organ."
    ),
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Species prototypes:", species_proto_path)
print("Species metadata:", species_proto_meta_path)
print("Species-organ prototypes:", species_organ_proto_path)
print("Species-organ metadata:", species_organ_proto_meta_path)
print("Quality:", species_quality_path)
print("Config:", config_path)


print("\n" + "=" * 80)
print("VALIDACIÓN FINAL")
print("=" * 80)

print("Species prototypes norms:")
sp_norms = np.linalg.norm(species_prototypes, axis=1)
print("min:", float(sp_norms.min()))
print("mean:", float(sp_norms.mean()))
print("max:", float(sp_norms.max()))

print("\nSpecies-organ prototypes norms:")
spo_norms = np.linalg.norm(species_organ_prototypes, axis=1)
print("min:", float(spo_norms.min()))
print("mean:", float(spo_norms.mean()))
print("max:", float(spo_norms.max()))

missing_species = set(species_candidates["species_id"].astype(int)) - set(species_proto_meta["species_id"].astype(int))
print("\nSpecies candidatas B:", species_candidates["species_id"].nunique())
print("Species con prototipo:", species_proto_meta["species_id"].nunique())
print("Species candidatas sin prototipo:", len(missing_species))

if len(missing_species) > 0:
    print("Ejemplos sin prototipo:", list(missing_species)[:20])
    raise RuntimeError("Hay especies candidatas sin prototipo.")

display(species_proto_meta.head())
display(species_organ_proto_meta.head())

# Limpieza
del ref_emb, ref_meta, ref_meta_full, reference_bank
gc.collect()

print("\n Prototipos visuales construidos.")

# **CELL 16 — Extract test embeddings for relevant views**

In [ ]:
import json
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"

BASE_RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"
BASE_FULL_INFER_DIR = BASE_RUN_DIR / "full_test_inference"
BASE_VIEWS_DIR = BASE_FULL_INFER_DIR / "views"

PROTOTYPE_DIR = STORAGE_ROOT / "runs" / "prototype_rerank_B"
TEST_EMB_DIR = PROTOTYPE_DIR / "test_embeddings"
TEST_EMB_CHUNKS_DIR = TEST_EMB_DIR / "chunks"

TEST_EMB_DIR.mkdir(parents=True, exist_ok=True)
TEST_EMB_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"

SELECTED_VIEWS_PATH = TEST_EMB_DIR / "test_selected_views.csv"

print("=" * 80)
print("CELDA 27 — EXTRAER EMBEDDINGS DE TEST PARA VISTAS RELEVANTES")
print("=" * 80)

print("TEST_PATH:", TEST_PATH, "| existe:", TEST_PATH.exists())
print("BASE_VIEWS_DIR:", BASE_VIEWS_DIR, "| existe:", BASE_VIEWS_DIR.exists())
print("TEST_EMB_DIR:", TEST_EMB_DIR)

if not TEST_PATH.exists():
    raise FileNotFoundError(TEST_PATH)

if not BASE_VIEWS_DIR.exists():
    raise FileNotFoundError(
        f"No existe BASE_VIEWS_DIR: {BASE_VIEWS_DIR}. "
        "Necesitamos las vistas ya generadas en la inferencia base."
    )


if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible.")

DEVICE = torch.device("cuda")

if "model" in globals():
    feature_model = model
    FEATURE_MODEL_NAME = "base_state_dict_model"
else:
    raise RuntimeError(
        "No encontré 'model' base en memoria. "
        "Los prototipos fueron creados con el modelo base; para consistencia, recarga el modelo base antes de esta celda."
    )

feature_model = feature_model.to(DEVICE)
feature_model.eval()

print("\nModelo extractor:", FEATURE_MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))


if "CFG" in globals():
    IMG_SIZE = int(CFG["model"]["img_size"])
    TILE_SIZE = int(CFG.get("tiling", {}).get("tile_size", 518))
else:
    IMG_SIZE = 518
    TILE_SIZE = 518

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = False
AMP = True


TOP_N_TILE_VIEWS = 24

CHUNK_SIZE_VIEWS = 4000

print("\nConfiguración:")
print("IMG_SIZE:", IMG_SIZE)
print("TILE_SIZE:", TILE_SIZE)
print("TOP_N_TILE_VIEWS:", TOP_N_TILE_VIEWS)
print("BATCH_SIZE:", BATCH_SIZE)
print("CHUNK_SIZE_VIEWS:", CHUNK_SIZE_VIEWS)
print("AMP:", AMP)


MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def safe_open_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    return img

def resize_square_pil(img: Image.Image, size: int):
    return img.convert("RGB").resize((size, size), Image.BICUBIC)

def center_crop_square(img: Image.Image):
    img = img.convert("RGB")
    w, h = img.size
    side = min(w, h)
    left = (w - side) // 2
    top = (h - side) // 2
    return img.crop((left, top, left + side, top + side))

def pil_to_tensor(img: Image.Image):
    img = img.convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1)
    x = (x - MEAN) / STD
    return x

def extract_dinov2_embedding(model_obj, images):
    """
    Extrae embeddings L2-normalizados desde forward_features.
    Debe ser consistente con CELDA 25.
    """
    with torch.no_grad():
        feats = model_obj.forward_features(images)

        if isinstance(feats, dict):
            if "x_norm_clstoken" in feats:
                emb = feats["x_norm_clstoken"]
            elif "pooled" in feats:
                emb = feats["pooled"]
            elif "features" in feats:
                emb = feats["features"]
                if emb.ndim == 3:
                    emb = emb[:, 0]
            else:
                emb = None
                for v in feats.values():
                    if torch.is_tensor(v):
                        if v.ndim == 2:
                            emb = v
                            break
                        if v.ndim == 3:
                            emb = v[:, 0]
                            break
                if emb is None:
                    raise RuntimeError(f"No pude extraer embedding desde dict keys={list(feats.keys())}")

        elif torch.is_tensor(feats):
            if feats.ndim == 3:
                try:
                    emb = model_obj.forward_head(feats, pre_logits=True)
                    if emb.ndim == 3:
                        emb = emb[:, 0]
                except Exception:
                    emb = feats[:, 0]
            elif feats.ndim == 2:
                emb = feats
            else:
                raise RuntimeError(f"Shape de features no soportado: {tuple(feats.shape)}")
        else:
            raise RuntimeError(f"Tipo de features no soportado: {type(feats)}")

        emb = emb.float()
        emb = F.normalize(emb, p=2, dim=1)

    return emb


test_df = pd.read_pickle(TEST_PATH, compression="gzip")
test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)
test_df["file_path"] = test_df["file_path"].astype(str)

test_df["file_exists"] = test_df["file_path"].apply(lambda p: Path(p).exists())

print("\nTest shape:", test_df.shape)
print("Quadrats:", test_df["quadrat_id"].nunique())
print("Archivos existentes:", test_df["file_exists"].sum(), "/", len(test_df))

if (~test_df["file_exists"]).sum() > 0:
    display(test_df.loc[~test_df["file_exists"], ["quadrat_id", "file_path"]].head())
    raise RuntimeError("Hay imágenes faltantes en test.")


if SELECTED_VIEWS_PATH.exists():
    print("\nCargando vistas seleccionadas existentes:")
    print(SELECTED_VIEWS_PATH)
    selected_views = pd.read_csv(SELECTED_VIEWS_PATH)

else:
    print("\n" + "=" * 80)
    print("CONSTRUYENDO LISTA DE VISTAS RELEVANTES")
    print("=" * 80)

    view_files = sorted(BASE_VIEWS_DIR.glob("views_chunk_*.csv"))
    print("Archivos de vistas encontrados:", len(view_files))

    if len(view_files) == 0:
        raise FileNotFoundError(f"No encontré views_chunk_*.csv en {BASE_VIEWS_DIR}")

    view_parts = []

    for vf in tqdm(view_files, desc="load_base_views"):
        dfv = pd.read_csv(vf)

        needed = ["quadrat_id", "file_path", "view_id", "view_type", "x", "y", "green_ratio"]
        missing = [c for c in needed if c not in dfv.columns]
        if missing:
            raise RuntimeError(f"Faltan columnas en {vf}: {missing}")

        dfv = dfv[needed].copy()
        dfv["quadrat_id"] = dfv["quadrat_id"].astype(str)
        dfv["view_type"] = dfv["view_type"].astype(str)
        dfv["green_ratio"] = pd.to_numeric(dfv["green_ratio"], errors="coerce").fillna(0.0)

        view_parts.append(dfv)

    all_views = pd.concat(view_parts, ignore_index=True)

    print("all_views shape:", all_views.shape)
    print("Quadrats en all_views:", all_views["quadrat_id"].nunique())

    selected_parts = []

    for qid, g in tqdm(all_views.groupby("quadrat_id"), desc="select_views_per_quadrat"):
        g = g.copy()

        
        keep_full_center = g[g["view_type"].isin(["full", "center", "small_resized"])].copy()

       
        tiles = g[g["view_type"] == "tile"].copy()
        tiles = tiles.sort_values("green_ratio", ascending=False).head(TOP_N_TILE_VIEWS)

        selected = pd.concat([keep_full_center, tiles], ignore_index=True)

        
        selected = selected.drop_duplicates(subset=["quadrat_id", "view_id"]).copy()

        selected_parts.append(selected)

    selected_views = pd.concat(selected_parts, ignore_index=True)

    
    selected_views = selected_views.sort_values(
        ["quadrat_id", "view_type", "green_ratio"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    selected_views["selected_view_global_id"] = [
        f"TESTVIEW_{i:08d}" for i in range(len(selected_views))
    ]

    selected_views.to_csv(SELECTED_VIEWS_PATH, index=False)

    del all_views, view_parts, selected_parts
    gc.collect()

print("\n" + "=" * 80)
print("RESUMEN VISTAS SELECCIONADAS")
print("=" * 80)

print("selected_views shape:", selected_views.shape)
print("Quadrats:", selected_views["quadrat_id"].nunique())

display(
    selected_views
    .groupby("quadrat_id")
    .size()
    .reset_index(name="n_selected_views")
    .describe()
)

display(
    selected_views["view_type"]
    .value_counts()
    .reset_index(name="n_views")
)

print("\nGreen ratio selected:")
display(selected_views["green_ratio"].describe())


class TestSelectedViewsDataset(Dataset):
    def __init__(self, views_df):
        self.df = views_df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        path = r["file_path"]
        view_type = r["view_type"]

        try:
            img = safe_open_image(path)

            if view_type == "full":
                view = resize_square_pil(img, IMG_SIZE)

            elif view_type == "center":
                view = resize_square_pil(center_crop_square(img), IMG_SIZE)

            elif view_type == "small_resized":
                view = resize_square_pil(img, IMG_SIZE)

            elif view_type == "tile":
                x0 = int(r["x"])
                y0 = int(r["y"])
                view = img.crop((x0, y0, x0 + TILE_SIZE, y0 + TILE_SIZE))
                view = resize_square_pil(view, IMG_SIZE)

            else:
                view = resize_square_pil(img, IMG_SIZE)

            x = pil_to_tensor(view)
            ok = True
            error = ""

        except Exception as e:
            x = torch.zeros(3, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
            ok = False
            error = str(e)

        return {
            "image": x,
            "selected_view_global_id": r["selected_view_global_id"],
            "quadrat_id": r["quadrat_id"],
            "file_path": r["file_path"],
            "view_id": int(r["view_id"]),
            "view_type": r["view_type"],
            "x": int(r["x"]),
            "y": int(r["y"]),
            "green_ratio": float(r["green_ratio"]),
            "ok": ok,
            "error": error,
        }

def collate_test_views(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)

    return {
        "image": images,
        "selected_view_global_id": [b["selected_view_global_id"] for b in batch],
        "quadrat_id": [b["quadrat_id"] for b in batch],
        "file_path": [b["file_path"] for b in batch],
        "view_id": [b["view_id"] for b in batch],
        "view_type": [b["view_type"] for b in batch],
        "x": [b["x"] for b in batch],
        "y": [b["y"] for b in batch],
        "green_ratio": [b["green_ratio"] for b in batch],
        "ok": [b["ok"] for b in batch],
        "error": [b["error"] for b in batch],
    }

n_views = len(selected_views)
n_chunks = int(np.ceil(n_views / CHUNK_SIZE_VIEWS))

chunk_plan = []

for chunk_id in range(n_chunks):
    start_idx = chunk_id * CHUNK_SIZE_VIEWS
    end_idx = min((chunk_id + 1) * CHUNK_SIZE_VIEWS, n_views)

    emb_path = TEST_EMB_CHUNKS_DIR / f"test_embeddings_chunk_{chunk_id:04d}.npy"
    meta_path = TEST_EMB_CHUNKS_DIR / f"test_metadata_chunk_{chunk_id:04d}.csv"

    chunk_plan.append({
        "chunk_id": chunk_id,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "n_views": end_idx - start_idx,
        "emb_path": str(emb_path),
        "meta_path": str(meta_path),
        "done": emb_path.exists() and meta_path.exists(),
    })

chunk_plan_df = pd.DataFrame(chunk_plan)
chunk_plan_path = TEST_EMB_DIR / "test_embedding_chunk_plan.csv"
chunk_plan_df.to_csv(chunk_plan_path, index=False)

print("\n" + "=" * 80)
print("PLAN DE CHUNKS TEST EMBEDDINGS")
print("=" * 80)
print("n_views:", n_views)
print("n_chunks:", n_chunks)
print("chunks completos:", chunk_plan_df["done"].sum(), "/", n_chunks)
display(chunk_plan_df.head())


start_total = time.time()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

completed_now = 0
skipped = 0

for item in chunk_plan:
    chunk_id = int(item["chunk_id"])
    start_idx = int(item["start_idx"])
    end_idx = int(item["end_idx"])

    emb_path = Path(item["emb_path"])
    meta_path = Path(item["meta_path"])

    print("\n" + "=" * 100)
    print(f"TEST EMB CHUNK {chunk_id:04d}/{n_chunks-1:04d} | views {start_idx}:{end_idx}")
    print("=" * 100)

    if emb_path.exists() and meta_path.exists():
        print(" Chunk ya existe. Se omite.")
        skipped += 1
        continue

    df_chunk = selected_views.iloc[start_idx:end_idx].reset_index(drop=True).copy()

    dataset = TestSelectedViewsDataset(df_chunk)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_test_views,
    )

    emb_parts = []
    meta_rows = []

    chunk_start = time.time()

    feature_model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"extract_test_emb_{chunk_id:04d}"):
            images = batch["image"].to(DEVICE, non_blocking=False)

            with torch.autocast(device_type="cuda", enabled=AMP):
                emb = extract_dinov2_embedding(feature_model, images)

            emb_np = emb.detach().cpu().numpy().astype(np.float32)
            emb_parts.append(emb_np)

            for i in range(len(batch["selected_view_global_id"])):
                meta_rows.append({
                    "selected_view_global_id": batch["selected_view_global_id"][i],
                    "quadrat_id": batch["quadrat_id"][i],
                    "file_path": batch["file_path"][i],
                    "view_id": int(batch["view_id"][i]),
                    "view_type": batch["view_type"][i],
                    "x": int(batch["x"][i]),
                    "y": int(batch["y"][i]),
                    "green_ratio": float(batch["green_ratio"][i]),
                    "ok": bool(batch["ok"][i]),
                    "error": batch["error"][i],
                    "chunk_id": chunk_id,
                })

            del images, emb, emb_np

    embeddings_chunk = np.concatenate(emb_parts, axis=0)
    meta_chunk = pd.DataFrame(meta_rows)

    if embeddings_chunk.shape[0] != len(meta_chunk):
        raise RuntimeError(
            f"Desalineación chunk {chunk_id}: embeddings {embeddings_chunk.shape[0]} vs meta {len(meta_chunk)}"
        )

    np.save(emb_path, embeddings_chunk)
    meta_chunk.to_csv(meta_path, index=False)

    chunk_time = time.time() - chunk_start

    print("Embeddings chunk shape:", embeddings_chunk.shape)
    print("Meta chunk shape:", meta_chunk.shape)
    print("Guardado:", emb_path)
    print("Guardado:", meta_path)
    print("Tiempo chunk:", round(chunk_time / 60, 2), "min")
    print("GPU asignada GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("GPU reservada GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

    completed_now += 1

    del df_chunk, dataset, loader, emb_parts, embeddings_chunk, meta_chunk, meta_rows
    gc.collect()
    torch.cuda.empty_cache()


print("\n" + "=" * 80)
print("COMBINANDO EMBEDDINGS TEST")
print("=" * 80)

emb_chunk_files = sorted(TEST_EMB_CHUNKS_DIR.glob("test_embeddings_chunk_*.npy"))
meta_chunk_files = sorted(TEST_EMB_CHUNKS_DIR.glob("test_metadata_chunk_*.csv"))

print("Emb chunks:", len(emb_chunk_files), "/", n_chunks)
print("Meta chunks:", len(meta_chunk_files), "/", n_chunks)

if len(emb_chunk_files) != n_chunks or len(meta_chunk_files) != n_chunks:
    raise RuntimeError("Faltan chunks de embeddings o metadata. Reejecuta la celda para completar.")

all_emb_parts = []
all_meta_parts = []

for emb_p, meta_p in tqdm(list(zip(emb_chunk_files, meta_chunk_files)), desc="load_test_emb_chunks"):
    all_emb_parts.append(np.load(emb_p))
    all_meta_parts.append(pd.read_csv(meta_p))

test_embeddings_all = np.concatenate(all_emb_parts, axis=0).astype(np.float32)
test_metadata_all = pd.concat(all_meta_parts, ignore_index=True)


norms = np.linalg.norm(test_embeddings_all, axis=1, keepdims=True)
test_embeddings_all = test_embeddings_all / np.maximum(norms, 1e-12)

if test_embeddings_all.shape[0] != len(test_metadata_all):
    raise RuntimeError(
        f"Desalineación final: embeddings {test_embeddings_all.shape[0]} vs metadata {len(test_metadata_all)}"
    )

all_emb_path = TEST_EMB_DIR / "test_embeddings_selected_all.npy"
all_meta_path = TEST_EMB_DIR / "test_embeddings_selected_metadata.csv"
config_path = TEST_EMB_DIR / "test_embedding_config.json"

np.save(all_emb_path, test_embeddings_all)
test_metadata_all.to_csv(all_meta_path, index=False)

config = {
    "feature_model_name": FEATURE_MODEL_NAME,
    "img_size": IMG_SIZE,
    "tile_size": TILE_SIZE,
    "top_n_tile_views": TOP_N_TILE_VIEWS,
    "batch_size": BATCH_SIZE,
    "chunk_size_views": CHUNK_SIZE_VIEWS,
    "n_selected_views": int(test_embeddings_all.shape[0]),
    "embedding_dim": int(test_embeddings_all.shape[1]),
    "n_quadrats": int(test_metadata_all["quadrat_id"].nunique()),
    "selected_views_path": str(SELECTED_VIEWS_PATH),
    "embeddings_path": str(all_emb_path),
    "metadata_path": str(all_meta_path),
    "chunk_plan_path": str(chunk_plan_path),
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

elapsed_total = time.time() - start_total

print("\n" + "=" * 80)
print("RESUMEN FINAL EMBEDDINGS TEST")
print("=" * 80)
print("test_embeddings_all shape:", test_embeddings_all.shape)
print("test_metadata_all shape:", test_metadata_all.shape)
print("Quadrats con embeddings:", test_metadata_all["quadrat_id"].nunique())
print("OK views:", test_metadata_all["ok"].sum(), "/", len(test_metadata_all))
print("Tiempo total esta corrida:", round(elapsed_total / 60, 2), "min")

print("\nVistas por quadrat:")
display(
    test_metadata_all
    .groupby("quadrat_id")
    .size()
    .reset_index(name="n_views")
    .describe()
)

print("\nView types:")
display(test_metadata_all["view_type"].value_counts().reset_index(name="n_views"))

print("\nNormas embeddings test:")
test_norms = np.linalg.norm(test_embeddings_all, axis=1)
print("min:", float(test_norms.min()))
print("mean:", float(test_norms.mean()))
print("max:", float(test_norms.max()))

print("\nArchivos guardados:")
print("Selected views:", SELECTED_VIEWS_PATH)
print("Embeddings:", all_emb_path)
print("Metadata:", all_meta_path)
print("Config:", config_path)

display(test_metadata_all.head())

# Limpieza ligera
del all_emb_parts, all_meta_parts
gc.collect()
torch.cuda.empty_cache()

print("\n Embeddings de test relevantes listos.")

# **CELL 17 — Create a global reference bank for the 7806 species**

In [ ]:
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


SEED = 42
rng = np.random.default_rng(SEED)

STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
RAW_METADATA_DIR = STORAGE_ROOT / "data" / "raw" / "metadata"

GLOBAL_PROTO_DIR = STORAGE_ROOT / "runs" / "global_prototype_recovery"
GLOBAL_PROTO_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"

TRAIN_CANDIDATE_PATHS = [
    INDEX_DIR / "train_manifest_enriched.pkl.gz",
    INDEX_DIR / "train_manifest_with_metadata.pkl.gz",
    INDEX_DIR / "train_manifest_metadata.pkl.gz",
    INDEX_DIR / "train_manifest.pkl.gz",
]

VAL_CANDIDATE_PATHS = [
    INDEX_DIR / "val_manifest_enriched.pkl.gz",
    INDEX_DIR / "val_manifest_with_metadata.pkl.gz",
    INDEX_DIR / "val_manifest_metadata.pkl.gz",
    INDEX_DIR / "val_manifest.pkl.gz",
]

METADATA_CANDIDATE_PATHS = [
    RAW_METADATA_DIR / "PlantCLEF2024_single_plant_training_metadata.csv",
    STORAGE_ROOT / "PlantCLEF2024_single_plant_training_metadata.csv",
    Path("/workspace/notebooks/andrea/PlantCLEF2024_single_plant_training_metadata.csv"),
    Path("/workspace/notebooks/andrea/plantclef-2026/PlantCLEF2024_single_plant_training_metadata.csv"),
]


MAX_REF_PER_SPECIES = 6
MAX_VAL_PER_SPECIES = 2
MAX_TRAIN_PER_SPECIES = 4
MAX_PER_ORGAN = 2

PREFERRED_ORGANS = [
    "habit",
    "leaf",
    "flower",
    "fruit",
    "bark",
    "branch",
    "scan",
    "unknown",
]

print("=" * 80)
print("CELDA 31 — BANCO GLOBAL DE REFERENCIAS PARA 7806 ESPECIES")
print("=" * 80)

print("STORAGE_ROOT:", STORAGE_ROOT)
print("INDEX_DIR:", INDEX_DIR)
print("GLOBAL_PROTO_DIR:", GLOBAL_PROTO_DIR)
print("SPECIES_TABLE_PATH:", SPECIES_TABLE_PATH, "| existe:", SPECIES_TABLE_PATH.exists())

if not SPECIES_TABLE_PATH.exists():
    raise FileNotFoundError(SPECIES_TABLE_PATH)


def find_existing_path(candidate_paths, label):
    print(f"\nBuscando {label}:")
    for p in candidate_paths:
        print(" -", p, "| existe:", p.exists())
        if p.exists():
            print(f"Usando {label}:", p)
            return p
    raise FileNotFoundError(f"No encontré archivo para {label}")


def load_metadata_organ_map():
    """
    Carga solo image_name y organ desde el CSV grande de metadata.
    Se usa únicamente si los manifests no traen organ.
    """
    metadata_path = None

    print("\nBuscando metadata CSV para organ:")
    for p in METADATA_CANDIDATE_PATHS:
        print(" -", p, "| existe:", p.exists())
        if p.exists():
            metadata_path = p
            break

    if metadata_path is None:
        print(" No encontré metadata CSV. Se usará organ='unknown'.")
        return None, None

    print("Cargando metadata mínima:", metadata_path)

    try:
        meta = pd.read_csv(
            metadata_path,
            sep=";",
            usecols=["image_name", "organ"],
            dtype={"image_name": "string", "organ": "string"},
            low_memory=False,
        )
    except Exception as e:
        print("Falló lectura con usecols. Intentando lectura flexible...")
        meta = pd.read_csv(
            metadata_path,
            sep=";",
            dtype=str,
            low_memory=False,
        )
        needed = ["image_name", "organ"]
        missing = [c for c in needed if c not in meta.columns]
        if missing:
            raise RuntimeError(f"No encontré columnas {missing} en metadata.")
        meta = meta[needed].copy()

    meta["image_name"] = meta["image_name"].astype(str)
    meta["organ"] = meta["organ"].fillna("unknown").astype(str).str.lower()

    meta = meta.drop_duplicates("image_name").reset_index(drop=True)

    print("Metadata organ map shape:", meta.shape)
    print("Órganos metadata:")
    display(meta["organ"].value_counts().reset_index(name="n_images").head(20))

    return meta, metadata_path


def load_manifest_minimal(path, split_name, metadata_organ_map=None):
    """
    Normaliza un manifest a columnas mínimas:
    file_path, file_name, species_id, organ, split_source.
    """
    print("\n" + "=" * 80)
    print(f"CARGANDO MANIFEST {split_name.upper()}")
    print("=" * 80)
    print("Path:", path)

    if path.suffix == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_pickle(path, compression="gzip")

    print("Shape original:", df.shape)
    print("Columnas iniciales:", list(df.columns)[:40])

    df = df.copy()

    # file_path
    if "file_path" not in df.columns:
        raise RuntimeError(f"{split_name}: no tiene columna file_path.")

    df["file_path"] = df["file_path"].astype(str)

    # file_name
    if "file_name" not in df.columns:
        df["file_name"] = df["file_path"].apply(lambda p: Path(p).name)
    df["file_name"] = df["file_name"].astype(str)

    # species_id
    if "species_id" not in df.columns:
        if "folder_label" in df.columns:
            df["species_id"] = pd.to_numeric(df["folder_label"], errors="coerce")
        else:
            raise RuntimeError(f"{split_name}: no tiene species_id ni folder_label.")

    df["species_id"] = pd.to_numeric(df["species_id"], errors="coerce")
    df = df.dropna(subset=["species_id"]).copy()
    df["species_id"] = df["species_id"].astype(int)

    # organ
    if "organ" in df.columns:
        df["organ"] = df["organ"].fillna("unknown").astype(str).str.lower()
    else:
        df["organ"] = "unknown"

    
    organ_unknown_rate = float((df["organ"] == "unknown").mean())

    if metadata_organ_map is not None and organ_unknown_rate > 0.95:
        print(f"{split_name}: organ parece ausente o casi todo unknown. Haciendo merge con metadata...")
        df = df.merge(
            metadata_organ_map,
            left_on="file_name",
            right_on="image_name",
            how="left",
            suffixes=("", "_meta")
        )

        if "organ_meta" in df.columns:
            df["organ"] = df["organ_meta"].fillna(df["organ"]).fillna("unknown")
            df = df.drop(columns=["organ_meta", "image_name"], errors="ignore")
        else:
            df = df.drop(columns=["image_name"], errors="ignore")

        df["organ"] = df["organ"].fillna("unknown").astype(str).str.lower()

    keep_cols = [
        "file_path",
        "file_name",
        "species_id",
        "organ",
    ]

    df = df[keep_cols].copy()
    df["split_source"] = split_name

   
    df["organ"] = df["organ"].astype("category")
    df["split_source"] = df["split_source"].astype("category")

    print("Shape minimal:", df.shape)
    print("Species únicas:", df["species_id"].nunique())
    print("Órganos:")
    display(df["organ"].value_counts().reset_index(name="n_images").head(20))

    return df


def stratified_sample_one_species(g):
    """
    Muestreo por especie:
    - primero val
    - luego train
    - balance por órgano
    """
    g = g.copy()

    organ_priority = {org: i for i, org in enumerate(PREFERRED_ORGANS)}
    g["organ_str"] = g["organ"].astype(str)
    g["organ_priority"] = g["organ_str"].map(organ_priority).fillna(len(PREFERRED_ORGANS)).astype(int)

    selected_idx = []

    for split_name, max_split in [("val", MAX_VAL_PER_SPECIES), ("train", MAX_TRAIN_PER_SPECIES)]:
        gs = g[g["split_source"].astype(str) == split_name].copy()

        if len(gs) == 0:
            continue

        split_selected = []

        
        for org in PREFERRED_ORGANS:
            go = gs[gs["organ_str"] == org].copy()

            if len(go) == 0:
                continue

            n_remaining_split = max_split - len(split_selected)
            if n_remaining_split <= 0:
                break

            n_take = min(MAX_PER_ORGAN, len(go), n_remaining_split)

            sampled_idx = rng.choice(go.index.values, size=n_take, replace=False)
            split_selected.extend(sampled_idx.tolist())

            if len(split_selected) >= max_split:
                break

        
        if len(split_selected) < max_split:
            remaining = gs[~gs.index.isin(split_selected)].copy()
            if len(remaining) > 0:
                n_take = min(max_split - len(split_selected), len(remaining))
                sampled_idx = rng.choice(remaining.index.values, size=n_take, replace=False)
                split_selected.extend(sampled_idx.tolist())

        selected_idx.extend(split_selected)

    if len(selected_idx) == 0:
        return g.head(0).drop(columns=["organ_str", "organ_priority"], errors="ignore")

    selected = g.loc[selected_idx].copy()
    selected = selected.drop_duplicates("file_path")

    
    if len(selected) > MAX_REF_PER_SPECIES:
        selected = selected.sort_values(["organ_priority", "split_source"]).copy()
        sampled_idx = rng.choice(selected.index.values, size=MAX_REF_PER_SPECIES, replace=False)
        selected = selected.loc[sampled_idx].copy()

    selected = selected.drop(columns=["organ_str", "organ_priority"], errors="ignore")
    return selected



print("\n" + "=" * 80)
print("CARGANDO SPECIES TABLE")
print("=" * 80)

species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")
species_table["species_id"] = species_table["species_id"].astype(int)

ALL_SPECIES = sorted(species_table["species_id"].astype(int).unique().tolist())

print("Species table shape:", species_table.shape)
print("Número de species_id:", len(ALL_SPECIES))
print("species_id min:", min(ALL_SPECIES))
print("species_id max:", max(ALL_SPECIES))

display(species_table.head())
display(species_table.tail())


metadata_organ_map, metadata_path = load_metadata_organ_map()


TRAIN_PATH = find_existing_path(TRAIN_CANDIDATE_PATHS, "train manifest")
VAL_PATH = find_existing_path(VAL_CANDIDATE_PATHS, "val manifest")

train_df = load_manifest_minimal(TRAIN_PATH, "train", metadata_organ_map=metadata_organ_map)
val_df = load_manifest_minimal(VAL_PATH, "val", metadata_organ_map=metadata_organ_map)


del metadata_organ_map
gc.collect()


print("\n" + "=" * 80)
print("UNIENDO TRAIN + VAL")
print("=" * 80)

reference_pool = pd.concat([val_df, train_df], ignore_index=True)

del train_df, val_df
gc.collect()

reference_pool = reference_pool.drop_duplicates("file_path").reset_index(drop=True)

print("reference_pool shape:", reference_pool.shape)
print("Species en pool:", reference_pool["species_id"].nunique())
print("Split distribution:")
display(reference_pool["split_source"].astype(str).value_counts().reset_index(name="n_images"))

print("Organ distribution:")
display(reference_pool["organ"].astype(str).value_counts().reset_index(name="n_images").head(20))


valid_species_set = set(ALL_SPECIES)

reference_pool = reference_pool[reference_pool["species_id"].isin(valid_species_set)].copy()

print("\nDespués de filtrar a species_table:")
print("reference_pool shape:", reference_pool.shape)
print("Species en pool:", reference_pool["species_id"].nunique())


print("\n" + "=" * 80)
print("MUESTREANDO REFERENCIAS GLOBALES")
print("=" * 80)

sampled_parts = []
missing_species = []


groups = dict(tuple(reference_pool.groupby("species_id", sort=False)))

for sp in tqdm(ALL_SPECIES, desc="sample_global_species"):
    if sp not in groups:
        missing_species.append(sp)
        continue

    g = groups[sp]

    sampled = stratified_sample_one_species(g)

    if len(sampled) == 0:
        missing_species.append(sp)
        continue

    sampled_parts.append(sampled)

if len(sampled_parts) == 0:
    raise RuntimeError("No se pudo construir el banco global de referencias.")

global_reference_bank = pd.concat(sampled_parts, ignore_index=True)


del reference_pool, groups, sampled_parts
gc.collect()


global_reference_bank = global_reference_bank.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left"
)

global_reference_bank = global_reference_bank.reset_index(drop=True)
global_reference_bank["global_ref_id"] = [
    f"GREF_{i:08d}" for i in range(len(global_reference_bank))
]


first_cols = [
    "global_ref_id",
    "species_id",
    "file_path",
    "file_name",
    "split_source",
    "organ",
    "species",
    "genus",
    "family",
]

other_cols = [c for c in global_reference_bank.columns if c not in first_cols]
global_reference_bank = global_reference_bank[first_cols + other_cols].copy()


print("\n" + "=" * 80)
print("VERIFICANDO ARCHIVOS SELECCIONADOS")
print("=" * 80)

global_reference_bank["file_exists"] = global_reference_bank["file_path"].apply(lambda p: Path(p).exists())

n_exists = int(global_reference_bank["file_exists"].sum())
n_total = len(global_reference_bank)

print("Archivos existentes:", n_exists, "/", n_total)

if n_exists < n_total:
    print(" Hay archivos faltantes. Se eliminarán del banco.")
    display(
        global_reference_bank
        .loc[~global_reference_bank["file_exists"], ["species_id", "file_path"]]
        .head(20)
    )

    global_reference_bank = global_reference_bank[global_reference_bank["file_exists"]].copy()
    global_reference_bank = global_reference_bank.reset_index(drop=True)
    global_reference_bank["global_ref_id"] = [
        f"GREF_{i:08d}" for i in range(len(global_reference_bank))
    ]


species_with_refs = set(global_reference_bank["species_id"].astype(int).unique().tolist())
missing_species_final = sorted(list(valid_species_set - species_with_refs))

missing_species_df = pd.DataFrame({"species_id": missing_species_final})
missing_species_df = missing_species_df.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left"
)


species_summary = (
    global_reference_bank
    .groupby("species_id")
    .agg(
        n_refs=("global_ref_id", "size"),
        n_train_refs=("split_source", lambda x: int((x.astype(str) == "train").sum())),
        n_val_refs=("split_source", lambda x: int((x.astype(str) == "val").sum())),
        n_organs=("organ", lambda x: int(x.astype(str).nunique())),
        organs=("organ", lambda x: ",".join(sorted(set(x.astype(str))))),
        species=("species", "first"),
        genus=("genus", "first"),
        family=("family", "first"),
    )
    .reset_index()
)

print("\n" + "=" * 80)
print("RESUMEN BANCO GLOBAL")
print("=" * 80)

print("global_reference_bank shape:", global_reference_bank.shape)
print("Species cubiertas:", global_reference_bank["species_id"].nunique(), "/", len(ALL_SPECIES))
print("Species sin referencias:", len(missing_species_final))
print("Promedio refs/species:", round(species_summary["n_refs"].mean(), 3))
print("Mediana refs/species:", round(species_summary["n_refs"].median(), 3))
print("Min refs/species:", int(species_summary["n_refs"].min()))
print("Max refs/species:", int(species_summary["n_refs"].max()))

print("\nDistribución refs por split:")
display(global_reference_bank["split_source"].astype(str).value_counts().reset_index(name="n_refs"))

print("\nDistribución refs por órgano:")
display(global_reference_bank["organ"].astype(str).value_counts().reset_index(name="n_refs"))

print("\nDistribución n_refs por especie:")
display(species_summary["n_refs"].describe())

print("\nTop especies con menos refs:")
display(
    species_summary
    .sort_values(["n_refs", "species_id"], ascending=[True, True])
    .head(30)
)

if len(missing_species_final) > 0:
    print("\nEspecies sin referencia:")
    display(missing_species_df.head(30))


bank_csv_path = GLOBAL_PROTO_DIR / "global_reference_bank.csv"
bank_pkl_path = GLOBAL_PROTO_DIR / "global_reference_bank.pkl.gz"
missing_path = GLOBAL_PROTO_DIR / "global_missing_species.csv"
species_summary_path = GLOBAL_PROTO_DIR / "global_reference_species_summary.csv"
config_path = GLOBAL_PROTO_DIR / "global_reference_config.json"

global_reference_bank.to_csv(bank_csv_path, index=False)
global_reference_bank.to_pickle(bank_pkl_path, compression="gzip")
missing_species_df.to_csv(missing_path, index=False)
species_summary.to_csv(species_summary_path, index=False)

config = {
    "seed": SEED,
    "strategy": "global_reference_bank_for_all_species",
    "n_species_total": int(len(ALL_SPECIES)),
    "n_species_with_refs": int(global_reference_bank["species_id"].nunique()),
    "n_missing_species": int(len(missing_species_final)),
    "n_reference_images": int(len(global_reference_bank)),
    "max_ref_per_species": MAX_REF_PER_SPECIES,
    "max_val_per_species": MAX_VAL_PER_SPECIES,
    "max_train_per_species": MAX_TRAIN_PER_SPECIES,
    "max_per_organ": MAX_PER_ORGAN,
    "preferred_organs": PREFERRED_ORGANS,
    "species_table_path": str(SPECIES_TABLE_PATH),
    "train_manifest_path": str(TRAIN_PATH),
    "val_manifest_path": str(VAL_PATH),
    "metadata_path": str(metadata_path) if metadata_path is not None else None,
    "outputs": {
        "global_reference_bank_csv": str(bank_csv_path),
        "global_reference_bank_pkl": str(bank_pkl_path),
        "global_missing_species": str(missing_path),
        "global_reference_species_summary": str(species_summary_path),
    },
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\n" + "=" * 80)
print("ARCHIVOS GUARDADOS")
print("=" * 80)
print("Global reference bank CSV:", bank_csv_path)
print("Global reference bank PKL:", bank_pkl_path)
print("Missing species:", missing_path)
print("Species summary:", species_summary_path)
print("Config:", config_path)

display(global_reference_bank.head())

gc.collect()

print("\n Banco global listo para embeddings.")

# **CELL 18 — Extract global reference embeddings**

In [ ]:
import json
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
GLOBAL_PROTO_DIR = STORAGE_ROOT / "runs" / "global_prototype_recovery"

GLOBAL_REF_BANK_PATH = GLOBAL_PROTO_DIR / "global_reference_bank.pkl.gz"

GLOBAL_EMB_DIR = GLOBAL_PROTO_DIR / "global_reference_embeddings"
GLOBAL_EMB_CHUNKS_DIR = GLOBAL_EMB_DIR / "chunks"

GLOBAL_EMB_DIR.mkdir(parents=True, exist_ok=True)
GLOBAL_EMB_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("CELDA 32 — EXTRAER EMBEDDINGS GLOBALES DE REFERENCIA")
print("=" * 80)
print("GLOBAL_REF_BANK_PATH:", GLOBAL_REF_BANK_PATH, "| existe:", GLOBAL_REF_BANK_PATH.exists())
print("GLOBAL_EMB_DIR:", GLOBAL_EMB_DIR)

if not GLOBAL_REF_BANK_PATH.exists():
    raise FileNotFoundError(GLOBAL_REF_BANK_PATH)


if not torch.cuda.is_available():
    raise RuntimeError("CUDA no está disponible.")

DEVICE = torch.device("cuda")

if "model" not in globals():
    raise RuntimeError(
        "No encontré el modelo base 'model' en memoria. "
        "Para mantener consistencia con los embeddings del test, recarga el modelo base "
        "de la CELDA 4 antes de ejecutar esta celda. No uses model_ema aquí."
    )

feature_model = model.to(DEVICE)
feature_model.eval()

FEATURE_MODEL_NAME = "base_state_dict_model"

print("\nModelo extractor:", FEATURE_MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))


if "CFG" in globals():
    IMG_SIZE = int(CFG["model"]["img_size"])
else:
    IMG_SIZE = 518

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = False
AMP = True

ks
CHUNK_SIZE_REFS = 4000

print("\nConfiguración:")
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)
print("NUM_WORKERS:", NUM_WORKERS)
print("PIN_MEMORY:", PIN_MEMORY)
print("AMP:", AMP)
print("CHUNK_SIZE_REFS:", CHUNK_SIZE_REFS)


MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def safe_open_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    return img

def resize_square_pil(img: Image.Image, size: int):
    return img.convert("RGB").resize((size, size), Image.BICUBIC)

def pil_to_tensor(img: Image.Image):
    img = img.convert("RGB")
    arr = np.array(img).astype(np.float32) / 255.0
    x = torch.from_numpy(arr).permute(2, 0, 1)
    x = (x - MEAN) / STD
    return x

def extract_dinov2_embedding(model_obj, images):
    """
    Extrae embeddings L2-normalizados desde forward_features.
    Debe coincidir con la lógica usada en CELDA 25 y CELDA 27.
    """
    with torch.no_grad():
        feats = model_obj.forward_features(images)

        if isinstance(feats, dict):
            if "x_norm_clstoken" in feats:
                emb = feats["x_norm_clstoken"]
            elif "pooled" in feats:
                emb = feats["pooled"]
            elif "features" in feats:
                emb = feats["features"]
                if emb.ndim == 3:
                    emb = emb[:, 0]
            else:
                emb = None
                for v in feats.values():
                    if torch.is_tensor(v):
                        if v.ndim == 2:
                            emb = v
                            break
                        if v.ndim == 3:
                            emb = v[:, 0]
                            break
                if emb is None:
                    raise RuntimeError(f"No pude extraer embedding desde dict keys={list(feats.keys())}")

        elif torch.is_tensor(feats):
            if feats.ndim == 3:
                try:
                    emb = model_obj.forward_head(feats, pre_logits=True)
                    if emb.ndim == 3:
                        emb = emb[:, 0]
                except Exception:
                    emb = feats[:, 0]
            elif feats.ndim == 2:
                emb = feats
            else:
                raise RuntimeError(f"Shape de features no soportado: {tuple(feats.shape)}")

        else:
            raise RuntimeError(f"Tipo de features no soportado: {type(feats)}")

        emb = emb.float()
        emb = F.normalize(emb, p=2, dim=1)

    return emb


class GlobalReferenceDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]

        try:
            img = safe_open_image(r["file_path"])
            img = resize_square_pil(img, IMG_SIZE)
            x = pil_to_tensor(img)
            ok = True
            error = ""
        except Exception as e:
            x = torch.zeros(3, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
            ok = False
            error = str(e)

        return {
            "image": x,
            "global_ref_id": r["global_ref_id"],
            "species_id": int(r["species_id"]),
            "file_path": r["file_path"],
            "file_name": r["file_name"],
            "organ": r["organ"],
            "split_source": r["split_source"],
            "ok": ok,
            "error": error,
        }

def collate_global_ref(batch):
    images = torch.stack([b["image"] for b in batch], dim=0)

    return {
        "image": images,
        "global_ref_id": [b["global_ref_id"] for b in batch],
        "species_id": [b["species_id"] for b in batch],
        "file_path": [b["file_path"] for b in batch],
        "file_name": [b["file_name"] for b in batch],
        "organ": [b["organ"] for b in batch],
        "split_source": [b["split_source"] for b in batch],
        "ok": [b["ok"] for b in batch],
        "error": [b["error"] for b in batch],
    }


global_ref_bank = pd.read_pickle(GLOBAL_REF_BANK_PATH, compression="gzip")

global_ref_bank["global_ref_id"] = global_ref_bank["global_ref_id"].astype(str)
global_ref_bank["species_id"] = global_ref_bank["species_id"].astype(int)
global_ref_bank["file_path"] = global_ref_bank["file_path"].astype(str)
global_ref_bank["file_name"] = global_ref_bank["file_name"].astype(str)
global_ref_bank["organ"] = global_ref_bank["organ"].fillna("unknown").astype(str).str.lower()
global_ref_bank["split_source"] = global_ref_bank["split_source"].astype(str)

print("\nBanco global cargado:")
print("Shape:", global_ref_bank.shape)
print("Species:", global_ref_bank["species_id"].nunique())
print("Refs:", len(global_ref_bank))


if "file_exists" in global_ref_bank.columns:
    print("file_exists sum:", int(global_ref_bank["file_exists"].sum()), "/", len(global_ref_bank))
else:
    global_ref_bank["file_exists"] = global_ref_bank["file_path"].apply(lambda p: Path(p).exists())
    print("file_exists sum:", int(global_ref_bank["file_exists"].sum()), "/", len(global_ref_bank))

if (~global_ref_bank["file_exists"]).sum() > 0:
    print(" Hay archivos faltantes. Se eliminarán antes de extraer embeddings.")
    display(global_ref_bank.loc[~global_ref_bank["file_exists"], ["global_ref_id", "species_id", "file_path"]].head())
    global_ref_bank = global_ref_bank[global_ref_bank["file_exists"]].reset_index(drop=True)


n_refs = len(global_ref_bank)
n_chunks = int(np.ceil(n_refs / CHUNK_SIZE_REFS))

chunk_plan = []

for chunk_id in range(n_chunks):
    start_idx = chunk_id * CHUNK_SIZE_REFS
    end_idx = min((chunk_id + 1) * CHUNK_SIZE_REFS, n_refs)

    emb_path = GLOBAL_EMB_CHUNKS_DIR / f"global_ref_embeddings_chunk_{chunk_id:04d}.npy"
    meta_path = GLOBAL_EMB_CHUNKS_DIR / f"global_ref_metadata_chunk_{chunk_id:04d}.csv"

    chunk_plan.append({
        "chunk_id": chunk_id,
        "start_idx": start_idx,
        "end_idx": end_idx,
        "n_refs": end_idx - start_idx,
        "emb_path": str(emb_path),
        "meta_path": str(meta_path),
        "done": emb_path.exists() and meta_path.exists(),
    })

chunk_plan_df = pd.DataFrame(chunk_plan)
chunk_plan_path = GLOBAL_EMB_DIR / "global_reference_embedding_chunk_plan.csv"
chunk_plan_df.to_csv(chunk_plan_path, index=False)

print("\nPlan de chunks:")
print("n_refs:", n_refs)
print("n_chunks:", n_chunks)
print("chunks completos:", int(chunk_plan_df["done"].sum()), "/", n_chunks)
print("chunk_plan:", chunk_plan_path)
display(chunk_plan_df)


start_total = time.time()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

completed_now = 0
skipped = 0

for item in chunk_plan:
    chunk_id = int(item["chunk_id"])
    start_idx = int(item["start_idx"])
    end_idx = int(item["end_idx"])

    emb_path = Path(item["emb_path"])
    meta_path = Path(item["meta_path"])

    print("\n" + "=" * 100)
    print(f"GLOBAL REF EMB CHUNK {chunk_id:04d}/{n_chunks-1:04d} | refs {start_idx}:{end_idx}")
    print("=" * 100)

    if emb_path.exists() and meta_path.exists():
        print(" Chunk ya existe. Se omite.")
        skipped += 1
        continue

    df_chunk = global_ref_bank.iloc[start_idx:end_idx].reset_index(drop=True).copy()

    dataset = GlobalReferenceDataset(df_chunk)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=collate_global_ref,
    )

    emb_parts = []
    meta_rows = []

    chunk_start = time.time()
    feature_model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"extract_global_ref_emb_{chunk_id:04d}"):
            images = batch["image"].to(DEVICE, non_blocking=False)

            with torch.autocast(device_type="cuda", enabled=AMP):
                emb = extract_dinov2_embedding(feature_model, images)

            emb_np = emb.detach().cpu().numpy().astype(np.float32)
            emb_parts.append(emb_np)

            for i in range(len(batch["global_ref_id"])):
                meta_rows.append({
                    "global_ref_id": batch["global_ref_id"][i],
                    "species_id": int(batch["species_id"][i]),
                    "file_path": batch["file_path"][i],
                    "file_name": batch["file_name"][i],
                    "organ": batch["organ"][i],
                    "split_source": batch["split_source"][i],
                    "ok": bool(batch["ok"][i]),
                    "error": batch["error"][i],
                    "chunk_id": chunk_id,
                })

            del images, emb, emb_np

    embeddings_chunk = np.concatenate(emb_parts, axis=0).astype(np.float32)
    meta_chunk = pd.DataFrame(meta_rows)

    if embeddings_chunk.shape[0] != len(meta_chunk):
        raise RuntimeError(
            f"Desalineación chunk {chunk_id}: embeddings {embeddings_chunk.shape[0]} vs meta {len(meta_chunk)}"
        )

    # Normalización por seguridad
    norms = np.linalg.norm(embeddings_chunk, axis=1, keepdims=True)
    embeddings_chunk = embeddings_chunk / np.maximum(norms, 1e-12)

    np.save(emb_path, embeddings_chunk)
    meta_chunk.to_csv(meta_path, index=False)

    chunk_time = time.time() - chunk_start

    print("Embeddings chunk shape:", embeddings_chunk.shape)
    print("Meta chunk shape:", meta_chunk.shape)
    print("Guardado:", emb_path)
    print("Guardado:", meta_path)
    print("Tiempo chunk:", round(chunk_time / 60, 2), "min")
    print("GPU asignada GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
    print("GPU reservada GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))

    completed_now += 1

    del df_chunk, dataset, loader, emb_parts, embeddings_chunk, meta_chunk, meta_rows
    gc.collect()
    torch.cuda.empty_cache()


print("\n" + "=" * 80)
print("COMBINANDO EMBEDDINGS GLOBALES")
print("=" * 80)

emb_chunk_files = sorted(GLOBAL_EMB_CHUNKS_DIR.glob("global_ref_embeddings_chunk_*.npy"))
meta_chunk_files = sorted(GLOBAL_EMB_CHUNKS_DIR.glob("global_ref_metadata_chunk_*.csv"))

print("Emb chunks:", len(emb_chunk_files), "/", n_chunks)
print("Meta chunks:", len(meta_chunk_files), "/", n_chunks)

if len(emb_chunk_files) != n_chunks or len(meta_chunk_files) != n_chunks:
    raise RuntimeError("Faltan chunks globales. Reejecuta esta celda para completar.")

all_emb_parts = []
all_meta_parts = []

for emb_p, meta_p in tqdm(list(zip(emb_chunk_files, meta_chunk_files)), desc="load_global_ref_emb_chunks"):
    all_emb_parts.append(np.load(emb_p).astype(np.float32))
    all_meta_parts.append(pd.read_csv(meta_p))

global_ref_embeddings_all = np.concatenate(all_emb_parts, axis=0).astype(np.float32)
global_ref_metadata_all = pd.concat(all_meta_parts, ignore_index=True)

if global_ref_embeddings_all.shape[0] != len(global_ref_metadata_all):
    raise RuntimeError(
        f"Desalineación final: embeddings {global_ref_embeddings_all.shape[0]} vs metadata {len(global_ref_metadata_all)}"
    )


norms = np.linalg.norm(global_ref_embeddings_all, axis=1, keepdims=True)
global_ref_embeddings_all = global_ref_embeddings_all / np.maximum(norms, 1e-12)


all_emb_path = GLOBAL_EMB_DIR / "global_reference_embeddings_all.npy"
all_meta_path = GLOBAL_EMB_DIR / "global_reference_embeddings_metadata.csv"
config_path = GLOBAL_EMB_DIR / "global_reference_embedding_config.json"

np.save(all_emb_path, global_ref_embeddings_all)
global_ref_metadata_all.to_csv(all_meta_path, index=False)

config = {
    "feature_model_name": FEATURE_MODEL_NAME,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "chunk_size_refs": CHUNK_SIZE_REFS,
    "n_reference_embeddings": int(global_ref_embeddings_all.shape[0]),
    "embedding_dim": int(global_ref_embeddings_all.shape[1]),
    "n_species": int(global_ref_metadata_all["species_id"].nunique()),
    "global_reference_bank_path": str(GLOBAL_REF_BANK_PATH),
    "embeddings_path": str(all_emb_path),
    "metadata_path": str(all_meta_path),
    "chunk_plan_path": str(chunk_plan_path),
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

elapsed_total = time.time() - start_total

print("\n" + "=" * 80)
print("RESUMEN FINAL EMBEDDINGS GLOBALES")
print("=" * 80)

print("global_ref_embeddings_all shape:", global_ref_embeddings_all.shape)
print("global_ref_metadata_all shape:", global_ref_metadata_all.shape)
print("Species con embeddings:", global_ref_metadata_all["species_id"].nunique())
print("OK images:", int(global_ref_metadata_all["ok"].sum()), "/", len(global_ref_metadata_all))
print("Chunks completados ahora:", completed_now)
print("Chunks omitidos:", skipped)
print("Tiempo total esta corrida:", round(elapsed_total / 60, 2), "min")

print("\nNormas embeddings globales:")
global_norms = np.linalg.norm(global_ref_embeddings_all, axis=1)
print("min:", float(global_norms.min()))
print("mean:", float(global_norms.mean()))
print("max:", float(global_norms.max()))

print("\nDistribución por split:")
display(global_ref_metadata_all["split_source"].value_counts().reset_index(name="n_refs"))

print("\nDistribución por órgano:")
display(global_ref_metadata_all["organ"].value_counts().reset_index(name="n_refs"))

print("\nArchivos guardados:")
print("Embeddings:", all_emb_path)
print("Metadata:", all_meta_path)
print("Config:", config_path)

display(global_ref_metadata_all.head())


del all_emb_parts, all_meta_parts
gc.collect()
torch.cuda.empty_cache()

print("\n Embeddings globales listos.")

# **CELL 19 — Build global prototypes by species**

In [ ]:
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"

GLOBAL_PROTO_ROOT = STORAGE_ROOT / "runs" / "global_prototype_recovery"
GLOBAL_EMB_DIR = GLOBAL_PROTO_ROOT / "global_reference_embeddings"
GLOBAL_PROTO_DIR = GLOBAL_PROTO_ROOT / "global_prototypes"
GLOBAL_PROTO_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_EMB_PATH = GLOBAL_EMB_DIR / "global_reference_embeddings_all.npy"
GLOBAL_META_PATH = GLOBAL_EMB_DIR / "global_reference_embeddings_metadata.csv"

GLOBAL_REF_BANK_PATH = GLOBAL_PROTO_ROOT / "global_reference_bank.pkl.gz"
SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"

print("=" * 80)
print("CELDA 33 — CONSTRUIR PROTOTIPOS GLOBALES")
print("=" * 80)

paths_to_check = [
    GLOBAL_EMB_PATH,
    GLOBAL_META_PATH,
    GLOBAL_REF_BANK_PATH,
    SPECIES_TABLE_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


print("\n" + "=" * 80)
print("CARGANDO EMBEDDINGS Y METADATA")
print("=" * 80)

global_emb = np.load(GLOBAL_EMB_PATH).astype(np.float32)
global_meta = pd.read_csv(GLOBAL_META_PATH)
global_ref_bank = pd.read_pickle(GLOBAL_REF_BANK_PATH, compression="gzip")
species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")

print("global_emb shape:", global_emb.shape)
print("global_meta shape:", global_meta.shape)
print("global_ref_bank shape:", global_ref_bank.shape)
print("species_table shape:", species_table.shape)

if global_emb.shape[0] != len(global_meta):
    raise RuntimeError(
        f"Desalineación: embeddings {global_emb.shape[0]} vs metadata {len(global_meta)}"
    )


global_meta["global_ref_id"] = global_meta["global_ref_id"].astype(str)
global_meta["species_id"] = global_meta["species_id"].astype(int)
global_meta["organ"] = global_meta["organ"].fillna("unknown").astype(str).str.lower()
global_meta["split_source"] = global_meta["split_source"].astype(str)

global_ref_bank["global_ref_id"] = global_ref_bank["global_ref_id"].astype(str)
global_ref_bank["species_id"] = global_ref_bank["species_id"].astype(int)

species_table["species_id"] = species_table["species_id"].astype(int)


global_emb = global_emb / np.maximum(
    np.linalg.norm(global_emb, axis=1, keepdims=True),
    1e-12
)

print("\nNormas global_emb:")
norms = np.linalg.norm(global_emb, axis=1)
print("min:", float(norms.min()))
print("mean:", float(norms.mean()))
print("max:", float(norms.max()))


bank_cols = [
    "global_ref_id",
    "species",
    "genus",
    "family",
]

bank_cols = [c for c in bank_cols if c in global_ref_bank.columns]

global_meta_full = global_meta.merge(
    global_ref_bank[bank_cols].drop_duplicates("global_ref_id"),
    on="global_ref_id",
    how="left"
)


global_meta_full = global_meta_full.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left",
    suffixes=("", "_table")
)

for col in ["species", "genus", "family"]:
    alt = f"{col}_table"
    if alt in global_meta_full.columns:
        global_meta_full[col] = global_meta_full[col].fillna(global_meta_full[alt])
        global_meta_full = global_meta_full.drop(columns=[alt])

print("\nglobal_meta_full shape:", global_meta_full.shape)
print("Species en metadata:", global_meta_full["species_id"].nunique())

display(global_meta_full["organ"].value_counts().reset_index(name="n_refs"))


def normalized_mean(vectors: np.ndarray):
    proto = vectors.mean(axis=0).astype(np.float32)
    norm = np.linalg.norm(proto)
    if norm <= 1e-12:
        return proto
    return proto / norm


def cosine_stats_to_proto(vectors: np.ndarray, proto: np.ndarray):
    sims = vectors @ proto.astype(np.float32)

    return {
        "sim_mean": float(np.mean(sims)),
        "sim_std": float(np.std(sims)),
        "sim_min": float(np.min(sims)),
        "sim_p10": float(np.percentile(sims, 10)),
        "sim_p25": float(np.percentile(sims, 25)),
        "sim_median": float(np.percentile(sims, 50)),
        "sim_p75": float(np.percentile(sims, 75)),
        "sim_p90": float(np.percentile(sims, 90)),
        "sim_max": float(np.max(sims)),
    }



print("\n" + "=" * 80)
print("PARTE A — PROTOTIPOS GLOBALES POR ESPECIE")
print("=" * 80)

all_species_ids = sorted(species_table["species_id"].astype(int).unique().tolist())

species_proto_vectors = []
species_proto_rows = []
quality_rows = []

for sp in tqdm(all_species_ids, desc="build_global_species_prototypes"):
    idx = global_meta_full.index[global_meta_full["species_id"] == sp].to_numpy()

    if len(idx) == 0:
        raise RuntimeError(f"Species sin embeddings: {sp}")

    vectors = global_emb[idx]
    proto = normalized_mean(vectors)
    stats = cosine_stats_to_proto(vectors, proto)

    g = global_meta_full.loc[idx].copy()

    n_refs = len(g)
    n_train = int((g["split_source"] == "train").sum())
    n_val = int((g["split_source"] == "val").sum())
    n_organs = int(g["organ"].nunique())
    dominant_organ = g["organ"].value_counts().index[0]
    organ_counts = g["organ"].value_counts().to_dict()

    species_name = g["species"].dropna().iloc[0] if g["species"].notna().any() else None
    genus = g["genus"].dropna().iloc[0] if g["genus"].notna().any() else None
    family = g["family"].dropna().iloc[0] if g["family"].notna().any() else None

    proto_index = len(species_proto_vectors)
    species_proto_vectors.append(proto)

    row = {
        "prototype_index": proto_index,
        "species_id": int(sp),
        "species": species_name,
        "genus": genus,
        "family": family,
        "n_refs": int(n_refs),
        "n_train_refs": n_train,
        "n_val_refs": n_val,
        "n_organs": n_organs,
        "dominant_organ": dominant_organ,
        "organ_counts_json": json.dumps(organ_counts),
        **stats,
    }

    species_proto_rows.append(row)

    quality_rows.append({
        "species_id": int(sp),
        "n_refs": int(n_refs),
        "n_organs": n_organs,
        "dominant_organ": dominant_organ,
        **stats,
    })

global_species_prototypes = np.stack(species_proto_vectors, axis=0).astype(np.float32)
global_species_proto_meta = pd.DataFrame(species_proto_rows)
global_species_quality = pd.DataFrame(quality_rows)

global_species_prototypes = global_species_prototypes / np.maximum(
    np.linalg.norm(global_species_prototypes, axis=1, keepdims=True),
    1e-12
)

print("global_species_prototypes shape:", global_species_prototypes.shape)
print("global_species_proto_meta shape:", global_species_proto_meta.shape)

print("\nResumen calidad prototipos globales por especie:")
display(
    global_species_proto_meta[
        [
            "n_refs",
            "n_organs",
            "sim_mean",
            "sim_std",
            "sim_p10",
            "sim_median",
            "sim_p90",
        ]
    ].describe()
)

print("\nEspecies con prototipos menos compactos:")
display(
    global_species_proto_meta
    .sort_values("sim_mean", ascending=True)
    [
        [
            "species_id",
            "species",
            "n_refs",
            "n_organs",
            "dominant_organ",
            "sim_mean",
            "sim_p10",
            "sim_median",
        ]
    ]
    .head(20)
)


print("\n" + "=" * 80)
print("PARTE B — PROTOTIPOS GLOBALES POR ESPECIE + ÓRGANO")
print("=" * 80)

species_organ_proto_vectors = []
species_organ_proto_rows = []

for (sp, organ), g in tqdm(
    global_meta_full.groupby(["species_id", "organ"]),
    desc="build_global_species_organ_prototypes"
):
    idx = g.index.to_numpy()
    vectors = global_emb[idx]

    proto = normalized_mean(vectors)
    stats = cosine_stats_to_proto(vectors, proto)

    n_refs = len(g)
    n_train = int((g["split_source"] == "train").sum())
    n_val = int((g["split_source"] == "val").sum())

    species_name = g["species"].dropna().iloc[0] if g["species"].notna().any() else None
    genus = g["genus"].dropna().iloc[0] if g["genus"].notna().any() else None
    family = g["family"].dropna().iloc[0] if g["family"].notna().any() else None

    proto_index = len(species_organ_proto_vectors)
    species_organ_proto_vectors.append(proto)

    species_organ_proto_rows.append({
        "prototype_index": proto_index,
        "species_id": int(sp),
        "organ": str(organ),
        "species": species_name,
        "genus": genus,
        "family": family,
        "n_refs": int(n_refs),
        "n_train_refs": n_train,
        "n_val_refs": n_val,
        **stats,
    })

global_species_organ_prototypes = np.stack(species_organ_proto_vectors, axis=0).astype(np.float32)
global_species_organ_proto_meta = pd.DataFrame(species_organ_proto_rows)

global_species_organ_prototypes = global_species_organ_prototypes / np.maximum(
    np.linalg.norm(global_species_organ_prototypes, axis=1, keepdims=True),
    1e-12
)

print("global_species_organ_prototypes shape:", global_species_organ_prototypes.shape)
print("global_species_organ_proto_meta shape:", global_species_organ_proto_meta.shape)
print("Species cubiertas por organ-prototypes:", global_species_organ_proto_meta["species_id"].nunique())

print("\nÓrganos en prototipos especie+órgano:")
display(global_species_organ_proto_meta["organ"].value_counts().reset_index(name="n_prototypes"))

print("\nResumen calidad prototipos especie+órgano:")
display(
    global_species_organ_proto_meta[
        [
            "n_refs",
            "sim_mean",
            "sim_std",
            "sim_p10",
            "sim_median",
            "sim_p90",
        ]
    ].describe()
)


print("\n" + "=" * 80)
print("GUARDANDO PROTOTIPOS GLOBALES")
print("=" * 80)

species_proto_path = GLOBAL_PROTO_DIR / "global_species_prototypes.npy"
species_proto_meta_path = GLOBAL_PROTO_DIR / "global_species_prototypes_metadata.csv"

species_organ_proto_path = GLOBAL_PROTO_DIR / "global_species_organ_prototypes.npy"
species_organ_proto_meta_path = GLOBAL_PROTO_DIR / "global_species_organ_prototypes_metadata.csv"

species_quality_path = GLOBAL_PROTO_DIR / "global_species_prototype_quality.csv"
config_path = GLOBAL_PROTO_DIR / "global_prototype_config.json"

np.save(species_proto_path, global_species_prototypes)
global_species_proto_meta.to_csv(species_proto_meta_path, index=False)

np.save(species_organ_proto_path, global_species_organ_prototypes)
global_species_organ_proto_meta.to_csv(species_organ_proto_meta_path, index=False)

global_species_quality.to_csv(species_quality_path, index=False)

config = {
    "source_global_embeddings": str(GLOBAL_EMB_PATH),
    "source_global_metadata": str(GLOBAL_META_PATH),
    "global_reference_bank": str(GLOBAL_REF_BANK_PATH),
    "n_reference_embeddings": int(global_emb.shape[0]),
    "embedding_dim": int(global_emb.shape[1]),
    "n_species_total": int(len(all_species_ids)),
    "n_species_prototypes": int(global_species_prototypes.shape[0]),
    "n_species_organ_prototypes": int(global_species_organ_prototypes.shape[0]),
    "species_prototypes_path": str(species_proto_path),
    "species_prototypes_metadata_path": str(species_proto_meta_path),
    "species_organ_prototypes_path": str(species_organ_proto_path),
    "species_organ_prototypes_metadata_path": str(species_organ_proto_meta_path),
    "species_quality_path": str(species_quality_path),
    "notes": (
        "Global prototypes built for all 7806 species. "
        "All vectors are L2-normalized means of L2-normalized DINOv2 embeddings."
    ),
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Species prototypes:", species_proto_path)
print("Species metadata:", species_proto_meta_path)
print("Species-organ prototypes:", species_organ_proto_path)
print("Species-organ metadata:", species_organ_proto_meta_path)
print("Quality:", species_quality_path)
print("Config:", config_path)


print("\n" + "=" * 80)
print("VALIDACIÓN FINAL")
print("=" * 80)

sp_norms = np.linalg.norm(global_species_prototypes, axis=1)
spo_norms = np.linalg.norm(global_species_organ_prototypes, axis=1)

print("Global species prototypes norms:")
print("min:", float(sp_norms.min()))
print("mean:", float(sp_norms.mean()))
print("max:", float(sp_norms.max()))

print("\nGlobal species-organ prototypes norms:")
print("min:", float(spo_norms.min()))
print("mean:", float(spo_norms.mean()))
print("max:", float(spo_norms.max()))

missing_species = set(all_species_ids) - set(global_species_proto_meta["species_id"].astype(int))
print("\nSpecies esperadas:", len(all_species_ids))
print("Species con prototipo:", global_species_proto_meta["species_id"].nunique())
print("Species sin prototipo:", len(missing_species))

if len(missing_species) > 0:
    print("Ejemplos sin prototipo:", list(missing_species)[:20])
    raise RuntimeError("Hay especies sin prototipo global.")

display(global_species_proto_meta.head())
display(global_species_organ_proto_meta.head())

# Limpieza ligera
del global_emb, global_meta, global_meta_full, global_ref_bank
gc.collect()

print("\n Prototipos globales listos.")

# **CELL 20 — Compare test against the 7806 global prototypes**

In [ ]:
import ast
import re
import json
import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

GLOBAL_PROTO_ROOT = STORAGE_ROOT / "runs" / "global_prototype_recovery"
GLOBAL_PROTO_DIR = GLOBAL_PROTO_ROOT / "global_prototypes"

TEST_EMB_DIR = STORAGE_ROOT / "runs" / "prototype_rerank_B" / "test_embeddings"

SIM_DIR = GLOBAL_PROTO_ROOT / "global_test_similarity"
VIEW_TOPK_DIR = SIM_DIR / "view_topk_chunks"

SIM_DIR.mkdir(parents=True, exist_ok=True)
VIEW_TOPK_DIR.mkdir(parents=True, exist_ok=True)

TEST_EMB_PATH = TEST_EMB_DIR / "test_embeddings_selected_all.npy"
TEST_META_PATH = TEST_EMB_DIR / "test_embeddings_selected_metadata.csv"

GLOBAL_SPECIES_PROTO_PATH = GLOBAL_PROTO_DIR / "global_species_prototypes.npy"
GLOBAL_SPECIES_PROTO_META_PATH = GLOBAL_PROTO_DIR / "global_species_prototypes_metadata.csv"

TEST_PATH = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"

V4_SUB_PATH = SUBMISSIONS_DIR / "submission_PROTO_RERANK_MILD_v4_near_max1.csv"
B_SUB_PATH = SUBMISSIONS_DIR / "submission_B_precision_v1.csv"

print("=" * 80)
print("CELDA 34 — TEST vs PROTOTIPOS GLOBALES 7806")
print("=" * 80)

paths_to_check = [
    TEST_EMB_PATH,
    TEST_META_PATH,
    GLOBAL_SPECIES_PROTO_PATH,
    GLOBAL_SPECIES_PROTO_META_PATH,
    TEST_PATH,
    SPECIES_TABLE_PATH,
    V4_SUB_PATH,
    B_SUB_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


TOPK_SPECIES_PER_VIEW = 80      
VIEW_CHUNK_SIZE = 2048         
TOP_CANDIDATES_PER_Q = 300      
USE_FP16 = True                 

print("\nConfiguración:")
print("TOPK_SPECIES_PER_VIEW:", TOPK_SPECIES_PER_VIEW)
print("VIEW_CHUNK_SIZE:", VIEW_CHUNK_SIZE)
print("TOP_CANDIDATES_PER_Q:", TOP_CANDIDATES_PER_Q)
print("USE_FP16:", USE_FP16)


def parse_species_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def topk_mean_series(x, k):
    arr = np.asarray(x, dtype=np.float32)
    if len(arr) == 0:
        return np.nan
    kk = min(k, len(arr))
    return float(np.partition(arr, -kk)[-kk:].mean())


def minmax_norm_group(x):
    arr = np.asarray(x, dtype=np.float32)
    mn = float(np.min(arr))
    mx = float(np.max(arr))
    if mx - mn < 1e-12:
        return np.ones_like(arr, dtype=np.float32) * 0.5
    return (arr - mn) / (mx - mn)


def compute_exact_visual_for_species(E, P_species):
    """
    E: n_views x dim
    P_species: n_species x dim
    Retorna métricas de similitud para cada especie.
    """
    S = E @ P_species.T  # n_views x n_species

    n_views = S.shape[0]
    k3 = min(3, n_views)
    k5 = min(5, n_views)

    sim_max = S.max(axis=0)
    sim_mean = S.mean(axis=0)

    sim_top3 = np.partition(S, -k3, axis=0)[-k3:, :].mean(axis=0)
    sim_top5 = np.partition(S, -k5, axis=0)[-k5:, :].mean(axis=0)

    return sim_max, sim_mean, sim_top3, sim_top5


def visual_score_formula(sim_top5, sim_max, sim_mean, sim_green_weighted, rank_bonus):
    return (
        0.40 * sim_top5
        + 0.25 * sim_max
        + 0.20 * sim_green_weighted
        + 0.10 * sim_mean
        + 0.05 * rank_bonus
    )



print("\n" + "=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

test_emb = np.load(TEST_EMB_PATH).astype(np.float32)
test_meta = pd.read_csv(TEST_META_PATH)

species_prototypes = np.load(GLOBAL_SPECIES_PROTO_PATH).astype(np.float32)
species_proto_meta = pd.read_csv(GLOBAL_SPECIES_PROTO_META_PATH)

species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")
test_df = pd.read_pickle(TEST_PATH, compression="gzip")

v4_sub = pd.read_csv(V4_SUB_PATH)
b_sub = pd.read_csv(B_SUB_PATH)

print("test_emb:", test_emb.shape)
print("test_meta:", test_meta.shape)
print("species_prototypes:", species_prototypes.shape)
print("species_proto_meta:", species_proto_meta.shape)

if test_emb.shape[0] != len(test_meta):
    raise RuntimeError("Desalineación entre test_emb y test_meta.")

if species_prototypes.shape[0] != len(species_proto_meta):
    raise RuntimeError("Desalineación entre species_prototypes y metadata.")

# Tipos
test_meta["quadrat_id"] = test_meta["quadrat_id"].astype(str)
test_meta["view_type"] = test_meta["view_type"].astype(str)
test_meta["green_ratio"] = pd.to_numeric(test_meta["green_ratio"], errors="coerce").fillna(0.0)

species_proto_meta["species_id"] = species_proto_meta["species_id"].astype(int)
species_proto_meta["prototype_index"] = species_proto_meta["prototype_index"].astype(int)

species_table["species_id"] = species_table["species_id"].astype(int)
test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)

v4_sub["quadrat_id"] = v4_sub["quadrat_id"].astype(str)
v4_sub["species_list_v4"] = v4_sub["species_ids"].apply(parse_species_ids)

b_sub["quadrat_id"] = b_sub["quadrat_id"].astype(str)
b_sub["species_list_B"] = b_sub["species_ids"].apply(parse_species_ids)


species_proto_meta = species_proto_meta.sort_values("prototype_index").reset_index(drop=True)
species_prototypes = species_prototypes[species_proto_meta["prototype_index"].values]

species_ids_arr = species_proto_meta["species_id"].astype(int).values
sp_to_proto_row = {int(sp): i for i, sp in enumerate(species_ids_arr)}


test_emb = test_emb / np.maximum(np.linalg.norm(test_emb, axis=1, keepdims=True), 1e-12)
species_prototypes = species_prototypes / np.maximum(np.linalg.norm(species_prototypes, axis=1, keepdims=True), 1e-12)

print("\nQuadrats test embeddings:", test_meta["quadrat_id"].nunique())
print("Species globales:", len(species_ids_arr))


v4_lookup = {
    row.quadrat_id: [int(sp) for sp in row.species_list_v4]
    for row in v4_sub.itertuples(index=False)
}

b_lookup = {
    row.quadrat_id: [int(sp) for sp in row.species_list_B]
    for row in b_sub.itertuples(index=False)
}

test_indices_by_qid = test_meta.groupby("quadrat_id").indices


print("\n" + "=" * 80)
print("PARTE A — TOP-K GLOBAL POR VISTA")
print("=" * 80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type != "cuda":
    raise RuntimeError("CUDA no disponible. Esta celda debe correr en GPU.")

dtype = torch.float16 if USE_FP16 else torch.float32

P_torch = torch.from_numpy(species_prototypes).to(device=device, dtype=dtype).T.contiguous()

n_views = test_emb.shape[0]
n_chunks = int(np.ceil(n_views / VIEW_CHUNK_SIZE))

chunk_plan = []

for chunk_id in range(n_chunks):
    start = chunk_id * VIEW_CHUNK_SIZE
    end = min((chunk_id + 1) * VIEW_CHUNK_SIZE, n_views)

    out_path = VIEW_TOPK_DIR / f"global_view_topk_chunk_{chunk_id:04d}.pkl.gz"

    chunk_plan.append({
        "chunk_id": chunk_id,
        "start": start,
        "end": end,
        "n_views": end - start,
        "out_path": str(out_path),
        "done": out_path.exists(),
    })

chunk_plan_df = pd.DataFrame(chunk_plan)
chunk_plan_path = SIM_DIR / "global_view_topk_chunk_plan.csv"
chunk_plan_df.to_csv(chunk_plan_path, index=False)

print("n_views:", n_views)
print("n_chunks:", n_chunks)
print("chunks existentes:", int(chunk_plan_df["done"].sum()), "/", n_chunks)

display(chunk_plan_df.head())

start_time = time.time()

for item in tqdm(chunk_plan, desc="global_view_topk_chunks"):
    chunk_id = int(item["chunk_id"])
    start = int(item["start"])
    end = int(item["end"])
    out_path = Path(item["out_path"])

    if out_path.exists():
        continue

    E_np = test_emb[start:end]
    meta_chunk = test_meta.iloc[start:end].reset_index(drop=True).copy()

    E_torch = torch.from_numpy(E_np).to(device=device, dtype=dtype)

    with torch.no_grad():
        sims = E_torch @ P_torch
        vals, inds = torch.topk(sims, k=TOPK_SPECIES_PER_VIEW, dim=1)

    vals_np = vals.detach().cpu().numpy().astype(np.float32)
    inds_np = inds.detach().cpu().numpy().astype(np.int32)

    bs = end - start

    qids_rep = np.repeat(meta_chunk["quadrat_id"].astype(str).values, TOPK_SPECIES_PER_VIEW)
    view_type_rep = np.repeat(meta_chunk["view_type"].astype(str).values, TOPK_SPECIES_PER_VIEW)
    green_rep = np.repeat(meta_chunk["green_ratio"].astype(float).values, TOPK_SPECIES_PER_VIEW)

    if "selected_view_global_id" in meta_chunk.columns:
        view_global_rep = np.repeat(meta_chunk["selected_view_global_id"].astype(str).values, TOPK_SPECIES_PER_VIEW)
    else:
        view_global_rep = np.repeat(np.arange(start, end).astype(str), TOPK_SPECIES_PER_VIEW)

    view_id_rep = np.repeat(meta_chunk["view_id"].astype(int).values, TOPK_SPECIES_PER_VIEW)

    rank_tile = np.tile(np.arange(1, TOPK_SPECIES_PER_VIEW + 1, dtype=np.int16), bs)

    proto_idx_flat = inds_np.reshape(-1)
    species_flat = species_ids_arr[proto_idx_flat]
    sim_flat = vals_np.reshape(-1)

    df_chunk = pd.DataFrame({
        "quadrat_id": qids_rep,
        "selected_view_global_id": view_global_rep,
        "view_id": view_id_rep,
        "view_type": view_type_rep,
        "green_ratio": green_rep.astype(np.float32),
        "view_rank": rank_tile,
        "prototype_index": proto_idx_flat.astype(np.int32),
        "species_id": species_flat.astype(np.int32),
        "sim": sim_flat.astype(np.float32),
    })

    df_chunk["green_weight"] = 0.5 + df_chunk["green_ratio"].clip(0, 1)
    df_chunk["sim_x_green_weight"] = df_chunk["sim"] * df_chunk["green_weight"]

    df_chunk.to_pickle(out_path, compression="gzip")

    del E_np, E_torch, sims, vals, inds, vals_np, inds_np, df_chunk
    gc.collect()
    torch.cuda.empty_cache()

elapsed = time.time() - start_time
print("Tiempo top-k global por vista:", round(elapsed / 60, 2), "min")


del P_torch
torch.cuda.empty_cache()
gc.collect()


print("\n" + "=" * 80)
print("PARTE B — AGREGANDO CANDIDATOS VISUALES GLOBALES")
print("=" * 80)

topk_files = sorted(VIEW_TOPK_DIR.glob("global_view_topk_chunk_*.pkl.gz"))
print("TopK chunks encontrados:", len(topk_files), "/", n_chunks)

if len(topk_files) != n_chunks:
    raise RuntimeError("Faltan chunks de top-k global. Reejecuta la celda.")

topk_parts = []

for p in tqdm(topk_files, desc="load_topk_chunks"):
    topk_parts.append(pd.read_pickle(p, compression="gzip"))

view_topk_df = pd.concat(topk_parts, ignore_index=True)

del topk_parts
gc.collect()

print("view_topk_df shape:", view_topk_df.shape)
print("Quadrats:", view_topk_df["quadrat_id"].nunique())
print("Species únicas topk:", view_topk_df["species_id"].nunique())


view_topk_df["hit_full"] = (view_topk_df["view_type"] == "full").astype(np.int8)
view_topk_df["hit_center"] = (view_topk_df["view_type"] == "center").astype(np.int8)
view_topk_df["hit_tile"] = (view_topk_df["view_type"] == "tile").astype(np.int8)

print("\nAgregando groupby quadrat/species...")

agg = (
    view_topk_df
    .groupby(["quadrat_id", "species_id"])
    .agg(
        n_view_hits=("sim", "size"),
        n_unique_views=("selected_view_global_id", "nunique"),
        sim_max=("sim", "max"),
        sim_mean=("sim", "mean"),
        sim_top3=("sim", lambda x: topk_mean_series(x, 3)),
        sim_top5=("sim", lambda x: topk_mean_series(x, 5)),
        sim_green_num=("sim_x_green_weight", "sum"),
        green_weight_sum=("green_weight", "sum"),
        min_view_rank=("view_rank", "min"),
        mean_view_rank=("view_rank", "mean"),
        n_full_hits=("hit_full", "sum"),
        n_center_hits=("hit_center", "sum"),
        n_tile_hits=("hit_tile", "sum"),
        mean_green=("green_ratio", "mean"),
        max_green=("green_ratio", "max"),
    )
    .reset_index()
)

agg["sim_green_weighted"] = (
    agg["sim_green_num"] / agg["green_weight_sum"].replace(0, np.nan)
).fillna(0.0)

agg["rank_bonus"] = 1.0 - (
    (agg["min_view_rank"].astype(float) - 1.0) / max(TOPK_SPECIES_PER_VIEW - 1, 1)
)
agg["rank_bonus"] = agg["rank_bonus"].clip(0, 1)

agg["visual_raw"] = visual_score_formula(
    agg["sim_top5"].values,
    agg["sim_max"].values,
    agg["sim_mean"].values,
    agg["sim_green_weighted"].values,
    agg["rank_bonus"].values,
)

agg["source_global_topk"] = 1

print("agg shape:", agg.shape)
print("Quadrats agg:", agg["quadrat_id"].nunique())
print("Species agg:", agg["species_id"].nunique())


print("\n" + "=" * 80)
print("PARTE C — FORZANDO ESPECIES DE v4 PARA COMPARACIÓN")
print("=" * 80)

existing_pairs = set(zip(agg["quadrat_id"].astype(str), agg["species_id"].astype(int)))

forced_rows = []

for qid, species_list in tqdm(v4_lookup.items(), desc="force_v4_species_visual"):
    missing_species = [
        int(sp) for sp in species_list
        if (qid, int(sp)) not in existing_pairs and int(sp) in sp_to_proto_row
    ]

    if len(missing_species) == 0:
        continue

    idxs = np.asarray(test_indices_by_qid[qid])
    E = test_emb[idxs]

    meta_q = test_meta.iloc[idxs].reset_index(drop=True)
    green = meta_q["green_ratio"].astype(float).values
    weights = 0.5 + np.clip(green, 0, 1)
    weights = weights / max(weights.sum(), 1e-12)

    proto_indices = [sp_to_proto_row[int(sp)] for sp in missing_species]
    P = species_prototypes[proto_indices]

    S = E @ P.T

    sim_max = S.max(axis=0)
    sim_mean = S.mean(axis=0)

    k3 = min(3, S.shape[0])
    k5 = min(5, S.shape[0])

    sim_top3 = np.partition(S, -k3, axis=0)[-k3:, :].mean(axis=0)
    sim_top5 = np.partition(S, -k5, axis=0)[-k5:, :].mean(axis=0)
    sim_green_weighted = weights @ S

    
    rank_bonus = np.zeros(len(missing_species), dtype=np.float32)

    visual_raw = visual_score_formula(
        sim_top5,
        sim_max,
        sim_mean,
        sim_green_weighted,
        rank_bonus
    )

    for j, sp in enumerate(missing_species):
        forced_rows.append({
            "quadrat_id": qid,
            "species_id": int(sp),
            "n_view_hits": int(S.shape[0]),
            "n_unique_views": int(S.shape[0]),
            "sim_max": float(sim_max[j]),
            "sim_mean": float(sim_mean[j]),
            "sim_top3": float(sim_top3[j]),
            "sim_top5": float(sim_top5[j]),
            "sim_green_num": np.nan,
            "green_weight_sum": np.nan,
            "min_view_rank": 999,
            "mean_view_rank": 999,
            "n_full_hits": int((meta_q["view_type"] == "full").sum()),
            "n_center_hits": int((meta_q["view_type"] == "center").sum()),
            "n_tile_hits": int((meta_q["view_type"] == "tile").sum()),
            "mean_green": float(green.mean()),
            "max_green": float(green.max()),
            "sim_green_weighted": float(sim_green_weighted[j]),
            "rank_bonus": 0.0,
            "visual_raw": float(visual_raw[j]),
            "source_global_topk": 0,
        })

if len(forced_rows) > 0:
    forced_df = pd.DataFrame(forced_rows)
    agg = pd.concat([agg, forced_df], ignore_index=True)
    print("Filas v4 forzadas:", len(forced_df))
else:
    print("No fue necesario forzar especies de v4.")


print("\n" + "=" * 80)
print("PARTE D — TOP300 CANDIDATOS GLOBALES POR QUADRAT")
print("=" * 80)

# Flags B/v4
v4_pairs = set()
for qid, xs in v4_lookup.items():
    for sp in xs:
        v4_pairs.add((qid, int(sp)))

b_pairs = set()
for qid, xs in b_lookup.items():
    for sp in xs:
        b_pairs.add((qid, int(sp)))

agg["in_v4"] = [
    int((qid, int(sp)) in v4_pairs)
    for qid, sp in zip(agg["quadrat_id"].astype(str), agg["species_id"].astype(int))
]

agg["in_B"] = [
    int((qid, int(sp)) in b_pairs)
    for qid, sp in zip(agg["quadrat_id"].astype(str), agg["species_id"].astype(int))
]


agg["visual_norm_global"] = 0.0

for qid, idx in tqdm(agg.groupby("quadrat_id").groups.items(), desc="normalize_global_visual"):
    idx = list(idx)
    agg.loc[idx, "visual_norm_global"] = minmax_norm_group(agg.loc[idx, "visual_raw"].values)


agg = agg.sort_values(
    ["quadrat_id", "visual_raw", "sim_top5", "sim_max"],
    ascending=[True, False, False, False]
).reset_index(drop=True)

agg["global_visual_rank"] = agg.groupby("quadrat_id").cumcount() + 1


agg = agg.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left"
)


top300 = (
    agg[
        (agg["global_visual_rank"] <= TOP_CANDIDATES_PER_Q)
        | (agg["in_v4"] == 1)
    ]
    .copy()
)

top300 = top300.sort_values(
    ["quadrat_id", "global_visual_rank"],
    ascending=[True, True]
).reset_index(drop=True)

print("agg all shape:", agg.shape)
print("top300 shape:", top300.shape)
print("Quadrats top300:", top300["quadrat_id"].nunique())
print("Species únicas top300:", top300["species_id"].nunique())

print("\nCandidatos por quadrat:")
display(
    top300
    .groupby("quadrat_id")
    .size()
    .reset_index(name="n_candidates")
    .describe()
)

print("\nTop global visual rank de especies v4:")
display(
    top300[top300["in_v4"] == 1]["global_visual_rank"].describe()
)

print("\nTop 20 global visual candidates:")
display(
    top300[
        [
            "quadrat_id",
            "global_visual_rank",
            "species_id",
            "species",
            "visual_raw",
            "visual_norm_global",
            "sim_top5",
            "sim_max",
            "n_view_hits",
            "min_view_rank",
            "in_v4",
            "in_B",
            "source_global_topk",
        ]
    ].head(20)
)


all_path = SIM_DIR / "global_visual_candidates_all.pkl.gz"
top300_csv_path = SIM_DIR / "global_visual_candidates_top300.csv"
top300_pkl_path = SIM_DIR / "global_visual_candidates_top300.pkl.gz"
summary_path = SIM_DIR / "global_visual_similarity_summary.csv"
config_path = SIM_DIR / "global_similarity_config.json"

agg.to_pickle(all_path, compression="gzip")
top300.to_csv(top300_csv_path, index=False)
top300.to_pickle(top300_pkl_path, compression="gzip")

summary_rows = []

summary_rows.append({
    "n_test_views": int(test_emb.shape[0]),
    "n_quadrats": int(test_meta["quadrat_id"].nunique()),
    "n_species_prototypes": int(species_prototypes.shape[0]),
    "topk_species_per_view": int(TOPK_SPECIES_PER_VIEW),
    "n_rows_view_topk": int(len(view_topk_df)),
    "n_rows_agg_all": int(len(agg)),
    "n_rows_top300": int(len(top300)),
    "n_unique_species_top300": int(top300["species_id"].nunique()),
    "n_v4_forced_rows": int(len(forced_rows)),
})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)

config = {
    "strategy": "global_species_prototype_similarity",
    "topk_species_per_view": TOPK_SPECIES_PER_VIEW,
    "view_chunk_size": VIEW_CHUNK_SIZE,
    "top_candidates_per_quadrat": TOP_CANDIDATES_PER_Q,
    "use_fp16": USE_FP16,
    "inputs": {
        "test_embeddings": str(TEST_EMB_PATH),
        "test_metadata": str(TEST_META_PATH),
        "global_species_prototypes": str(GLOBAL_SPECIES_PROTO_PATH),
        "global_species_prototype_metadata": str(GLOBAL_SPECIES_PROTO_META_PATH),
        "v4_submission": str(V4_SUB_PATH),
        "b_submission": str(B_SUB_PATH),
    },
    "outputs": {
        "view_topk_dir": str(VIEW_TOPK_DIR),
        "all_candidates": str(all_path),
        "top300_csv": str(top300_csv_path),
        "top300_pkl": str(top300_pkl_path),
        "summary": str(summary_path),
    },
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("\n" + "=" * 80)
print("ARCHIVOS GUARDADOS")
print("=" * 80)
print("All candidates:", all_path)
print("Top300 CSV:", top300_csv_path)
print("Top300 PKL:", top300_pkl_path)
print("Summary:", summary_path)
print("Config:", config_path)


del view_topk_df, agg
gc.collect()
torch.cuda.empty_cache()

print("\nCandidatos visuales globales listos.")

# **CELL 21 — Create ecological consensus by site-date**

In [ ]:
import ast
import re
import json
import gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

BASE_RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"
GLOBAL_PROTO_ROOT = STORAGE_ROOT / "runs" / "global_prototype_recovery"
GLOBAL_SIM_DIR = GLOBAL_PROTO_ROOT / "global_test_similarity"

ECO_DIR = STORAGE_ROOT / "runs" / "eco_consensus"
ECO_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"


CHAMPION_SUB_PATH = SUBMISSIONS_DIR / "submission_GLOBAL_PROTO_ULTRA_TARGET20_v1.csv"


V4_SUB_PATH = SUBMISSIONS_DIR / "submission_PROTO_RERANK_MILD_v4_near_max1.csv"
B_SUB_PATH = SUBMISSIONS_DIR / "submission_B_precision_v1.csv"

BASE_AGG_PATH = BASE_RUN_DIR / "aggregation" / "quadrat_species_scores.pkl.gz"
GLOBAL_TOP300_PATH = GLOBAL_SIM_DIR / "global_visual_candidates_top300.pkl.gz"

print("=" * 80)
print("CELDA 37 — CREAR CONSENSO ECOLÓGICO POR SITIO-FECHA")
print("=" * 80)

paths_to_check = [
    TEST_PATH,
    SPECIES_TABLE_PATH,
    CHAMPION_SUB_PATH,
    V4_SUB_PATH,
    B_SUB_PATH,
    BASE_AGG_PATH,
    GLOBAL_TOP300_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


KNOWN_REAL_SCORES = {
    "B_precision_v1": 0.31540,
    "PROTO_RERANK_MILD_v4_near_max1": 0.31649,
    "GLOBAL_PROTO_ULTRA_TARGET20_v1": 0.31660,
}

# Top-N que se usa de cada fuente para construir el consenso.
BASE_TOP_N_PER_Q = 80
GLOBAL_VISUAL_TOP_N_PER_Q = 80

# Para guardar consenso por grupo.
TOP_SPECIES_PER_GROUP_TO_SAVE = 120


SMALL_GROUP_MIN_SIZE = 2
LARGE_GROUP_SIZE = 25

print("\nConfiguración:")
print("BASE_TOP_N_PER_Q:", BASE_TOP_N_PER_Q)
print("GLOBAL_VISUAL_TOP_N_PER_Q:", GLOBAL_VISUAL_TOP_N_PER_Q)
print("TOP_SPECIES_PER_GROUP_TO_SAVE:", TOP_SPECIES_PER_GROUP_TO_SAVE)
print("SMALL_GROUP_MIN_SIZE:", SMALL_GROUP_MIN_SIZE)
print("LARGE_GROUP_SIZE:", LARGE_GROUP_SIZE)
print("KNOWN_REAL_SCORES:")
print(json.dumps(KNOWN_REAL_SCORES, indent=2))


def parse_species_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def species_list_to_str(xs):
    return "[" + ", ".join(map(str, [int(x) for x in xs])) + "]"


def extract_date_from_qid(qid):
    """
    Extrae fecha tipo YYYYMMDD al final del quadrat_id.
    Ejemplos:
      CBN-PdlC-E3-20130723 -> 20130723
      LISAH-BOU-0-157-20240502 -> 20240502
    """
    qid = str(qid)
    m = re.search(r"(\d{8})$", qid)
    if m:
        return m.group(1)
    return "unknown_date"


def remove_final_date(qid):
    qid = str(qid)
    return re.sub(r"-?\d{8}$", "", qid).strip("-")


def get_tokens_no_date(qid):
    prefix = remove_final_date(qid)
    tokens = [t for t in prefix.split("-") if t != ""]
    return tokens


def site_l2(qid):
    tokens = get_tokens_no_date(qid)
    if len(tokens) >= 2:
        return "-".join(tokens[:2])
    if len(tokens) == 1:
        return tokens[0]
    return "unknown_site"


def site_l3(qid):
    tokens = get_tokens_no_date(qid)
    if len(tokens) >= 3:
        return "-".join(tokens[:3])
    if len(tokens) >= 2:
        return "-".join(tokens[:2])
    if len(tokens) == 1:
        return tokens[0]
    return "unknown_site"


def prefix_drop_last_token(qid):
    """
    Intenta quitar el identificador local del quadrat antes de la fecha.
    Ejemplos:
      CBN-PdlC-E3-20130723 -> CBN-PdlC
      CBN-Pla-A3-20160728 -> CBN-Pla
      LISAH-BOU-0-157-20240502 -> LISAH-BOU-0
      2024-CEV3-20240602 -> 2024
    """
    tokens = get_tokens_no_date(qid)
    if len(tokens) >= 2:
        return "-".join(tokens[:-1])
    if len(tokens) == 1:
        return tokens[0]
    return "unknown_site"


def eco_keys_from_qid(qid):
    date = extract_date_from_qid(qid)
    l2 = site_l2(qid)
    l3 = site_l3(qid)
    drop1 = prefix_drop_last_token(qid)

    return {
        "qid_date": date,
        "site_l2": l2,
        "site_l3": l3,
        "site_drop1": drop1,
        "eco_key_l2_date": f"{l2}__{date}",
        "eco_key_l3_date": f"{l3}__{date}",
        "eco_key_drop1_date": f"{drop1}__{date}",
        "eco_key_l2_all_dates": f"{l2}__ALLDATES",
        "eco_key_date_only": f"DATE__{date}",
    }


def minmax_norm_group(values):
    arr = np.asarray(values, dtype=np.float32)
    mn = float(np.nanmin(arr))
    mx = float(np.nanmax(arr))
    if mx - mn < 1e-12:
        return np.ones_like(arr, dtype=np.float32) * 0.5
    return (arr - mn) / (mx - mn)


def reciprocal_rank(rank):
    rank = float(rank)
    if rank <= 0 or not np.isfinite(rank):
        return 0.0
    return 1.0 / rank


def safe_mean(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(x.mean())


def safe_max(x):
    x = pd.Series(x).dropna()
    if len(x) == 0:
        return np.nan
    return float(x.max())



print("\n" + "=" * 80)
print("CARGANDO TEST Y SPECIES TABLE")
print("=" * 80)

test_df = pd.read_pickle(TEST_PATH, compression="gzip")
test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)

species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")
species_table["species_id"] = species_table["species_id"].astype(int)

VALID_SPECIES = set(species_table["species_id"].astype(int).tolist())

print("test_df shape:", test_df.shape)
print("Quadrats:", test_df["quadrat_id"].nunique())
print("species_table shape:", species_table.shape)


# Crear llaves ecológicas

print("\n" + "=" * 80)
print("CREANDO LLAVES ECOLÓGICAS")
print("=" * 80)

key_rows = []

for qid in test_df["quadrat_id"].astype(str).tolist():
    row = {"quadrat_id": qid}
    row.update(eco_keys_from_qid(qid))
    key_rows.append(row)

eco_groups = pd.DataFrame(key_rows)

# Calcular tamaños por cada llave candidata
for key_col in [
    "eco_key_drop1_date",
    "eco_key_l3_date",
    "eco_key_l2_date",
    "eco_key_l2_all_dates",
    "eco_key_date_only",
]:
    sizes = eco_groups.groupby(key_col)["quadrat_id"].transform("nunique")
    eco_groups[f"{key_col}_size"] = sizes


def choose_primary_group(row):
    if row["eco_key_drop1_date_size"] >= SMALL_GROUP_MIN_SIZE:
        return row["eco_key_drop1_date"], "drop1_date"
    if row["eco_key_l2_date_size"] >= SMALL_GROUP_MIN_SIZE:
        return row["eco_key_l2_date"], "l2_date"
    if row["eco_key_l2_all_dates_size"] >= SMALL_GROUP_MIN_SIZE:
        return row["eco_key_l2_all_dates"], "l2_all_dates"
    return f"SELF__{row['quadrat_id']}", "self"

primary = eco_groups.apply(choose_primary_group, axis=1)
eco_groups["eco_group_id"] = [x[0] for x in primary]
eco_groups["eco_group_strategy"] = [x[1] for x in primary]

eco_groups["eco_group_size"] = eco_groups.groupby("eco_group_id")["quadrat_id"].transform("nunique")
eco_groups["is_singleton_group"] = eco_groups["eco_group_size"] == 1
eco_groups["is_large_group"] = eco_groups["eco_group_size"] >= LARGE_GROUP_SIZE

print("eco_groups shape:", eco_groups.shape)
print("Grupos ecológicos:", eco_groups["eco_group_id"].nunique())

print("\nDistribución de estrategia de grupo:")
display(eco_groups["eco_group_strategy"].value_counts().reset_index(name="n_quadrats"))

print("\nDistribución tamaño de grupo:")
display(eco_groups[["eco_group_id", "eco_group_size"]].drop_duplicates()["eco_group_size"].describe())

print("\nTop grupos más grandes:")
display(
    eco_groups[["eco_group_id", "eco_group_strategy", "eco_group_size"]]
    .drop_duplicates()
    .sort_values("eco_group_size", ascending=False)
    .head(30)
)


print("\n" + "=" * 80)
print("CARGANDO SUBMISSIONS BASE")
print("=" * 80)

champion_sub = pd.read_csv(CHAMPION_SUB_PATH)
v4_sub = pd.read_csv(V4_SUB_PATH)
b_sub = pd.read_csv(B_SUB_PATH)

for df, list_col in [
    (champion_sub, "species_list_champion"),
    (v4_sub, "species_list_v4"),
    (b_sub, "species_list_B"),
]:
    df["quadrat_id"] = df["quadrat_id"].astype(str)
    df[list_col] = df["species_ids"].apply(parse_species_ids)

champion_sub["K_champion"] = champion_sub["species_list_champion"].apply(len)
v4_sub["K_v4"] = v4_sub["species_list_v4"].apply(len)
b_sub["K_B"] = b_sub["species_list_B"].apply(len)

print("Champion:", champion_sub.shape)
print("v4:", v4_sub.shape)
print("B:", b_sub.shape)

print("\nK champion:")
display(champion_sub["K_champion"].value_counts().sort_index().reset_index(name="n_images"))


print("\n" + "=" * 80)
print("CONSTRUYENDO FORMATO LONG DE PREDICCIONES")
print("=" * 80)

long_parts = []

def submission_to_long(df, species_col, source_name, source_weight):
    rows = []
    for row in df.itertuples(index=False):
        qid = row.quadrat_id
        species_list = getattr(row, species_col)
        K = len(species_list)
        for rank, sp in enumerate(species_list, start=1):
            rows.append({
                "quadrat_id": qid,
                "species_id": int(sp),
                "source": source_name,
                "source_weight": float(source_weight),
                "pred_rank": int(rank),
                "pred_K": int(K),
                "rr": reciprocal_rank(rank),
                "is_selected_source": 1,
            })
    return pd.DataFrame(rows)

long_parts.append(submission_to_long(champion_sub, "species_list_champion", "champion_global_ultra", 1.00))
long_parts.append(submission_to_long(v4_sub, "species_list_v4", "v4_proto", 0.92))
long_parts.append(submission_to_long(b_sub, "species_list_B", "B_precision", 0.82))

sub_long = pd.concat(long_parts, ignore_index=True)

# Agregar grupo ecológico
sub_long = sub_long.merge(
    eco_groups[["quadrat_id", "eco_group_id", "eco_group_strategy", "eco_group_size"]],
    on="quadrat_id",
    how="left"
)

print("sub_long shape:", sub_long.shape)
print("Sources:")
display(sub_long["source"].value_counts().reset_index(name="n_rows"))


print("\n" + "=" * 80)
print("CARGANDO BASE_AGG TOP-N")
print("=" * 80)

base_agg = pd.read_pickle(BASE_AGG_PATH, compression="gzip")

needed_base_cols = [
    "quadrat_id",
    "species_id",
    "final_score",
    "max_score",
    "n_views_species",
    "freq_score",
]

missing = [c for c in needed_base_cols if c not in base_agg.columns]
if missing:
    raise RuntimeError(f"Faltan columnas en base_agg: {missing}")

base_agg = base_agg[needed_base_cols].copy()
base_agg["quadrat_id"] = base_agg["quadrat_id"].astype(str)
base_agg["species_id"] = base_agg["species_id"].astype(int)

base_agg = base_agg.sort_values(
    ["quadrat_id", "final_score"],
    ascending=[True, False]
).reset_index(drop=True)

base_agg["base_rank"] = base_agg.groupby("quadrat_id").cumcount() + 1

base_top1 = (
    base_agg
    .groupby("quadrat_id")["final_score"]
    .max()
    .reset_index(name="base_top1_score")
)

base_agg = base_agg.merge(base_top1, on="quadrat_id", how="left")
base_agg["base_norm_top1"] = (
    base_agg["final_score"] / base_agg["base_top1_score"].replace(0, np.nan)
).fillna(0.0)

base_top = base_agg[base_agg["base_rank"] <= BASE_TOP_N_PER_Q].copy()

base_top = base_top.rename(columns={
    "final_score": "base_score",
    "max_score": "base_max_score",
    "n_views_species": "base_n_views_species",
    "freq_score": "base_freq_score",
})

base_top = base_top.merge(
    eco_groups[["quadrat_id", "eco_group_id", "eco_group_strategy", "eco_group_size"]],
    on="quadrat_id",
    how="left"
)

print("base_top shape:", base_top.shape)
print("Quadrats:", base_top["quadrat_id"].nunique())
print("Species:", base_top["species_id"].nunique())


print("\n" + "=" * 80)
print("CARGANDO GLOBAL VISUAL TOP-N")
print("=" * 80)

global_top300 = pd.read_pickle(GLOBAL_TOP300_PATH, compression="gzip")

global_top300["quadrat_id"] = global_top300["quadrat_id"].astype(str)
global_top300["species_id"] = global_top300["species_id"].astype(int)

global_top = global_top300[
    global_top300["global_visual_rank"] <= GLOBAL_VISUAL_TOP_N_PER_Q
].copy()

global_top = global_top.merge(
    eco_groups[["quadrat_id", "eco_group_id", "eco_group_strategy", "eco_group_size"]],
    on="quadrat_id",
    how="left",
    suffixes=("", "_eco")
)

print("global_top shape:", global_top.shape)
print("Quadrats:", global_top["quadrat_id"].nunique())
print("Species:", global_top["species_id"].nunique())


del global_top300
gc.collect()


print("\n" + "=" * 80)
print("CONSTRUYENDO CONSENSO POR GRUPO/ESPECIE")
print("=" * 80)


sub_group = (
    sub_long
    .groupby(["eco_group_id", "species_id"])
    .agg(
        n_pred_rows=("species_id", "size"),
        n_quadrats_pred=("quadrat_id", "nunique"),
        sources_present=("source", lambda x: ",".join(sorted(set(x)))),
        source_weight_sum=("source_weight", "sum"),
        rr_sum=("rr", "sum"),
        rr_mean=("rr", "mean"),
        best_pred_rank=("pred_rank", "min"),
        mean_pred_rank=("pred_rank", "mean"),
    )
    .reset_index()
)


base_group = (
    base_top
    .groupby(["eco_group_id", "species_id"])
    .agg(
        n_base_rows=("species_id", "size"),
        n_quadrats_base=("quadrat_id", "nunique"),
        base_score_mean=("base_score", "mean"),
        base_score_max=("base_score", "max"),
        base_norm_mean=("base_norm_top1", "mean"),
        base_norm_max=("base_norm_top1", "max"),
        base_best_rank=("base_rank", "min"),
        base_mean_rank=("base_rank", "mean"),
        base_freq_mean=("base_freq_score", "mean"),
        base_n_views_mean=("base_n_views_species", "mean"),
    )
    .reset_index()
)


global_group = (
    global_top
    .groupby(["eco_group_id", "species_id"])
    .agg(
        n_global_rows=("species_id", "size"),
        n_quadrats_global=("quadrat_id", "nunique"),
        global_visual_rank_best=("global_visual_rank", "min"),
        global_visual_rank_mean=("global_visual_rank", "mean"),
        visual_raw_mean=("visual_raw", "mean"),
        visual_raw_max=("visual_raw", "max"),
        visual_norm_mean=("visual_norm_global", "mean"),
        visual_norm_max=("visual_norm_global", "max"),
        sim_top5_mean=("sim_top5", "mean"),
        sim_max_max=("sim_max", "max"),
        n_unique_views_mean=("n_unique_views", "mean"),
        n_tile_hits_mean=("n_tile_hits", "mean"),
    )
    .reset_index()
)

# Merge
consensus = sub_group.merge(
    base_group,
    on=["eco_group_id", "species_id"],
    how="outer"
).merge(
    global_group,
    on=["eco_group_id", "species_id"],
    how="outer"
)

# Fill
fill_zero_cols = [
    "n_pred_rows", "n_quadrats_pred", "source_weight_sum", "rr_sum",
    "n_base_rows", "n_quadrats_base",
    "n_global_rows", "n_quadrats_global",
]

for c in fill_zero_cols:
    if c in consensus.columns:
        consensus[c] = consensus[c].fillna(0)

consensus["sources_present"] = consensus["sources_present"].fillna("")


group_size_df = (
    eco_groups[["eco_group_id", "eco_group_strategy", "eco_group_size"]]
    .drop_duplicates()
)

consensus = consensus.merge(
    group_size_df,
    on="eco_group_id",
    how="left"
)


consensus["pred_group_freq"] = (
    consensus["n_quadrats_pred"] / consensus["eco_group_size"].replace(0, np.nan)
).fillna(0.0)

consensus["base_group_freq"] = (
    consensus["n_quadrats_base"] / consensus["eco_group_size"].replace(0, np.nan)
).fillna(0.0)

consensus["global_group_freq"] = (
    consensus["n_quadrats_global"] / consensus["eco_group_size"].replace(0, np.nan)
).fillna(0.0)


for col, out_col in [
    ("source_weight_sum", "source_weight_norm_group"),
    ("rr_sum", "rr_sum_norm_group"),
    ("base_score_max", "base_score_norm_group"),
    ("base_norm_max", "base_norm_norm_group"),
    ("visual_raw_max", "visual_raw_norm_group"),
    ("visual_norm_max", "visual_norm_norm_group"),
]:
    consensus[out_col] = 0.0
    for gid, idx in consensus.groupby("eco_group_id").groups.items():
        idx = list(idx)
        values = consensus.loc[idx, col].fillna(0.0).values
        consensus.loc[idx, out_col] = minmax_norm_group(values)

# Rank scores
consensus["base_rank_score"] = (
    1.0 - ((consensus["base_best_rank"].fillna(999) - 1) / max(BASE_TOP_N_PER_Q - 1, 1))
).clip(0, 1)

consensus["global_rank_score"] = (
    1.0 - ((consensus["global_visual_rank_best"].fillna(999) - 1) / max(GLOBAL_VISUAL_TOP_N_PER_Q - 1, 1))
).clip(0, 1)

# Score ecológico final

consensus["eco_consensus_score"] = (
    0.28 * consensus["pred_group_freq"]
    + 0.20 * consensus["source_weight_norm_group"]
    + 0.14 * consensus["rr_sum_norm_group"]
    + 0.12 * consensus["base_group_freq"]
    + 0.10 * consensus["base_score_norm_group"]
    + 0.08 * consensus["global_group_freq"]
    + 0.06 * consensus["visual_raw_norm_group"]
    + 0.02 * consensus["global_rank_score"]
)


consensus["has_prediction_support"] = consensus["n_quadrats_pred"] > 0
consensus["has_base_support"] = consensus["n_quadrats_base"] > 0
consensus["has_global_support"] = consensus["n_quadrats_global"] > 0

consensus["n_support_channels"] = (
    consensus["has_prediction_support"].astype(int)
    + consensus["has_base_support"].astype(int)
    + consensus["has_global_support"].astype(int)
)


consensus = consensus.sort_values(
    ["eco_group_id", "eco_consensus_score", "pred_group_freq", "source_weight_sum", "base_score_max", "visual_raw_max"],
    ascending=[True, False, False, False, False, False]
).reset_index(drop=True)

consensus["eco_rank_in_group"] = consensus.groupby("eco_group_id").cumcount() + 1


consensus_top = consensus[
    consensus["eco_rank_in_group"] <= TOP_SPECIES_PER_GROUP_TO_SAVE
].copy()


consensus_top = consensus_top.merge(
    species_table[["species_id", "species", "genus", "family"]],
    on="species_id",
    how="left"
)

print("consensus all shape:", consensus.shape)
print("consensus_top shape:", consensus_top.shape)
print("Grupos en consensus:", consensus_top["eco_group_id"].nunique())
print("Species únicas consensus_top:", consensus_top["species_id"].nunique())

print("\nCandidatos por grupo:")
display(
    consensus_top
    .groupby("eco_group_id")
    .size()
    .reset_index(name="n_species")
    .describe()
)

print("\nDistribución eco_consensus_score:")
display(consensus_top["eco_consensus_score"].describe())

print("\nTop 30 consenso ecológico:")
display(
    consensus_top[
        [
            "eco_group_id",
            "eco_group_strategy",
            "eco_group_size",
            "eco_rank_in_group",
            "species_id",
            "species",
            "eco_consensus_score",
            "pred_group_freq",
            "base_group_freq",
            "global_group_freq",
            "n_quadrats_pred",
            "n_quadrats_base",
            "n_quadrats_global",
            "sources_present",
            "base_best_rank",
            "global_visual_rank_best",
            "n_support_channels",
        ]
    ].head(30)
)


print("\n" + "=" * 80)
print("CREANDO RESUMEN DE GRUPOS")
print("=" * 80)

group_summary = (
    eco_groups
    .groupby(["eco_group_id", "eco_group_strategy"])
    .agg(
        eco_group_size=("quadrat_id", "nunique"),
        qids=("quadrat_id", lambda x: ",".join(list(x)[:20])),
        dates=("qid_date", lambda x: ",".join(sorted(set(x)))),
        sites_l2=("site_l2", lambda x: ",".join(sorted(set(x)))),
        sites_drop1=("site_drop1", lambda x: ",".join(sorted(set(x)))),
    )
    .reset_index()
)

group_species_stats = (
    consensus_top
    .groupby("eco_group_id")
    .agg(
        n_consensus_species=("species_id", "nunique"),
        top1_score=("eco_consensus_score", "max"),
        mean_top10_score=("eco_consensus_score", lambda x: float(pd.Series(x).head(10).mean())),
        n_species_pred_supported=("has_prediction_support", "sum"),
        n_species_base_supported=("has_base_support", "sum"),
        n_species_global_supported=("has_global_support", "sum"),
    )
    .reset_index()
)

group_summary = group_summary.merge(
    group_species_stats,
    on="eco_group_id",
    how="left"
)

group_summary["is_large_group"] = group_summary["eco_group_size"] >= LARGE_GROUP_SIZE
group_summary["is_singleton_group"] = group_summary["eco_group_size"] == 1

print("group_summary shape:", group_summary.shape)

display(
    group_summary
    .sort_values("eco_group_size", ascending=False)
    .head(30)
)

print("\nDistribución tamaño grupos:")
display(group_summary["eco_group_size"].describe())


print("\n" + "=" * 80)
print("GUARDANDO CONSENSO ECOLÓGICO")
print("=" * 80)

eco_groups_path = ECO_DIR / "eco_quadrat_groups.csv"
group_summary_path = ECO_DIR / "eco_group_summary.csv"
consensus_csv_path = ECO_DIR / "eco_group_species_consensus.csv"
consensus_pkl_path = ECO_DIR / "eco_group_species_consensus.pkl.gz"
config_path = ECO_DIR / "eco_consensus_config.json"

eco_groups.to_csv(eco_groups_path, index=False)
group_summary.to_csv(group_summary_path, index=False)
consensus_top.to_csv(consensus_csv_path, index=False)
consensus_top.to_pickle(consensus_pkl_path, compression="gzip")

config = {
    "known_real_scores": KNOWN_REAL_SCORES,
    "strategy": "eco_consensus_by_site_date",
    "base_top_n_per_q": BASE_TOP_N_PER_Q,
    "global_visual_top_n_per_q": GLOBAL_VISUAL_TOP_N_PER_Q,
    "top_species_per_group_to_save": TOP_SPECIES_PER_GROUP_TO_SAVE,
    "small_group_min_size": SMALL_GROUP_MIN_SIZE,
    "large_group_size": LARGE_GROUP_SIZE,
    "inputs": {
        "test": str(TEST_PATH),
        "species_table": str(SPECIES_TABLE_PATH),
        "champion_submission": str(CHAMPION_SUB_PATH),
        "v4_submission": str(V4_SUB_PATH),
        "B_submission": str(B_SUB_PATH),
        "base_agg": str(BASE_AGG_PATH),
        "global_top300": str(GLOBAL_TOP300_PATH),
    },
    "outputs": {
        "eco_groups": str(eco_groups_path),
        "group_summary": str(group_summary_path),
        "consensus_csv": str(consensus_csv_path),
        "consensus_pkl": str(consensus_pkl_path),
    },
    "score_formula": {
        "pred_group_freq": 0.28,
        "source_weight_norm_group": 0.20,
        "rr_sum_norm_group": 0.14,
        "base_group_freq": 0.12,
        "base_score_norm_group": 0.10,
        "global_group_freq": 0.08,
        "visual_raw_norm_group": 0.06,
        "global_rank_score": 0.02,
    },
}

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("Eco groups:", eco_groups_path)
print("Group summary:", group_summary_path)
print("Consensus CSV:", consensus_csv_path)
print("Consensus PKL:", consensus_pkl_path)
print("Config:", config_path)


del base_agg, base_top, global_top, sub_long, consensus
gc.collect()

print("\n Consenso ecológico listo.")

# **CELL 22 — Generate ECO_CONSENSUS submissions**

In [ ]:
import ast
import re
import json
import gc
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

BASE_RUN_DIR = STORAGE_ROOT / "runs" / "dinov2_tiling_adaptive_k_v1"
GLOBAL_PROTO_ROOT = STORAGE_ROOT / "runs" / "global_prototype_recovery"
GLOBAL_SIM_DIR = GLOBAL_PROTO_ROOT / "global_test_similarity"

ECO_DIR = STORAGE_ROOT / "runs" / "eco_consensus"
ECO_SUB_DIR = ECO_DIR / "eco_submissions"

ECO_SUB_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"

CHAMPION_SUB_PATH = SUBMISSIONS_DIR / "submission_GLOBAL_PROTO_ULTRA_TARGET20_v1.csv"

ECO_GROUPS_PATH = ECO_DIR / "eco_quadrat_groups.csv"
ECO_CONSENSUS_PATH = ECO_DIR / "eco_group_species_consensus.pkl.gz"
ECO_GROUP_SUMMARY_PATH = ECO_DIR / "eco_group_summary.csv"

BASE_AGG_PATH = BASE_RUN_DIR / "aggregation" / "quadrat_species_scores.pkl.gz"
GLOBAL_TOP300_PATH = GLOBAL_SIM_DIR / "global_visual_candidates_top300.pkl.gz"

print("=" * 80)
print("CELDA 38 — GENERAR SUBMISSIONS ECO_CONSENSUS")
print("=" * 80)

paths_to_check = [
    TEST_PATH,
    SPECIES_TABLE_PATH,
    CHAMPION_SUB_PATH,
    ECO_GROUPS_PATH,
    ECO_CONSENSUS_PATH,
    ECO_GROUP_SUMMARY_PATH,
    BASE_AGG_PATH,
    GLOBAL_TOP300_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


KNOWN_REAL_SCORES = {
    "B_precision_v1": 0.31540,
    "PROTO_RERANK_MILD_v4_near_max1": 0.31649,
    "GLOBAL_PROTO_ULTRA_TARGET20_v1": 0.31660,
}


ECO_CONFIGS = {

    "ECO_CONSENSUS_MAX1_SAFE_v1": {
        "mode": "replace",
        "max_new_species_per_quadrat": 1,
        "max_group_rank": 8,
        "min_eco_score": 0.42,
        "min_pred_group_freq": 0.04,
        "min_base_group_freq": 0.25,
        "min_global_group_freq": 0.15,
        "min_support_channels": 2,
        "require_local_support": True,
        "local_base_rank_cap": 120,
        "local_global_rank_cap": 80,
        "local_min_base_norm": 0.04,
        "local_min_visual_norm": 0.45,
        "max_removed_protect_score": 0.55,
        "min_removed_base_rank": 8,
        "min_removed_global_rank": 120,
        "min_removed_eco_rank": 25,
        "min_score_gap": 0.06,
        "max_group_size": 60,
        "allow_singleton_groups": False,
        "w_eco": 0.46,
        "w_pred_freq": 0.12,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.12,
        "w_local_visual": 0.08,
        "w_rank": 0.04,
    },

    
    "ECO_CONSENSUS_MAX1_BALANCED_v1": {
        "mode": "replace",
        "max_new_species_per_quadrat": 1,
        "max_group_rank": 15,
        "min_eco_score": 0.30,
        "min_pred_group_freq": 0.02,
        "min_base_group_freq": 0.18,
        "min_global_group_freq": 0.10,
        "min_support_channels": 2,
        "require_local_support": False,
        "local_base_rank_cap": 180,
        "local_global_rank_cap": 120,
        "local_min_base_norm": 0.02,
        "local_min_visual_norm": 0.35,
        "max_removed_protect_score": 0.65,
        "min_removed_base_rank": 6,
        "min_removed_global_rank": 100,
        "min_removed_eco_rank": 20,
        "min_score_gap": 0.03,
        "max_group_size": 80,
        "allow_singleton_groups": False,
        "w_eco": 0.52,
        "w_pred_freq": 0.12,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.08,
        "w_local_visual": 0.06,
        "w_rank": 0.04,
    },

    
    "ECO_CONSENSUS_MAX1_AGGRESSIVE_v1": {
        "mode": "replace",
        "max_new_species_per_quadrat": 1,
        "max_group_rank": 25,
        "min_eco_score": 0.22,
        "min_pred_group_freq": 0.00,
        "min_base_group_freq": 0.12,
        "min_global_group_freq": 0.08,
        "min_support_channels": 2,
        "require_local_support": False,
        "local_base_rank_cap": 250,
        "local_global_rank_cap": 160,
        "local_min_base_norm": 0.00,
        "local_min_visual_norm": 0.25,
        "max_removed_protect_score": 0.72,
        "min_removed_base_rank": 5,
        "min_removed_global_rank": 80,
        "min_removed_eco_rank": 15,
        "min_score_gap": 0.00,
        "max_group_size": 120,
        "allow_singleton_groups": False,
        "w_eco": 0.58,
        "w_pred_freq": 0.08,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.06,
        "w_local_visual": 0.06,
        "w_rank": 0.04,
    },

   
    "ECO_CONSENSUS_MAX2_BALANCED_v1": {
        "mode": "replace",
        "max_new_species_per_quadrat": 2,
        "max_group_rank": 18,
        "min_eco_score": 0.28,
        "min_pred_group_freq": 0.02,
        "min_base_group_freq": 0.16,
        "min_global_group_freq": 0.10,
        "min_support_channels": 2,
        "require_local_support": False,
        "local_base_rank_cap": 180,
        "local_global_rank_cap": 120,
        "local_min_base_norm": 0.02,
        "local_min_visual_norm": 0.30,
        "max_removed_protect_score": 0.62,
        "min_removed_base_rank": 6,
        "min_removed_global_rank": 100,
        "min_removed_eco_rank": 20,
        "min_score_gap": 0.04,
        "max_group_size": 80,
        "allow_singleton_groups": False,
        "w_eco": 0.52,
        "w_pred_freq": 0.12,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.08,
        "w_local_visual": 0.06,
        "w_rank": 0.04,
    },

    
    "ECO_CONSENSUS_MAX2_AGGRESSIVE_v1": {
        "mode": "replace",
        "max_new_species_per_quadrat": 2,
        "max_group_rank": 30,
        "min_eco_score": 0.20,
        "min_pred_group_freq": 0.00,
        "min_base_group_freq": 0.10,
        "min_global_group_freq": 0.06,
        "min_support_channels": 2,
        "require_local_support": False,
        "local_base_rank_cap": 300,
        "local_global_rank_cap": 180,
        "local_min_base_norm": 0.00,
        "local_min_visual_norm": 0.20,
        "max_removed_protect_score": 0.74,
        "min_removed_base_rank": 5,
        "min_removed_global_rank": 70,
        "min_removed_eco_rank": 12,
        "min_score_gap": 0.00,
        "max_group_size": 120,
        "allow_singleton_groups": False,
        "w_eco": 0.60,
        "w_pred_freq": 0.06,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.06,
        "w_local_visual": 0.06,
        "w_rank": 0.04,
    },

    
    "ECO_CONSENSUS_ADD1_SAFE_v1": {
        "mode": "add1",
        "max_new_species_per_quadrat": 1,
        "max_group_rank": 8,
        "min_eco_score": 0.46,
        "min_pred_group_freq": 0.04,
        "min_base_group_freq": 0.28,
        "min_global_group_freq": 0.18,
        "min_support_channels": 2,
        "require_local_support": True,
        "local_base_rank_cap": 120,
        "local_global_rank_cap": 80,
        "local_min_base_norm": 0.04,
        "local_min_visual_norm": 0.45,
        "max_removed_protect_score": 1.0,
        "min_removed_base_rank": 0,
        "min_removed_global_rank": 0,
        "min_removed_eco_rank": 0,
        "min_score_gap": 0.00,
        "max_group_size": 60,
        "allow_singleton_groups": False,
        "w_eco": 0.50,
        "w_pred_freq": 0.14,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.10,
        "w_local_visual": 0.05,
        "w_rank": 0.03,
    },

    
    "ECO_CONSENSUS_ADD1_BALANCED_v1": {
        "mode": "add1",
        "max_new_species_per_quadrat": 1,
        "max_group_rank": 15,
        "min_eco_score": 0.34,
        "min_pred_group_freq": 0.02,
        "min_base_group_freq": 0.20,
        "min_global_group_freq": 0.10,
        "min_support_channels": 2,
        "require_local_support": False,
        "local_base_rank_cap": 180,
        "local_global_rank_cap": 120,
        "local_min_base_norm": 0.02,
        "local_min_visual_norm": 0.30,
        "max_removed_protect_score": 1.0,
        "min_removed_base_rank": 0,
        "min_removed_global_rank": 0,
        "min_removed_eco_rank": 0,
        "min_score_gap": 0.00,
        "max_group_size": 80,
        "allow_singleton_groups": False,
        "w_eco": 0.56,
        "w_pred_freq": 0.10,
        "w_base_group": 0.10,
        "w_global_group": 0.08,
        "w_local_base": 0.08,
        "w_local_visual": 0.05,
        "w_rank": 0.03,
    },
}

print("\nConfiguraciones ECO:")
print(json.dumps(ECO_CONFIGS, indent=2))


def parse_species_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def species_list_to_str(xs):
    return "[" + ", ".join(map(str, [int(x) for x in xs])) + "]"


def safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def safe_int(x, default=999999):
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


def safe_mean(x):
    s = pd.Series(x).dropna()
    if len(s) == 0:
        return np.nan
    return float(s.mean())


def validate_submission_df(df_sub, test_df, valid_species, max_k=8, min_k=2):
    errors = []

    if list(df_sub.columns) != ["quadrat_id", "species_ids"]:
        errors.append(f"Columnas incorrectas: {df_sub.columns.tolist()}")

    if len(df_sub) != len(test_df):
        errors.append(f"Filas incorrectas: {len(df_sub)} vs esperado {len(test_df)}")

    valid_quadrats = set(test_df["quadrat_id"].astype(str))
    qids = set(df_sub["quadrat_id"].astype(str))

    if valid_quadrats - qids:
        errors.append(f"Faltan quadrats: {len(valid_quadrats - qids)}")

    if qids - valid_quadrats:
        errors.append(f"Sobran quadrats: {len(qids - valid_quadrats)}")

    if df_sub["quadrat_id"].duplicated().sum() > 0:
        errors.append("Hay quadrat_id duplicados.")

    lists = df_sub["species_ids"].apply(parse_species_ids)
    k = lists.apply(len)

    if (k < min_k).any():
        errors.append(f"Filas con K < {min_k}: {(k < min_k).sum()}")

    if (k > max_k).any():
        errors.append(f"Filas con K > {max_k}: {(k > max_k).sum()}")

    repeated = lists.apply(lambda x: len(x) != len(set(x))).sum()
    if repeated > 0:
        errors.append(f"Filas con especies repetidas: {repeated}")

    invalid = 0
    for sp_list in lists:
        invalid += sum(1 for sp in sp_list if int(sp) not in valid_species)

    if invalid > 0:
        errors.append(f"Species inválidas: {invalid}")

    return errors, k


def rank_score(rank, max_rank):
    rank = safe_float(rank, default=max_rank + 1)
    if rank <= 0:
        return 0.0
    return max(0.0, 1.0 - ((rank - 1.0) / max(max_rank - 1.0, 1.0)))


def build_local_lookup_df(base_df_q, global_df_q):
    """
    Une soporte local del clasificador base y soporte visual global para un quadrat.
    """
    if base_df_q is None:
        base_df_q = pd.DataFrame(columns=["species_id"])

    if global_df_q is None:
        global_df_q = pd.DataFrame(columns=["species_id"])

    b = base_df_q.copy()
    g = global_df_q.copy()

    if len(b) > 0:
        b = b.sort_values("base_rank", ascending=True).drop_duplicates("species_id")
    if len(g) > 0:
        g = g.sort_values("global_visual_rank", ascending=True).drop_duplicates("species_id")

    local = b.merge(g, on="species_id", how="outer")

    fill_cols_float = [
        "base_score",
        "base_norm_top1",
        "base_max_score",
        "base_freq_score",
        "visual_norm_global",
        "visual_raw",
    ]

    for c in fill_cols_float:
        if c in local.columns:
            local[c] = local[c].fillna(0.0)
        else:
            local[c] = 0.0

    for c in ["base_rank", "global_visual_rank", "n_unique_views", "n_tile_hits"]:
        if c in local.columns:
            local[c] = local[c].fillna(999999)
        else:
            local[c] = 999999

    return local


def score_candidate_row(row, cfg):
    """
    Score de una especie candidata para entrar desde consenso ecológico.
    """
    eco = safe_float(row.get("eco_consensus_score", 0.0))
    pred_freq = safe_float(row.get("pred_group_freq", 0.0))
    base_freq = safe_float(row.get("base_group_freq", 0.0))
    global_freq = safe_float(row.get("global_group_freq", 0.0))
    local_base = safe_float(row.get("base_norm_top1", 0.0))
    local_visual = safe_float(row.get("visual_norm_global", 0.0))
    eco_rank = safe_float(row.get("eco_rank_in_group", 999999))

    rscore = rank_score(eco_rank, max(cfg["max_group_rank"], 1))

    return (
        cfg["w_eco"] * eco
        + cfg["w_pred_freq"] * pred_freq
        + cfg["w_base_group"] * base_freq
        + cfg["w_global_group"] * global_freq
        + cfg["w_local_base"] * local_base
        + cfg["w_local_visual"] * local_visual
        + cfg["w_rank"] * rscore
    )


def protect_current_row(row, K):
    """
    Score de protección de una especie actual.
    Bajo score = más fácil de remover.
    """
    local_base = safe_float(row.get("base_norm_top1", 0.0))
    local_visual = safe_float(row.get("visual_norm_global", 0.0))
    eco_score = safe_float(row.get("eco_consensus_score", 0.0))
    pred_freq = safe_float(row.get("pred_group_freq", 0.0))
    rank_in_pred = safe_float(row.get("rank_in_champion", K))

    if K <= 1:
        rank_protect = 1.0
    else:
        rank_protect = 1.0 - ((rank_in_pred - 1.0) / (K - 1.0))
        rank_protect = max(0.0, min(1.0, rank_protect))

    return (
        0.34 * local_base
        + 0.24 * local_visual
        + 0.22 * eco_score
        + 0.10 * pred_freq
        + 0.10 * rank_protect
    )


def get_species_row_from_df(df, species_id):
    hit = df[df["species_id"].astype(int) == int(species_id)]
    if len(hit) == 0:
        return {}
    return hit.iloc[0].to_dict()


def build_eco_candidate_for_qid(qid, champion_list, group_df, local_df, group_info, cfg):
    """
    Genera lista ECO para un quadrat.
    """
    mode = cfg["mode"]
    K = len(champion_list)
    current_set = set(int(x) for x in champion_list)

    
    eco_group_size = int(group_info.get("eco_group_size", 1))
    is_singleton = eco_group_size <= 1

    if is_singleton and not cfg["allow_singleton_groups"]:
        return list(champion_list), {
            "changed_set": 0,
            "reason": "singleton_group",
            "K": K,
            "K_champion": K,
        }

    if eco_group_size > int(cfg["max_group_size"]):
        return list(champion_list), {
            "changed_set": 0,
            "reason": "group_too_large",
            "K": K,
            "K_champion": K,
        }


    cand = group_df.copy()

    cand = cand[
        (cand["eco_rank_in_group"] <= cfg["max_group_rank"])
        & (cand["eco_consensus_score"] >= cfg["min_eco_score"])
        & (cand["n_support_channels"] >= cfg["min_support_channels"])
        & (
            (cand["pred_group_freq"] >= cfg["min_pred_group_freq"])
            | (cand["base_group_freq"] >= cfg["min_base_group_freq"])
            | (cand["global_group_freq"] >= cfg["min_global_group_freq"])
        )
    ].copy()

    cand = cand[~cand["species_id"].astype(int).isin(current_set)].copy()

    if len(cand) == 0:
        return list(champion_list), {
            "changed_set": 0,
            "reason": "no_group_candidate",
            "K": K,
            "K_champion": K,
        }

   
    cand = cand.merge(
        local_df,
        on="species_id",
        how="left",
        suffixes=("", "_local")
    )

   
    for c in ["base_score", "base_norm_top1", "base_max_score", "base_freq_score", "visual_norm_global", "visual_raw"]:
        cand[c] = cand[c].fillna(0.0)

    for c in ["base_rank", "global_visual_rank", "n_unique_views", "n_tile_hits"]:
        cand[c] = cand[c].fillna(999999)

    cand["local_supported"] = (
        (cand["base_rank"] <= cfg["local_base_rank_cap"])
        | (cand["global_visual_rank"] <= cfg["local_global_rank_cap"])
        | (cand["base_norm_top1"] >= cfg["local_min_base_norm"])
        | (cand["visual_norm_global"] >= cfg["local_min_visual_norm"])
    )

    if cfg["require_local_support"]:
        cand = cand[cand["local_supported"]].copy()

    if len(cand) == 0:
        return list(champion_list), {
            "changed_set": 0,
            "reason": "no_local_supported_candidate",
            "K": K,
            "K_champion": K,
        }

    cand["eco_candidate_score"] = cand.apply(lambda r: score_candidate_row(r, cfg), axis=1)

    cand = cand.sort_values(
        ["eco_candidate_score", "eco_consensus_score", "pred_group_freq", "base_group_freq", "global_group_freq"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)


    if mode == "add1":
        selected = list(champion_list)

        if K < 8:
            best = cand.iloc[0]
            add_sp = int(best["species_id"])
            selected.append(add_sp)

        selected = list(dict.fromkeys([int(x) for x in selected]))[:8]

        added_to_champion = [sp for sp in selected if sp not in current_set]
        changed = int(set(selected) != current_set)

        debug = {
            "changed_set": changed,
            "reason": "add1" if changed else "already_K8_or_no_add",
            "K": len(selected),
            "K_champion": K,
            "n_removed_from_champion": 0,
            "n_added_to_champion": len(added_to_champion),
            "removed_from_champion": species_list_to_str([]),
            "added_to_champion": species_list_to_str(added_to_champion),
            "selected_species": species_list_to_str(selected),
            "champion_species": species_list_to_str(champion_list),
            "best_candidate_score": float(cand["eco_candidate_score"].iloc[0]),
            "best_candidate_eco_rank": float(cand["eco_rank_in_group"].iloc[0]),
            "best_candidate_eco_score": float(cand["eco_consensus_score"].iloc[0]),
            "best_candidate_pred_freq": float(cand["pred_group_freq"].iloc[0]),
            "best_candidate_base_freq": float(cand["base_group_freq"].iloc[0]),
            "best_candidate_global_freq": float(cand["global_group_freq"].iloc[0]),
            "best_candidate_base_rank": float(cand["base_rank"].iloc[0]),
            "best_candidate_global_rank": float(cand["global_visual_rank"].iloc[0]),
            "eco_group_size": eco_group_size,
        }

        return selected, debug


    current_rows = []

    for rank, sp in enumerate(champion_list, start=1):
        sp = int(sp)

        g_row = get_species_row_from_df(group_df, sp)
        l_row = get_species_row_from_df(local_df, sp)

        row = {
            "species_id": sp,
            "rank_in_champion": rank,
            "eco_consensus_score": safe_float(g_row.get("eco_consensus_score", 0.0)),
            "eco_rank_in_group": safe_float(g_row.get("eco_rank_in_group", 999999)),
            "pred_group_freq": safe_float(g_row.get("pred_group_freq", 0.0)),
            "base_group_freq": safe_float(g_row.get("base_group_freq", 0.0)),
            "global_group_freq": safe_float(g_row.get("global_group_freq", 0.0)),
            "n_support_channels": safe_int(g_row.get("n_support_channels", 0), 0),
            "base_score": safe_float(l_row.get("base_score", 0.0)),
            "base_norm_top1": safe_float(l_row.get("base_norm_top1", 0.0)),
            "base_rank": safe_float(l_row.get("base_rank", 999999)),
            "visual_norm_global": safe_float(l_row.get("visual_norm_global", 0.0)),
            "visual_raw": safe_float(l_row.get("visual_raw", 0.0)),
            "global_visual_rank": safe_float(l_row.get("global_visual_rank", 999999)),
        }

        current_rows.append(row)

    current_df = pd.DataFrame(current_rows)
    current_df["protect_score"] = current_df.apply(lambda r: protect_current_row(r, K), axis=1)

    current_df["is_removable"] = (
        (current_df["protect_score"] <= cfg["max_removed_protect_score"])
        & (
            (current_df["base_rank"] >= cfg["min_removed_base_rank"])
            | (current_df["global_visual_rank"] >= cfg["min_removed_global_rank"])
            | (current_df["eco_rank_in_group"] >= cfg["min_removed_eco_rank"])
        )
    )

    removable = current_df[current_df["is_removable"]].copy()

    if len(removable) == 0:
        return list(champion_list), {
            "changed_set": 0,
            "reason": "no_removable_species",
            "K": K,
            "K_champion": K,
        }

    selected = list(champion_list)
    changes = []

   
    max_new = int(cfg["max_new_species_per_quadrat"])

    for step in range(max_new):
        cand_available = cand[
            ~cand["species_id"].astype(int).isin(set(selected))
        ].copy()

        removable_available = current_df[
            current_df["species_id"].astype(int).isin(set(selected))
            & (current_df["is_removable"])
        ].copy()

        if len(cand_available) == 0 or len(removable_available) == 0:
            break

        cand_available = cand_available.sort_values(
            ["eco_candidate_score", "eco_consensus_score", "pred_group_freq", "base_group_freq"],
            ascending=[False, False, False, False]
        ).reset_index(drop=True)

        removable_available = removable_available.sort_values(
            ["protect_score", "base_norm_top1", "visual_norm_global", "eco_consensus_score"],
            ascending=[True, True, True, True]
        ).reset_index(drop=True)

        add_row = cand_available.iloc[0]
        remove_row = removable_available.iloc[0]

        add_score = float(add_row["eco_candidate_score"])
        remove_protect = float(remove_row["protect_score"])

        if add_score < remove_protect + cfg["min_score_gap"]:
            break

        add_sp = int(add_row["species_id"])
        remove_sp = int(remove_row["species_id"])

        
        selected = [
            add_sp if int(sp) == remove_sp else int(sp)
            for sp in selected
        ]

        selected = list(dict.fromkeys(selected))

       
        if len(selected) < K:
            for sp in champion_list:
                if int(sp) not in selected:
                    selected.append(int(sp))
                if len(selected) >= K:
                    break

        selected = selected[:K]

        changes.append({
            "add_sp": add_sp,
            "remove_sp": remove_sp,
            "add_score": add_score,
            "remove_protect": remove_protect,
            "add_eco_rank": float(add_row["eco_rank_in_group"]),
            "add_eco_score": float(add_row["eco_consensus_score"]),
            "add_pred_freq": float(add_row["pred_group_freq"]),
            "add_base_freq": float(add_row["base_group_freq"]),
            "add_global_freq": float(add_row["global_group_freq"]),
            "add_base_rank": float(add_row["base_rank"]),
            "add_global_rank": float(add_row["global_visual_rank"]),
            "remove_base_rank": float(remove_row["base_rank"]),
            "remove_global_rank": float(remove_row["global_visual_rank"]),
            "remove_eco_rank": float(remove_row["eco_rank_in_group"]),
            "remove_base_norm": float(remove_row["base_norm_top1"]),
            "remove_visual_norm": float(remove_row["visual_norm_global"]),
            "remove_eco_score": float(remove_row["eco_consensus_score"]),
        })

    selected = list(dict.fromkeys([int(x) for x in selected]))

    if len(selected) < K:
        for sp in champion_list:
            if int(sp) not in selected:
                selected.append(int(sp))
            if len(selected) >= K:
                break

    selected = selected[:K]

    selected_set = set(selected)
    removed_from_champion = [int(sp) for sp in champion_list if int(sp) not in selected_set]
    added_to_champion = [int(sp) for sp in selected if int(sp) not in current_set]

    changed = int(set(selected) != current_set)

    debug = {
        "changed_set": changed,
        "reason": "replaced" if changed else "no_score_gap",
        "K": len(selected),
        "K_champion": K,
        "n_removed_from_champion": len(removed_from_champion),
        "n_added_to_champion": len(added_to_champion),
        "removed_from_champion": species_list_to_str(removed_from_champion),
        "added_to_champion": species_list_to_str(added_to_champion),
        "selected_species": species_list_to_str(selected),
        "champion_species": species_list_to_str(champion_list),
        "eco_group_size": eco_group_size,
        "n_group_candidates_initial": int(len(group_df)),
        "n_candidates_eligible": int(len(cand)),
        "n_removable_species": int(len(removable)),
        "mean_add_score": safe_mean([c["add_score"] for c in changes]),
        "mean_remove_protect": safe_mean([c["remove_protect"] for c in changes]),
        "mean_add_eco_rank": safe_mean([c["add_eco_rank"] for c in changes]),
        "mean_add_eco_score": safe_mean([c["add_eco_score"] for c in changes]),
        "mean_add_pred_freq": safe_mean([c["add_pred_freq"] for c in changes]),
        "mean_add_base_freq": safe_mean([c["add_base_freq"] for c in changes]),
        "mean_add_global_freq": safe_mean([c["add_global_freq"] for c in changes]),
        "mean_add_base_rank": safe_mean([c["add_base_rank"] for c in changes]),
        "mean_add_global_rank": safe_mean([c["add_global_rank"] for c in changes]),
        "mean_remove_base_rank": safe_mean([c["remove_base_rank"] for c in changes]),
        "mean_remove_global_rank": safe_mean([c["remove_global_rank"] for c in changes]),
        "mean_remove_eco_rank": safe_mean([c["remove_eco_rank"] for c in changes]),
        "mean_remove_base_norm": safe_mean([c["remove_base_norm"] for c in changes]),
        "mean_remove_visual_norm": safe_mean([c["remove_visual_norm"] for c in changes]),
        "mean_remove_eco_score": safe_mean([c["remove_eco_score"] for c in changes]),
    }

    return selected, debug


print("\n" + "=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

test_df = pd.read_pickle(TEST_PATH, compression="gzip")
test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)

species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")
species_table["species_id"] = species_table["species_id"].astype(int)
VALID_SPECIES = set(species_table["species_id"].astype(int).tolist())

champion_sub = pd.read_csv(CHAMPION_SUB_PATH)
champion_sub["quadrat_id"] = champion_sub["quadrat_id"].astype(str)
champion_sub["species_list_champion"] = champion_sub["species_ids"].apply(parse_species_ids)
champion_sub["K_champion"] = champion_sub["species_list_champion"].apply(len)

eco_groups = pd.read_csv(ECO_GROUPS_PATH)
eco_groups["quadrat_id"] = eco_groups["quadrat_id"].astype(str)

group_summary = pd.read_csv(ECO_GROUP_SUMMARY_PATH)
consensus = pd.read_pickle(ECO_CONSENSUS_PATH, compression="gzip")

consensus["species_id"] = consensus["species_id"].astype(int)

print("test_df:", test_df.shape)
print("champion_sub:", champion_sub.shape)
print("eco_groups:", eco_groups.shape)
print("group_summary:", group_summary.shape)
print("consensus:", consensus.shape)


print("\n" + "=" * 80)
print("CARGANDO SOPORTE LOCAL BASE")
print("=" * 80)

base_agg = pd.read_pickle(BASE_AGG_PATH, compression="gzip")

needed_base_cols = [
    "quadrat_id",
    "species_id",
    "final_score",
    "max_score",
    "n_views_species",
    "freq_score",
]

missing = [c for c in needed_base_cols if c not in base_agg.columns]
if missing:
    raise RuntimeError(f"Faltan columnas en base_agg: {missing}")

base_agg = base_agg[needed_base_cols].copy()
base_agg["quadrat_id"] = base_agg["quadrat_id"].astype(str)
base_agg["species_id"] = base_agg["species_id"].astype(int)

base_agg = base_agg.sort_values(
    ["quadrat_id", "final_score"],
    ascending=[True, False]
).reset_index(drop=True)

base_agg["base_rank"] = base_agg.groupby("quadrat_id").cumcount() + 1

base_top1 = (
    base_agg
    .groupby("quadrat_id")["final_score"]
    .max()
    .reset_index(name="base_top1_score")
)

base_agg = base_agg.merge(base_top1, on="quadrat_id", how="left")
base_agg["base_norm_top1"] = (
    base_agg["final_score"] / base_agg["base_top1_score"].replace(0, np.nan)
).fillna(0.0)


base_local = base_agg[base_agg["base_rank"] <= 300].copy()

base_local = base_local.rename(columns={
    "final_score": "base_score",
    "max_score": "base_max_score",
    "n_views_species": "base_n_views_species",
    "freq_score": "base_freq_score",
})

base_local = base_local[
    [
        "quadrat_id",
        "species_id",
        "base_score",
        "base_norm_top1",
        "base_max_score",
        "base_freq_score",
        "base_rank",
    ]
].copy()

print("base_local:", base_local.shape)

del base_agg
gc.collect()


print("\n" + "=" * 80)
print("CARGANDO SOPORTE VISUAL GLOBAL LOCAL")
print("=" * 80)

global_top300 = pd.read_pickle(GLOBAL_TOP300_PATH, compression="gzip")

global_top300["quadrat_id"] = global_top300["quadrat_id"].astype(str)
global_top300["species_id"] = global_top300["species_id"].astype(int)

global_local = global_top300[
    [
        "quadrat_id",
        "species_id",
        "global_visual_rank",
        "visual_norm_global",
        "visual_raw",
        "n_unique_views",
        "n_tile_hits",
    ]
].copy()

print("global_local:", global_local.shape)

del global_top300
gc.collect()


print("\n" + "=" * 80)
print("CREANDO LOOKUPS")
print("=" * 80)

champion_lookup = {
    row.quadrat_id: [int(sp) for sp in row.species_list_champion]
    for row in champion_sub.itertuples(index=False)
}

eco_group_lookup = (
    eco_groups
    .set_index("quadrat_id")
    .to_dict(orient="index")
)

group_info_lookup = (
    group_summary
    .set_index("eco_group_id")
    .to_dict(orient="index")
)

consensus_groups = {
    gid: df.reset_index(drop=True)
    for gid, df in consensus.groupby("eco_group_id")
}

base_groups = {
    qid: df.drop(columns=["quadrat_id"]).reset_index(drop=True)
    for qid, df in base_local.groupby("quadrat_id")
}

global_groups = {
    qid: df.drop(columns=["quadrat_id"]).reset_index(drop=True)
    for qid, df in global_local.groupby("quadrat_id")
}

print("champion_lookup:", len(champion_lookup))
print("eco_group_lookup:", len(eco_group_lookup))
print("group_info_lookup:", len(group_info_lookup))
print("consensus_groups:", len(consensus_groups))
print("base_groups:", len(base_groups))
print("global_groups:", len(global_groups))


print("\n" + "=" * 80)
print("GENERANDO SUBMISSIONS ECO_CONSENSUS")
print("=" * 80)

summary_rows = []

for cand_name, cfg in ECO_CONFIGS.items():
    print("\n" + "#" * 100)
    print("CANDIDATA:", cand_name)
    print("#" * 100)
    print(json.dumps(cfg, indent=2))

    sub_rows = []
    debug_rows = []

    for qid in tqdm(test_df["quadrat_id"].astype(str).tolist(), desc=f"generate_{cand_name}"):
        if qid not in champion_lookup:
            raise RuntimeError(f"No hay champion para quadrat_id={qid}")

        champion_list = champion_lookup[qid]

        eco_info = eco_group_lookup.get(qid, None)
        if eco_info is None:
            selected = champion_list
            debug = {
                "changed_set": 0,
                "reason": "no_eco_group",
                "K": len(selected),
                "K_champion": len(champion_list),
            }
        else:
            gid = eco_info["eco_group_id"]
            group_df = consensus_groups.get(gid, pd.DataFrame())
            group_info = group_info_lookup.get(gid, {})

            base_df_q = base_groups.get(qid, pd.DataFrame(columns=["species_id"]))
            global_df_q = global_groups.get(qid, pd.DataFrame(columns=["species_id"]))

            local_df = build_local_lookup_df(base_df_q, global_df_q)

            selected, debug = build_eco_candidate_for_qid(
                qid=qid,
                champion_list=champion_list,
                group_df=group_df,
                local_df=local_df,
                group_info=group_info,
                cfg=cfg,
            )

            debug["eco_group_id"] = gid
            debug["eco_group_strategy"] = eco_info.get("eco_group_strategy", None)

        selected = [int(sp) for sp in selected if int(sp) in VALID_SPECIES]

       
        if cfg["mode"] == "replace":
            if len(selected) < len(champion_list):
                for sp in champion_list:
                    if int(sp) not in selected and int(sp) in VALID_SPECIES:
                        selected.append(int(sp))
                    if len(selected) >= len(champion_list):
                        break
            selected = selected[:len(champion_list)]

        8
        if cfg["mode"] == "add1":
            selected = list(dict.fromkeys(selected))[:8]

        sub_rows.append({
            "quadrat_id": qid,
            "species_ids": species_list_to_str(selected),
        })

        debug_rows.append({
            "quadrat_id": qid,
            **debug,
        })

    df_sub = pd.DataFrame(sub_rows)
    df_debug = pd.DataFrame(debug_rows)

   
    errors, k_series = validate_submission_df(
        df_sub,
        test_df=test_df,
        valid_species=VALID_SPECIES,
        max_k=8,
        min_k=2,
    )

    if errors:
        print(" Errores:")
        for e in errors:
            print("-", e)
        raise RuntimeError(f"Submission inválida: {cand_name}")

    print(" Submission válida")

    changed_images = int(df_debug["changed_set"].sum())
    pct_changed = changed_images / len(df_debug) * 100

    removed_col = "n_removed_from_champion"
    added_col = "n_added_to_champion"

    if removed_col not in df_debug.columns:
        df_debug[removed_col] = 0
    if added_col not in df_debug.columns:
        df_debug[added_col] = 0

    total_removed = int(df_debug[removed_col].fillna(0).sum())
    total_added = int(df_debug[added_col].fillna(0).sum())
    max_removed = int(df_debug[removed_col].fillna(0).max())
    max_added = int(df_debug[added_col].fillna(0).max())

    k_dist = k_series.value_counts().sort_index().reset_index()
    k_dist.columns = ["K", "n_images"]

    print("Changed images:", changed_images, f"({pct_changed:.2f}%)")
    print("Total removed:", total_removed)
    print("Total added:", total_added)
    print("Max removed/image:", max_removed)
    print("Max added/image:", max_added)
    print("Mean K:", round(float(k_series.mean()), 4))
    print("Pct K=8:", round(float((k_series == 8).mean() * 100), 2))

    print("\nDistribución K:")
    display(k_dist)

    print("\nRazones:")
    display(df_debug["reason"].value_counts().reset_index(name="n_images"))

    changed_debug = df_debug[df_debug["changed_set"] == 1].copy()

    if len(changed_debug) > 0:
        cols_to_show = [
            c for c in [
                "n_removed_from_champion",
                "n_added_to_champion",
                "n_candidates_eligible",
                "n_removable_species",
                "mean_add_score",
                "mean_remove_protect",
                "mean_add_eco_rank",
                "mean_add_eco_score",
                "mean_add_pred_freq",
                "mean_add_base_freq",
                "mean_add_global_freq",
                "mean_add_base_rank",
                "mean_add_global_rank",
                "mean_remove_base_rank",
                "mean_remove_global_rank",
                "mean_remove_eco_rank",
                "mean_remove_base_norm",
                "mean_remove_visual_norm",
                "mean_remove_eco_score",
                "best_candidate_score",
                "best_candidate_eco_rank",
                "best_candidate_eco_score",
            ] if c in changed_debug.columns
        ]

        print("\nResumen cambios:")
        display(changed_debug[cols_to_show].describe())

    
    sub_path_public = SUBMISSIONS_DIR / f"submission_{cand_name}.csv"
    sub_path_run = ECO_SUB_DIR / f"submission_{cand_name}.csv"
    debug_path = ECO_SUB_DIR / f"debug_{cand_name}.csv"

    df_sub.to_csv(sub_path_public, index=False)
    df_sub.to_csv(sub_path_run, index=False)
    df_debug.to_csv(debug_path, index=False)

  
    added_counter = Counter()
    removed_counter = Counter()

    if "added_to_champion" in df_debug.columns:
        for xs in df_debug["added_to_champion"].fillna("[]").apply(parse_species_ids):
            added_counter.update(xs)

    if "removed_from_champion" in df_debug.columns:
        for xs in df_debug["removed_from_champion"].fillna("[]").apply(parse_species_ids):
            removed_counter.update(xs)

    added_df = pd.DataFrame(
        added_counter.items(),
        columns=["species_id", "n_added"]
    ).sort_values("n_added", ascending=False)

    removed_df = pd.DataFrame(
        removed_counter.items(),
        columns=["species_id", "n_removed"]
    ).sort_values("n_removed", ascending=False)

    if len(added_df) > 0:
        added_df = added_df.merge(
            species_table[["species_id", "species", "genus", "family"]],
            on="species_id",
            how="left"
        )

    if len(removed_df) > 0:
        removed_df = removed_df.merge(
            species_table[["species_id", "species", "genus", "family"]],
            on="species_id",
            how="left"
        )

    added_path = ECO_SUB_DIR / f"added_species_{cand_name}.csv"
    removed_path = ECO_SUB_DIR / f"removed_species_{cand_name}.csv"

    added_df.to_csv(added_path, index=False)
    removed_df.to_csv(removed_path, index=False)

    summary_rows.append({
        "candidate": cand_name,
        **cfg,
        "valid": True,
        "known_real_score": np.nan,
        "mean_K": float(k_series.mean()),
        "min_K": int(k_series.min()),
        "max_K": int(k_series.max()),
        "pct_K2": float((k_series == 2).mean() * 100),
        "pct_K8": float((k_series == 8).mean() * 100),
        "changed_images": changed_images,
        "pct_changed_images": float(pct_changed),
        "total_removed_from_champion": total_removed,
        "total_added_to_champion": total_added,
        "max_removed_per_image": max_removed,
        "max_added_per_image": max_added,
        "mean_added_per_changed": total_added / max(changed_images, 1),
        "mean_removed_per_changed": total_removed / max(changed_images, 1),

        "mean_add_score": safe_mean(changed_debug.get("mean_add_score", [])),
        "mean_remove_protect": safe_mean(changed_debug.get("mean_remove_protect", [])),
        "mean_add_eco_rank": safe_mean(changed_debug.get("mean_add_eco_rank", [])),
        "mean_add_eco_score": safe_mean(changed_debug.get("mean_add_eco_score", [])),
        "mean_add_pred_freq": safe_mean(changed_debug.get("mean_add_pred_freq", [])),
        "mean_add_base_freq": safe_mean(changed_debug.get("mean_add_base_freq", [])),
        "mean_add_global_freq": safe_mean(changed_debug.get("mean_add_global_freq", [])),
        "mean_add_base_rank": safe_mean(changed_debug.get("mean_add_base_rank", [])),
        "mean_add_global_rank": safe_mean(changed_debug.get("mean_add_global_rank", [])),
        "mean_remove_base_rank": safe_mean(changed_debug.get("mean_remove_base_rank", [])),
        "mean_remove_global_rank": safe_mean(changed_debug.get("mean_remove_global_rank", [])),
        "mean_remove_eco_rank": safe_mean(changed_debug.get("mean_remove_eco_rank", [])),
        "mean_remove_base_norm": safe_mean(changed_debug.get("mean_remove_base_norm", [])),
        "mean_remove_visual_norm": safe_mean(changed_debug.get("mean_remove_visual_norm", [])),
        "mean_remove_eco_score": safe_mean(changed_debug.get("mean_remove_eco_score", [])),

        "public_submission_path": str(sub_path_public),
        "run_submission_path": str(sub_path_run),
        "debug_path": str(debug_path),
        "added_species_path": str(added_path),
        "removed_species_path": str(removed_path),
    })

    del df_sub, df_debug, changed_debug, added_df, removed_df
    gc.collect()


summary_df = pd.DataFrame(summary_rows)


summary_df["is_replace"] = summary_df["mode"] == "replace"
summary_df["is_add1"] = summary_df["mode"] == "add1"

summary_df["safe_replace_band"] = (
    (summary_df["is_replace"])
    & (summary_df["pct_changed_images"] >= 6.0)
    & (summary_df["pct_changed_images"] <= 28.0)
)

summary_df["strong_replace_band"] = (
    (summary_df["is_replace"])
    & (summary_df["pct_changed_images"] >= 12.0)
    & (summary_df["pct_changed_images"] <= 35.0)
)

summary_df["safe_add_band"] = (
    (summary_df["is_add1"])
    & (summary_df["pct_changed_images"] >= 3.0)
    & (summary_df["pct_changed_images"] <= 25.0)
    & (summary_df["pct_K8"] <= 45.0)
)

summary_df["distance_to_16pct"] = (summary_df["pct_changed_images"] - 16.34).abs()
summary_df["distance_to_22pct"] = (summary_df["pct_changed_images"] - 22.0).abs()

summary_df["recommendation_score"] = 0.0
summary_df.loc[summary_df["safe_replace_band"], "recommendation_score"] += 2.0
summary_df.loc[summary_df["strong_replace_band"], "recommendation_score"] += 0.8
summary_df.loc[summary_df["safe_add_band"], "recommendation_score"] += 0.5
summary_df.loc[summary_df["max_new_species_per_quadrat"] == 1, "recommendation_score"] += 0.3
summary_df.loc[summary_df["pct_changed_images"].between(8, 25), "recommendation_score"] += 0.5
summary_df.loc[summary_df["pct_changed_images"] > 40, "recommendation_score"] -= 1.5
summary_df.loc[summary_df["pct_changed_images"] < 2, "recommendation_score"] -= 1.0
summary_df.loc[summary_df["is_add1"], "recommendation_score"] -= 0.2

summary_df = summary_df.sort_values(
    [
        "recommendation_score",
        "safe_replace_band",
        "distance_to_16pct",
        "pct_changed_images",
    ],
    ascending=[False, False, True, True]
).reset_index(drop=True)

summary_path = ECO_SUB_DIR / "eco_consensus_submission_summary.csv"
summary_df.to_csv(summary_path, index=False)

config_path = ECO_SUB_DIR / "eco_consensus_submission_config.json"
with open(config_path, "w") as f:
    json.dump(
        {
            "known_real_scores": KNOWN_REAL_SCORES,
            "eco_configs": ECO_CONFIGS,
            "inputs": {
                "test": str(TEST_PATH),
                "species_table": str(SPECIES_TABLE_PATH),
                "champion_submission": str(CHAMPION_SUB_PATH),
                "eco_groups": str(ECO_GROUPS_PATH),
                "eco_consensus": str(ECO_CONSENSUS_PATH),
                "group_summary": str(ECO_GROUP_SUMMARY_PATH),
                "base_agg": str(BASE_AGG_PATH),
                "global_top300": str(GLOBAL_TOP300_PATH),
            },
            "outputs": {
                "summary": str(summary_path),
            },
        },
        f,
        indent=2
    )

print("\n" + "=" * 80)
print("RESUMEN ECO_CONSENSUS SUBMISSIONS")
print("=" * 80)

display(
    summary_df[
        [
            "candidate",
            "mode",
            "max_new_species_per_quadrat",
            "pct_changed_images",
            "changed_images",
            "total_removed_from_champion",
            "total_added_to_champion",
            "max_removed_per_image",
            "max_added_per_image",
            "mean_K",
            "pct_K2",
            "pct_K8",
            "mean_add_eco_rank",
            "mean_add_eco_score",
            "mean_add_pred_freq",
            "mean_add_base_freq",
            "mean_add_global_freq",
            "mean_add_base_rank",
            "mean_add_global_rank",
            "mean_remove_base_rank",
            "mean_remove_global_rank",
            "mean_remove_eco_rank",
            "safe_replace_band",
            "safe_add_band",
            "recommendation_score",
            "public_submission_path",
        ]
    ]
)

print("\nArchivos guardados:")
print("Summary:", summary_path)
print("Config:", config_path)


print("\n" + "=" * 80)
print("RECOMENDACIÓN PRELIMINAR")
print("=" * 80)

replace_df = summary_df[summary_df["safe_replace_band"]].copy()

if len(replace_df) > 0:
    recommended = replace_df.sort_values(
        ["recommendation_score", "distance_to_16pct", "pct_changed_images"],
        ascending=[False, True, True]
    ).iloc[0]
else:
    recommended = summary_df.iloc[0]

print("Primera candidata ECO recomendada:")
print("candidate:", recommended["candidate"])
print("mode:", recommended["mode"])
print("pct_changed_images:", round(float(recommended["pct_changed_images"]), 2))
print("changed_images:", int(recommended["changed_images"]))
print("total_added_to_champion:", int(recommended["total_added_to_champion"]))
print("total_removed_from_champion:", int(recommended["total_removed_from_champion"]))
print("mean_add_eco_rank:", recommended["mean_add_eco_rank"])
print("mean_add_eco_score:", recommended["mean_add_eco_score"])
print("mean_K:", round(float(recommended["mean_K"]), 4))
print("pct_K8:", round(float(recommended["pct_K8"]), 2))
print("path:", recommended["public_submission_path"])

print("\nIMPORTANTE:")
print("No subas todas. Primero revisamos el resumen.")
print("Esta estrategia sí puede cambiar más que las ultra-globales, por eso hay que escoger una sola.")
print("Champion actual a superar: GLOBAL_PROTO_ULTRA_TARGET20_v1 = 0.31660.")

# Limpieza
del base_local, global_local
gc.collect()

print("\n Submissions ECO_CONSENSUS generada.")

In [ ]:
import ast
import re
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd


STORAGE_ROOT = Path("/workspace/data/andrea/plantclef-2026")
INDEX_DIR = STORAGE_ROOT / "data" / "indexes"
SUBMISSIONS_DIR = STORAGE_ROOT / "submissions"

ECO_DIR = STORAGE_ROOT / "runs" / "eco_consensus"
ECO_SUB_DIR = ECO_DIR / "eco_submissions"
FINE_OUT_DIR = ECO_DIR / "eco_add_fine_top150"

FINE_OUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

TEST_PATH = INDEX_DIR / "test_2025_manifest_clean.pkl.gz"
SPECIES_TABLE_PATH = INDEX_DIR / "species_table.pkl.gz"

BASE_TOP900_PATH = SUBMISSIONS_DIR / "submission_ECO_CAPPED_SAFE_TOP900.csv"


CURRENT_CHAMPION_PATH = SUBMISSIONS_DIR / "submission_ECO_ADD_OVER_TOP900_ADD1_SAFE_TOP150.csv"


ADD1_SAFE_DEBUG_PATH = ECO_SUB_DIR / "debug_ECO_CONSENSUS_ADD1_SAFE_v1.csv"

print("=" * 80)
print("CELDA 42 — REFINAMIENTO FINO ECO_ADD ALREDEDOR DE TOP150")
print("=" * 80)

paths_to_check = [
    TEST_PATH,
    SPECIES_TABLE_PATH,
    BASE_TOP900_PATH,
    CURRENT_CHAMPION_PATH,
    ADD1_SAFE_DEBUG_PATH,
]

for p in paths_to_check:
    print(p, "| existe:", p.exists())

for p in paths_to_check:
    if not p.exists():
        raise FileNotFoundError(p)


KNOWN_REAL_SCORES = {
    "ECO_CAPPED_SAFE_TOP900": 0.32234,
    "ECO_ADD_OVER_TOP900_ADD1_SAFE_TOP150": 0.33087,
    "ECO_ADD_OVER_TOP900_ADD1_SAFE_TOP250": "bajo bastante",
}

print("\nScores conocidos:")
print(json.dumps(KNOWN_REAL_SCORES, indent=2))


FINE_CAPS = [
    75, 90, 100, 110, 120, 125, 130, 135, 140, 145,
    150, 155, 160, 165, 170, 175, 180, 190, 200, 210, 225
]


FILTERED_VARIANTS = [
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_KLE5",
        "target_cap": 150,
        "filters": {
            "max_K_before": 5,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_KLE6",
        "target_cap": 150,
        "filters": {
            "max_K_before": 6,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP160_KLE6",
        "target_cap": 160,
        "filters": {
            "max_K_before": 6,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP180_KLE6",
        "target_cap": 180,
        "filters": {
            "max_K_before": 6,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_LOCAL_STRICT",
        "target_cap": 150,
        "filters": {
            "local_base_rank_cap": 80,
            "local_global_rank_cap": 80,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP160_LOCAL_STRICT",
        "target_cap": 160,
        "filters": {
            "local_base_rank_cap": 80,
            "local_global_rank_cap": 80,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_LOCAL_VERYSTRICT",
        "target_cap": 150,
        "filters": {
            "local_base_rank_cap": 50,
            "local_global_rank_cap": 50,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_FREQ_STRICT",
        "target_cap": 150,
        "filters": {
            "min_pred_freq": 0.50,
            "min_base_freq": 0.90,
            "min_global_freq": 0.75,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP180_FREQ_STRICT",
        "target_cap": 180,
        "filters": {
            "min_pred_freq": 0.45,
            "min_base_freq": 0.88,
            "min_global_freq": 0.72,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_QUALITY075",
        "target_cap": 150,
        "filters": {
            "min_add_quality": 0.75,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP180_QUALITY070",
        "target_cap": 180,
        "filters": {
            "min_add_quality": 0.70,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP150_GROUPLE30",
        "target_cap": 150,
        "filters": {
            "max_eco_group_size": 30,
        },
    },
    {
        "name": "ECO_ADD_FINE_SAFE_TOP180_GROUPLE40",
        "target_cap": 180,
        "filters": {
            "max_eco_group_size": 40,
        },
    },
]

print("\nFINE_CAPS:", FINE_CAPS)
print("\nFILTERED_VARIANTS:")
print(json.dumps(FILTERED_VARIANTS, indent=2))


def parse_species_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return [int(v) for v in val]
        return [int(val)]
    except Exception:
        nums = re.findall(r"\d+", s)
        return [int(n) for n in nums]


def species_list_to_str(xs):
    return "[" + ", ".join(map(str, [int(x) for x in xs])) + "]"


def safe_numeric(df, col, default=np.nan):
    if col not in df.columns:
        df[col] = default
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)
    return df


def safe_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def validate_submission_df(df_sub, test_df, valid_species, max_k=8, min_k=2):
    errors = []

    if list(df_sub.columns) != ["quadrat_id", "species_ids"]:
        errors.append(f"Columnas incorrectas: {df_sub.columns.tolist()}")

    if len(df_sub) != len(test_df):
        errors.append(f"Filas incorrectas: {len(df_sub)} vs esperado {len(test_df)}")

    valid_quadrats = set(test_df["quadrat_id"].astype(str))
    qids = set(df_sub["quadrat_id"].astype(str))

    if valid_quadrats - qids:
        errors.append(f"Faltan quadrats: {len(valid_quadrats - qids)}")

    if qids - valid_quadrats:
        errors.append(f"Sobran quadrats: {len(qids - valid_quadrats)}")

    if df_sub["quadrat_id"].duplicated().sum() > 0:
        errors.append("Hay quadrat_id duplicados.")

    lists = df_sub["species_ids"].apply(parse_species_ids)
    k = lists.apply(len)

    if (k < min_k).any():
        errors.append(f"Filas con K < {min_k}: {(k < min_k).sum()}")

    if (k > max_k).any():
        errors.append(f"Filas con K > {max_k}: {(k > max_k).sum()}")

    repeated = lists.apply(lambda x: len(x) != len(set(x))).sum()
    if repeated > 0:
        errors.append(f"Filas con especies repetidas: {repeated}")

    invalid = 0
    for sp_list in lists:
        invalid += sum(1 for sp in sp_list if int(sp) not in valid_species)

    if invalid > 0:
        errors.append(f"Species inválidas: {invalid}")

    return errors, k


def infer_added_species(row):
    added = parse_species_ids(row.get("added_to_champion", "[]"))
    if len(added) > 0:
        return int(added[0])

    selected = parse_species_ids(row.get("selected_species", "[]"))
    champion = set(parse_species_ids(row.get("champion_species", "[]")))

    diff = [int(sp) for sp in selected if int(sp) not in champion]
    if len(diff) > 0:
        return int(diff[0])

    return None


def build_add_quality_score(debug_df):
    """
    Misma lógica base de Celda 41.
    """
    df = debug_df.copy()

    numeric_defaults = {
        "best_candidate_score": 0.0,
        "best_candidate_eco_rank": 999999.0,
        "best_candidate_eco_score": 0.0,
        "best_candidate_pred_freq": 0.0,
        "best_candidate_base_freq": 0.0,
        "best_candidate_global_freq": 0.0,
        "best_candidate_base_rank": 999999.0,
        "best_candidate_global_rank": 999999.0,
        "eco_group_size": 1.0,
    }

    for col, default in numeric_defaults.items():
        df = safe_numeric(df, col, default)

    df["eco_rank_score"] = 1.0 / df["best_candidate_eco_rank"].clip(lower=1.0)
    df["base_rank_score"] = 1.0 / df["best_candidate_base_rank"].clip(lower=1.0)
    df["global_rank_score"] = 1.0 / df["best_candidate_global_rank"].clip(lower=1.0)

    df["add_quality"] = (
        0.30 * df["best_candidate_score"].clip(0, 1)
        + 0.24 * df["best_candidate_eco_score"].clip(0, 1)
        + 0.14 * df["best_candidate_pred_freq"].clip(0, 1)
        + 0.12 * df["best_candidate_base_freq"].clip(0, 1)
        + 0.10 * df["best_candidate_global_freq"].clip(0, 1)
        + 0.06 * df["eco_rank_score"].clip(0, 1)
        + 0.02 * df["base_rank_score"].clip(0, 1)
        + 0.02 * df["global_rank_score"].clip(0, 1)
    )

    return df


def passes_filters(row, current_list, filters):
    """
    Aplica filtros opcionales sobre una adición candidata.
    """
    K_before = len(current_list)

    # Evitar K=8
    if K_before >= 8:
        return False

    if "max_K_before" in filters:
        if K_before > int(filters["max_K_before"]):
            return False

    if "local_base_rank_cap" in filters or "local_global_rank_cap" in filters:
        base_cap = int(filters.get("local_base_rank_cap", 999999))
        global_cap = int(filters.get("local_global_rank_cap", 999999))

        base_rank = safe_float(row.get("best_candidate_base_rank", 999999), 999999)
        global_rank = safe_float(row.get("best_candidate_global_rank", 999999), 999999)

       
        if not ((base_rank <= base_cap) or (global_rank <= global_cap)):
            return False

    if "min_pred_freq" in filters:
        if safe_float(row.get("best_candidate_pred_freq", 0.0), 0.0) < float(filters["min_pred_freq"]):
            return False

    if "min_base_freq" in filters:
        if safe_float(row.get("best_candidate_base_freq", 0.0), 0.0) < float(filters["min_base_freq"]):
            return False

    if "min_global_freq" in filters:
        if safe_float(row.get("best_candidate_global_freq", 0.0), 0.0) < float(filters["min_global_freq"]):
            return False

    if "min_add_quality" in filters:
        if safe_float(row.get("add_quality", 0.0), 0.0) < float(filters["min_add_quality"]):
            return False

    if "max_eco_group_size" in filters:
        if safe_float(row.get("eco_group_size", 999999), 999999) > float(filters["max_eco_group_size"]):
            return False

    return True


def compare_submissions(path_a, path_b):
    """
    Cuenta cuántas filas difieren entre dos submissions.
    """
    a = pd.read_csv(path_a)
    b = pd.read_csv(path_b)

    a["quadrat_id"] = a["quadrat_id"].astype(str)
    b["quadrat_id"] = b["quadrat_id"].astype(str)

    merged = a.merge(
        b,
        on="quadrat_id",
        how="outer",
        suffixes=("_a", "_b")
    )

    return int((merged["species_ids_a"] != merged["species_ids_b"]).sum())


def generate_add_submission(candidate_name, target_cap, filters, changed_ranked, base_df, base_lookup):
    """
    Genera una submission agregando especies sobre BASE_TOP900.
    """
    additions_applied = {}
    skipped_rows = []

    for row in changed_ranked.itertuples(index=False):
        qid = row.quadrat_id
        sp = int(row.proposed_add_species)

        if qid not in base_lookup:
            skipped_rows.append({
                "quadrat_id": qid,
                "species_id": sp,
                "reason": "qid_not_in_base",
            })
            continue

        current_list = base_lookup[qid]

        if len(current_list) >= 8:
            skipped_rows.append({
                "quadrat_id": qid,
                "species_id": sp,
                "reason": "already_K8",
            })
            continue

        if sp in current_list:
            skipped_rows.append({
                "quadrat_id": qid,
                "species_id": sp,
                "reason": "already_present",
            })
            continue

        if qid in additions_applied:
            skipped_rows.append({
                "quadrat_id": qid,
                "species_id": sp,
                "reason": "already_added_this_qid",
            })
            continue

        row_dict = row._asdict()

        if not passes_filters(row_dict, current_list, filters):
            skipped_rows.append({
                "quadrat_id": qid,
                "species_id": sp,
                "reason": "filtered_out",
            })
            continue

        additions_applied[qid] = {
            "species_id": sp,
            "add_quality": float(row.add_quality),
            "best_candidate_score": float(row.best_candidate_score),
            "best_candidate_eco_rank": float(row.best_candidate_eco_rank),
            "best_candidate_eco_score": float(row.best_candidate_eco_score),
            "best_candidate_pred_freq": float(row.best_candidate_pred_freq),
            "best_candidate_base_freq": float(row.best_candidate_base_freq),
            "best_candidate_global_freq": float(row.best_candidate_global_freq),
            "best_candidate_base_rank": float(row.best_candidate_base_rank),
            "best_candidate_global_rank": float(row.best_candidate_global_rank),
            "eco_group_size": float(row.eco_group_size),
        }

        if len(additions_applied) >= int(target_cap):
            break

    out_rows = []
    debug_rows = []

    for row in base_df.itertuples(index=False):
        qid = row.quadrat_id
        base_list = [int(sp) for sp in row.base_species]
        final_list = list(base_list)

        if qid in additions_applied:
            add_sp = int(additions_applied[qid]["species_id"])
            final_list.append(add_sp)

        final_list = list(dict.fromkeys(final_list))[:8]

        changed = int(set(final_list) != set(base_list))

        out_rows.append({
            "quadrat_id": qid,
            "species_ids": species_list_to_str(final_list),
        })

        debug_rows.append({
            "quadrat_id": qid,
            "candidate": candidate_name,
            "K_before": len(base_list),
            "K_after": len(final_list),
            "changed_set": changed,
            "added_species": species_list_to_str([additions_applied[qid]["species_id"]]) if qid in additions_applied else "[]",
            "base_species": species_list_to_str(base_list),
            "final_species": species_list_to_str(final_list),
            **({f"add_{k}": v for k, v in additions_applied[qid].items()} if qid in additions_applied else {}),
        })

    return pd.DataFrame(out_rows), pd.DataFrame(debug_rows), pd.DataFrame(skipped_rows)



print("\n" + "=" * 80)
print("CARGANDO DATOS")
print("=" * 80)

test_df = pd.read_pickle(TEST_PATH, compression="gzip")
test_df["quadrat_id"] = test_df["quadrat_id"].astype(str)

species_table = pd.read_pickle(SPECIES_TABLE_PATH, compression="gzip")
species_table["species_id"] = species_table["species_id"].astype(int)
VALID_SPECIES = set(species_table["species_id"].astype(int).tolist())

base_top900 = pd.read_csv(BASE_TOP900_PATH)
base_top900["quadrat_id"] = base_top900["quadrat_id"].astype(str)
base_top900["base_species"] = base_top900["species_ids"].apply(parse_species_ids)
base_top900["K_base"] = base_top900["base_species"].apply(len)

add_debug = pd.read_csv(ADD1_SAFE_DEBUG_PATH)
add_debug["quadrat_id"] = add_debug["quadrat_id"].astype(str)

print("test_df:", test_df.shape)
print("base_top900:", base_top900.shape)
print("add_debug:", add_debug.shape)

print("\nK base TOP900:")
display(base_top900["K_base"].value_counts().sort_index().reset_index(name="n_images"))

base_lookup = {
    row.quadrat_id: [int(sp) for sp in row.base_species]
    for row in base_top900.itertuples(index=False)
}


print("\n" + "=" * 80)
print("PREPARANDO RANKING ADD1_SAFE")
print("=" * 80)

changed = add_debug[add_debug["changed_set"] == 1].copy()

changed["proposed_add_species"] = changed.apply(infer_added_species, axis=1)
changed = changed.dropna(subset=["proposed_add_species"]).copy()
changed["proposed_add_species"] = changed["proposed_add_species"].astype(int)

changed = changed[changed["proposed_add_species"].isin(VALID_SPECIES)].copy()

changed = build_add_quality_score(changed)

changed = changed.sort_values(
    [
        "add_quality",
        "best_candidate_score",
        "best_candidate_eco_score",
        "best_candidate_pred_freq",
        "best_candidate_base_freq",
        "best_candidate_global_freq",
    ],
    ascending=[False, False, False, False, False, False]
).reset_index(drop=True)

print("Adiciones candidatas:", len(changed))

print("\nTop 25 adiciones:")
display(
    changed[
        [
            "quadrat_id",
            "proposed_add_species",
            "add_quality",
            "best_candidate_score",
            "best_candidate_eco_rank",
            "best_candidate_eco_score",
            "best_candidate_pred_freq",
            "best_candidate_base_freq",
            "best_candidate_global_freq",
            "best_candidate_base_rank",
            "best_candidate_global_rank",
            "eco_group_size",
            "added_to_champion",
        ]
    ].head(25)
)


variant_specs = []


for cap in FINE_CAPS:
    variant_specs.append({
        "candidate": f"ECO_ADD_FINE_SAFE_TOP{cap}",
        "target_cap": int(cap),
        "variant_type": "plain_cap",
        "filters": {},
    })


for spec in FILTERED_VARIANTS:
    variant_specs.append({
        "candidate": spec["name"],
        "target_cap": int(spec["target_cap"]),
        "variant_type": "filtered",
        "filters": spec["filters"],
    })

print("\nNúmero de variantes a generar:", len(variant_specs))


print("\n" + "=" * 80)
print("GENERANDO VARIANTES FINAS")
print("=" * 80)

summary_rows = []

for spec in variant_specs:
    cand_name = spec["candidate"]
    target_cap = int(spec["target_cap"])
    filters = spec["filters"]
    variant_type = spec["variant_type"]

    df_sub, df_debug, df_skipped = generate_add_submission(
        candidate_name=cand_name,
        target_cap=target_cap,
        filters=filters,
        changed_ranked=changed,
        base_df=base_top900,
        base_lookup=base_lookup,
    )

    errors, k_series = validate_submission_df(
        df_sub,
        test_df=test_df,
        valid_species=VALID_SPECIES,
        max_k=8,
        min_k=2,
    )

    if errors:
        print(" Errores en", cand_name)
        for e in errors:
            print("-", e)
        raise RuntimeError(f"Submission inválida: {cand_name}")

    n_added = int(df_debug["changed_set"].sum())
    pct_added = n_added / len(test_df) * 100

    sub_path_public = SUBMISSIONS_DIR / f"submission_{cand_name}.csv"
    sub_path_run = FINE_OUT_DIR / f"submission_{cand_name}.csv"
    debug_path = FINE_OUT_DIR / f"debug_{cand_name}.csv"
    skipped_path = FINE_OUT_DIR / f"skipped_{cand_name}.csv"

    df_sub.to_csv(sub_path_public, index=False)
    df_sub.to_csv(sub_path_run, index=False)
    df_debug.to_csv(debug_path, index=False)
    df_skipped.to_csv(skipped_path, index=False)

    changed_debug = df_debug[df_debug["changed_set"] == 1].copy()

   
    diff_vs_current = compare_submissions(sub_path_public, CURRENT_CHAMPION_PATH)

    summary_rows.append({
        "candidate": cand_name,
        "variant_type": variant_type,
        "target_cap": target_cap,
        "actual_added_images": n_added,
        "pct_added_images": pct_added,
        "diff_vs_current_TOP150": diff_vs_current,
        "mean_K": float(k_series.mean()),
        "min_K": int(k_series.min()),
        "max_K": int(k_series.max()),
        "pct_K2": float((k_series == 2).mean() * 100),
        "pct_K8": float((k_series == 8).mean() * 100),
        "mean_add_quality": float(changed_debug["add_add_quality"].mean()) if len(changed_debug) else np.nan,
        "min_add_quality": float(changed_debug["add_add_quality"].min()) if len(changed_debug) else np.nan,
        "mean_candidate_score": float(changed_debug["add_best_candidate_score"].mean()) if len(changed_debug) else np.nan,
        "mean_eco_rank": float(changed_debug["add_best_candidate_eco_rank"].mean()) if len(changed_debug) else np.nan,
        "mean_eco_score": float(changed_debug["add_best_candidate_eco_score"].mean()) if len(changed_debug) else np.nan,
        "mean_pred_freq": float(changed_debug["add_best_candidate_pred_freq"].mean()) if len(changed_debug) else np.nan,
        "mean_base_freq": float(changed_debug["add_best_candidate_base_freq"].mean()) if len(changed_debug) else np.nan,
        "mean_global_freq": float(changed_debug["add_best_candidate_global_freq"].mean()) if len(changed_debug) else np.nan,
        "mean_base_rank": float(changed_debug["add_best_candidate_base_rank"].mean()) if len(changed_debug) else np.nan,
        "mean_global_rank": float(changed_debug["add_best_candidate_global_rank"].mean()) if len(changed_debug) else np.nan,
        "filters_json": json.dumps(filters),
        "public_submission_path": str(sub_path_public),
        "run_submission_path": str(sub_path_run),
        "debug_path": str(debug_path),
        "skipped_path": str(skipped_path),
    })

    print(
        f" {cand_name}: target={target_cap}, actual={n_added} "
        f"({pct_added:.2f}%), diff_vs_TOP150={diff_vs_current}, "
        f"mean_K={float(k_series.mean()):.4f}, K8={float((k_series == 8).mean() * 100):.2f}%"
    )


summary_df = pd.DataFrame(summary_rows)

summary_df["distance_to_150"] = (summary_df["actual_added_images"] - 150).abs()
summary_df["distance_to_140"] = (summary_df["actual_added_images"] - 140).abs()
summary_df["distance_to_160"] = (summary_df["actual_added_images"] - 160).abs()

summary_df["recommendation_score"] = 0.0


summary_df.loc[summary_df["actual_added_images"].between(120, 180), "recommendation_score"] += 2.0
summary_df.loc[summary_df["actual_added_images"].between(135, 165), "recommendation_score"] += 0.8
summary_df.loc[summary_df["variant_type"] == "plain_cap", "recommendation_score"] += 0.3


summary_df["recommendation_score"] += summary_df["mean_add_quality"].fillna(0) * 1.2


summary_df.loc[summary_df["diff_vs_current_TOP150"] == 0, "recommendation_score"] -= 2.0
summary_df.loc[summary_df["actual_added_images"] > 210, "recommendation_score"] -= 1.0
summary_df.loc[summary_df["actual_added_images"] < 90, "recommendation_score"] -= 0.8
summary_df.loc[summary_df["pct_K8"] > 34.0, "recommendation_score"] -= 0.5

summary_df = summary_df.sort_values(
    [
        "recommendation_score",
        "distance_to_150",
        "mean_add_quality",
    ],
    ascending=[False, True, False]
).reset_index(drop=True)

summary_path = FINE_OUT_DIR / "eco_add_fine_top150_summary.csv"
config_path = FINE_OUT_DIR / "eco_add_fine_top150_config.json"

summary_df.to_csv(summary_path, index=False)

with open(config_path, "w") as f:
    json.dump(
        {
            "known_real_scores": KNOWN_REAL_SCORES,
            "fine_caps": FINE_CAPS,
            "filtered_variants": FILTERED_VARIANTS,
            "base_top900": str(BASE_TOP900_PATH),
            "current_champion_top150": str(CURRENT_CHAMPION_PATH),
            "add1_safe_debug": str(ADD1_SAFE_DEBUG_PATH),
            "outputs": {
                "summary": str(summary_path),
            },
        },
        f,
        indent=2,
    )

print("\n" + "=" * 80)
print("RESUMEN ECO_ADD FINE TOP150")
print("=" * 80)

display(
    summary_df[
        [
            "candidate",
            "variant_type",
            "target_cap",
            "actual_added_images",
            "pct_added_images",
            "diff_vs_current_TOP150",
            "mean_K",
            "pct_K2",
            "pct_K8",
            "mean_add_quality",
            "min_add_quality",
            "mean_candidate_score",
            "mean_eco_rank",
            "mean_eco_score",
            "mean_pred_freq",
            "mean_base_freq",
            "mean_global_freq",
            "mean_base_rank",
            "mean_global_rank",
            "recommendation_score",
            "public_submission_path",
        ]
    ]
)

print("\nArchivos guardados:")
print("Summary:", summary_path)
print("Config:", config_path)

print("\n" + "=" * 80)
print("RECOMENDACIÓN PRELIMINAR")
print("=" * 80)


rec_df = summary_df[summary_df["diff_vs_current_TOP150"] > 0].copy()

if len(rec_df) == 0:
    recommended = summary_df.iloc[0]
else:
    recommended = rec_df.iloc[0]

print("Primera candidata fina recomendada:")
print("candidate:", recommended["candidate"])
print("actual_added_images:", int(recommended["actual_added_images"]))
print("pct_added_images:", round(float(recommended["pct_added_images"]), 2))
print("diff_vs_current_TOP150:", int(recommended["diff_vs_current_TOP150"]))
print("mean_K:", round(float(recommended["mean_K"]), 4))
print("pct_K8:", round(float(recommended["pct_K8"]), 2))
print("mean_add_quality:", round(float(recommended["mean_add_quality"]), 4))
print("path:", recommended["public_submission_path"])

print("\nOrden práctico sugerido:")
print("1) Probar TOP140 o TOP160 primero, según el resumen.")
print("2) Si TOP140 mejora: probar TOP130/TOP145.")
print("3) Si TOP160 mejora: probar TOP170/TOP180.")
print("4) Si ninguna mejora: mantener TOP150 = 0.33087.")
print("5) No volver a TOP250 o más por ahora.")

gc.collect()

print("\n Refinamiento fino ECO_ADD generado.")